# **Brazil**

In [ ]:
# ============================================
# STEP 1: LIBRARY IMPORTS
# ============================================

# Core data manipulation
import pandas as pd
import numpy as np

# Geospatial clustering
from sklearn.cluster import DBSCAN
from math import radians, sin, cos, sqrt, atan2

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# File handling
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


In [ ]:
# ============================================
# STEP 2: DATA LOADING & CLEANING
# ============================================

# Load the data (change the file path for each country)
df = pd.read_excel("Levi's_Brazil_Final_All_caluclated _Data.xlsx")

# Drop the empty trailing column (has space as name)
df = df.loc[:, ~df.columns.str.contains('Unnamed|^ $')]

# Basic info
print(f"Country: Brazil")  # Update per country
print(f"Total stores loaded: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")

# Clean and standardize data
df_clean = df.copy()

# Remove rows with missing coordinates (can't cluster without location)
df_clean = df_clean.dropna(subset=['location_lat', 'location_lng'])

# Fill missing ratings with 0 (treat as low-performing)
df_clean['totalRating'] = df_clean['totalRating'].fillna(0)
df_clean['reviewsCount'] = df_clean['reviewsCount'].fillna(0)

# Standardize postal codes to string
df_clean['postalCode'] = df_clean['postalCode'].astype(str).str.strip()

# Filter for actual Levi's stores (remove non-retail entries)
# Keep rows where categoryName contains 'Levi' or common retail keywords
df_clean = df_clean[
    df_clean['categoryName'].str.contains('Levi|Clothing|Apparel|Fashion|Store',
                                          case=False, na=False)
]

print(f"\nAfter cleaning:")
print(f"Stores remaining: {len(df_clean)}")
print(f"Removed: {len(df) - len(df_clean)} stores")
print(f"\nMissing values per column:")
print(df_clean.isnull().sum())

# Preview the cleaned data
print("\nSample data:")
print(df_clean[['title', 'city', 'totalRating', 'reviewsCount',
                'location_lat', 'location_lng']].head())

Country: Brazil
Total stores loaded: 123

Columns: ['title', 'address', 'neighborhood', 'street', 'city', 'postalCode', 'state', 'countryCode', 'website', 'phone', 'location_lat', 'location_lng', 'plusCode', 'categoryName', 'totalRating', 'reviewsCount']

After cleaning:
Stores remaining: 123
Removed: 0 stores

Missing values per column:
title           0
address         0
neighborhood    2
street          1
city            0
postalCode      0
state           0
countryCode     0
website         9
phone           2
location_lat    0
location_lng    0
plusCode        1
categoryName    0
totalRating     0
reviewsCount    0
dtype: int64

Sample data:
                            title         city  totalRating  reviewsCount  location_lat  location_lng
0    Original - Levis & John Jonh     Dourados          4.5            18    -22.226961    -54.808236
1               Levi's - Dourados     Dourados          5.0             1    -22.226822    -54.811371
2                          Levi's  Jabo

In [ ]:
# ============================================
# STEP 3: CLUSTER ALL STORES BY CITY
# ============================================

# Count stores per city
city_summary = df_clean.groupby('city').agg({
    'title': 'count',
    'state': 'first',
    'location_lat': 'mean',
    'location_lng': 'mean'
}).rename(columns={'title': 'store_count'}).reset_index()

# Sort by store count
city_summary = city_summary.sort_values('store_count', ascending=False)

print("=" * 80)
print("CITY CLUSTERING - ALL 123 STORES")
print("=" * 80)

# Split into single-store and multi-store cities for analysis
multi_store_cities = city_summary[city_summary['store_count'] >= 2]
single_store_cities = city_summary[city_summary['store_count'] == 1]

print(f"\nTotal stores in dataset: {len(df_clean)}")
print(f"Total cities: {len(city_summary)}")
print(f"\nCity Distribution:")
print(f"  Cities with 1 store: {len(single_store_cities)} cities ({single_store_cities['store_count'].sum()} stores)")
print(f"  Cities with 2+ stores: {len(multi_store_cities)} cities ({multi_store_cities['store_count'].sum()} stores)")

# Verify totals
total_check = single_store_cities['store_count'].sum() + multi_store_cities['store_count'].sum()
print(f"\n✓ Verification: {total_check} stores accounted for (should be {len(df_clean)})")

# Show ALL cities with their store counts
print("\n" + "=" * 80)
print("COMPLETE CITY BREAKDOWN (ALL CITIES)")
print("=" * 80)
print(city_summary.to_string(index=False))

# Add city cluster assignment to main dataframe (KEEP ALL STORES)
df_clean['city_cluster'] = df_clean['city']
df_clean['stores_in_city'] = df_clean['city'].map(city_summary.set_index('city')['store_count'])

print("\n" + "=" * 80)
print("CLUSTER ASSIGNMENT COMPLETE")
print("=" * 80)
print(f"✓ All {len(df_clean)} stores assigned to {len(city_summary)} city clusters")

# Show sample of clustered data
print("\nSample of clustered stores:")
print(df_clean[['title', 'city', 'city_cluster', 'stores_in_city', 'totalRating', 'reviewsCount']].head(10))

# Summary by cluster size
print("\n" + "=" * 80)
print("CLUSTER SIZE DISTRIBUTION")
print("=" * 80)

cluster_size_distribution = df_clean.groupby('stores_in_city').size().reset_index(name='number_of_cities')
cluster_size_distribution = cluster_size_distribution.sort_values('stores_in_city', ascending=False)

print("\nStores per City | Number of Cities | Total Stores")
print("-" * 60)
for _, row in cluster_size_distribution.iterrows():
    total_stores_in_category = row['stores_in_city'] * row['number_of_cities']
    print(f"{int(row['stores_in_city']):>15} | {int(row['number_of_cities']):>16} | {int(total_stores_in_category):>12}")

print("\n" + "=" * 80)
print(f"✓ ALL {len(df_clean)} STORES CLUSTERED BY CITY")
print("=" * 80)

CITY CLUSTERING - ALL 123 STORES

Total stores in dataset: 123
Total cities: 77

City Distribution:
  Cities with 1 store: 59 cities (59 stores)
  Cities with 2+ stores: 18 cities (64 stores)

✓ Verification: 123 stores accounted for (should be 123)

COMPLETE CITY BREAKDOWN (ALL CITIES)
                 city  store_count                        state  location_lat  location_lng
            São Paulo           22           State of São Paulo    -23.567987    -46.664695
       Rio de Janeiro            6      State of Rio de Janeiro    -22.924611    -43.251814
             Curitiba            5              State of Paraná    -25.440910    -49.268491
       Belo Horizonte            3        State of Minas Gerais    -19.947586    -43.942731
             Brasília            2             Federal District    -15.837770    -47.999525
              Goiânia            2               State of Goiás    -16.709154    -49.254838
        Foz do Iguaçu            2              State of Paraná    -

In [ ]:
# ============================================
# STEP 4: DISPLAY STORES IN EACH MULTI-STORE CITY
# ============================================

print("=" * 80)
print("DETAILED STORE BREAKDOWN BY CITY")
print("=" * 80)

# Get list of cities with multiple stores
multi_store_city_list = multi_store_cities['city'].tolist()

# Loop through each multi-store city
for idx, city_name in enumerate(multi_store_city_list, 1):

    # Get all stores in this city
    city_stores = df_clean[df_clean['city'] == city_name].copy()

    # Sort by rating (highest first)
    city_stores = city_stores.sort_values('totalRating', ascending=False)

    # Get state info
    state_name = city_stores['state'].iloc[0]

    print(f"\n{'='*80}")
    print(f"CITY {idx}: {city_name}, {state_name}")
    print(f"Total Stores: {len(city_stores)}")
    print(f"{'='*80}")

    # Display each store
    for store_idx, (_, store) in enumerate(city_stores.iterrows(), 1):
        print(f"\n  Store {store_idx}:")
        print(f"    Name:          {store['title']}")
        print(f"    Address:       {store['address']}")
        print(f"    Rating:        {store['totalRating']:.1f} ⭐")
        print(f"    Review Count:  {int(store['reviewsCount'])} reviews")
        print(f"    Coordinates:   ({store['location_lat']:.4f}, {store['location_lng']:.4f})")

    print(f"\n{'-'*80}")

print(f"\n{'='*80}")
print(f"END OF CITY BREAKDOWN")
print(f"{'='*80}")

DETAILED STORE BREAKDOWN BY CITY

CITY 1: São Paulo, State of São Paulo
Total Stores: 22

  Store 1:
    Name:          Levi's - Tietê Plaza Shopping
    Address:       Av. Raimundo Pereira de Magalhães, 1465 - Jardim Iris, São Paulo - SP, 05145-000, Brazil
    Rating:        5.0 ⭐
    Review Count:  2 reviews
    Coordinates:   (-23.5066, -46.7186)

  Store 2:
    Name:          Levi's Store Lar Center
    Address:       Av. Otto Baumgart, 500 - Vila Guilherme, São Paulo - SP, 02049-000, Brazil
    Rating:        5.0 ⭐
    Review Count:  1 reviews
    Coordinates:   (-23.5144, -46.6153)

  Store 3:
    Name:          Levi's - Shops Jardins
    Address:       Rua Haddock Lobo, 1626 - Jardins, São Paulo - SP, 01414-002, Brazil
    Rating:        5.0 ⭐
    Review Count:  7 reviews
    Coordinates:   (-23.5645, -46.6690)

  Store 4:
    Name:          Levi's
    Address:       R. Frei Caneca, 569 - Consolação, São Paulo - SP, 01307-001, Brazil
    Rating:        4.7 ⭐
    Review Count:  1

In [ ]:
# ============================================
# STEP 5: CALCULATE DISTANCES BETWEEN STORES
# ============================================

import requests
import time
from itertools import combinations

def get_road_distance(lat1, lon1, lat2, lon2):
    """
    Get driving distance using OSRM
    Returns: distance in miles
    """
    try:
        url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
        params = {'overview': 'false'}

        response = requests.get(url, params=params, timeout=10)

        if response.status_code == 200:
            data = response.json()
            if data['code'] == 'Ok':
                distance_meters = data['routes'][0]['distance']
                distance_miles = distance_meters * 0.000621371
                return distance_miles

        return None

    except Exception as e:
        return None


# Get cities with 2+ stores
multi_store_cities = city_summary[city_summary['store_count'] >= 2]

# Store all distances
all_distances = []

print("=" * 100)
print("CALCULATING DISTANCES BETWEEN STORES (BY CITY CLUSTER)")
print("=" * 100)

# Process each city cluster
for city_idx, (_, city_row) in enumerate(multi_store_cities.iterrows(), 1):
    city_name = city_row['city']
    state_name = city_row['state']
    store_count = city_row['store_count']

    print(f"\nCity {city_idx}/{len(multi_store_cities)}: {city_name}, {state_name} ({store_count} stores)")

    # Get all stores in this city
    city_stores = df_clean[df_clean['city'] == city_name].reset_index(drop=True)

    # Calculate distances for all pairs
    for i, j in combinations(range(len(city_stores)), 2):
        store_a = city_stores.iloc[i]
        store_b = city_stores.iloc[j]

        # Get road distance
        distance = get_road_distance(
            store_a['location_lat'], store_a['location_lng'],
            store_b['location_lat'], store_b['location_lng']
        )

        if distance is not None:
            all_distances.append({
                'city': city_name,
                'state': state_name,
                'store_a_name': store_a['title'],
                'store_a_address': store_a['address'],
                'store_a_rating': store_a['totalRating'],
                'store_a_reviews': store_a['reviewsCount'],
                'store_b_name': store_b['title'],
                'store_b_address': store_b['address'],
                'store_b_rating': store_b['totalRating'],
                'store_b_reviews': store_b['reviewsCount'],
                'distance_miles': distance
            })

        time.sleep(0.3)  # Rate limiting

    print(f"  ✓ Calculated {len(city_stores) * (len(city_stores) - 1) // 2} distances")

# Convert to DataFrame
df_distances = pd.DataFrame(all_distances)

print("\n" + "=" * 100)
print("DISTANCE CALCULATION COMPLETE")
print("=" * 100)
print(f"Total store pairs analyzed: {len(df_distances)}")

# ====================================
# DISPLAY BY CITY CLUSTER
# ====================================

print("\n" + "=" * 100)
print("DISTANCES BY CITY CLUSTER (SORTED BY DISTANCE)")
print("=" * 100)

for city_name in df_distances['city'].unique():
    city_data = df_distances[df_distances['city'] == city_name].copy()
    city_data = city_data.sort_values('distance_miles').reset_index(drop=True)

    state_name = city_data['state'].iloc[0]

    print(f"\n{'='*100}")
    print(f"CLUSTER: {city_name}, {state_name}")
    print(f"{'='*100}")

    # Create display table
    display_table = pd.DataFrame({
        'Store A': city_data['store_a_name'],
        'Rating A': city_data['store_a_rating'].round(1),
        'Reviews A': city_data['store_a_reviews'].astype(int),
        'Store B': city_data['store_b_name'],
        'Rating B': city_data['store_b_rating'].round(1),
        'Reviews B': city_data['store_b_reviews'].astype(int),
        'Distance (miles)': city_data['distance_miles'].round(2)
    })

    print(display_table.to_string(index=True))
    print(f"\nTotal pairs in this cluster: {len(city_data)}")
    print(f"Distance range: {city_data['distance_miles'].min():.2f} - {city_data['distance_miles'].max():.2f} miles")

# ====================================
# CONSOLIDATED TABLE
# ====================================

print("\n" + "=" * 100)
print("CONSOLIDATED TABLE - ALL DISTANCES (SORTED BY DISTANCE)")
print("=" * 100)

consolidated_table = pd.DataFrame({
    'City': df_distances['city'],
    'State': df_distances['state'],
    'Store A': df_distances['store_a_name'],
    'Rating A': df_distances['store_a_rating'].round(1),
    'Reviews A': df_distances['store_a_reviews'].astype(int),
    'Store B': df_distances['store_b_name'],
    'Rating B': df_distances['store_b_rating'].round(1),
    'Reviews B': df_distances['store_b_reviews'].astype(int),
    'Distance (miles)': df_distances['distance_miles'].round(2)
})

consolidated_table = consolidated_table.sort_values('Distance (miles)').reset_index(drop=True)

print(consolidated_table.to_string(index=True))

print("\n" + "=" * 100)
print("ANALYSIS COMPLETE")
print("=" * 100)

CALCULATING DISTANCES BETWEEN STORES (BY CITY CLUSTER)

City 1/18: São Paulo, State of São Paulo (22 stores)
  ✓ Calculated 231 distances

City 2/18: Rio de Janeiro, State of Rio de Janeiro (6 stores)
  ✓ Calculated 15 distances

City 3/18: Curitiba, State of Paraná (5 stores)
  ✓ Calculated 10 distances

City 4/18: Belo Horizonte, State of Minas Gerais (3 stores)
  ✓ Calculated 3 distances

City 5/18: Brasília, Federal District (2 stores)
  ✓ Calculated 1 distances

City 6/18: Goiânia, State of Goiás (2 stores)
  ✓ Calculated 1 distances

City 7/18: Foz do Iguaçu, State of Paraná (2 stores)
  ✓ Calculated 1 distances

City 8/18: Manaus, State of Amazonas (2 stores)
  ✓ Calculated 1 distances

City 9/18: Joinville, State of Santa Catarina (2 stores)
  ✓ Calculated 1 distances

City 10/18: Dourados, State of Mato Grosso do Sul (2 stores)
  ✓ Calculated 1 distances

City 11/18: Campinas, State of São Paulo (2 stores)
  ✓ Calculated 1 distances

City 12/18: Balneário Camboriú, State of Sa

In [ ]:
# ============================================
# STEP 5: ROAD DISTANCE CALCULATION
# ============================================

import requests
import time
from itertools import combinations

def get_road_distance_and_route(lat1, lon1, lat2, lon2):
    """
    Get driving distance AND route geometry using OSRM
    Returns: (distance_miles, route_geometry)
    """
    try:
        url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
        params = {
            'overview': 'full',
            'geometries': 'geojson'
        }

        response = requests.get(url, params=params, timeout=10)

        if response.status_code == 200:
            data = response.json()
            if data['code'] == 'Ok':
                route = data['routes'][0]
                distance_meters = route['distance']
                distance_miles = distance_meters * 0.000621371
                geometry = route['geometry']
                return distance_miles, geometry

        return None, None

    except Exception as e:
        return None, None


# Get list of multi-store cities
multi_store_city_list = multi_store_cities['city'].tolist()

# Store results
all_distance_data = []

print("=" * 80)
print("CALCULATING ROAD DISTANCES BY CITY CLUSTER")
print("=" * 80)

# Process each multi-store city
for city_name in multi_store_city_list:

    city_stores = df_clean[df_clean['city'] == city_name].copy()
    store_count = len(city_stores)

    print(f"\nAnalyzing: {city_name} ({store_count} stores)")

    # Get all pairwise combinations of stores
    store_list = city_stores.reset_index(drop=True)

    for i, j in combinations(range(len(store_list)), 2):
        store_a = store_list.iloc[i]
        store_b = store_list.iloc[j]

        # Calculate ROAD distance with route geometry
        distance_miles, route_geometry = get_road_distance_and_route(
            store_a['location_lat'], store_a['location_lng'],
            store_b['location_lat'], store_b['location_lng']
        )

        if distance_miles is not None:
            distance_data = {
                'city': city_name,
                'state': store_a['state'],
                'store_1_name': store_a['title'],
                'store_1_address': store_a['address'],
                'store_1_rating': store_a['totalRating'],
                'store_1_reviews': store_a['reviewsCount'],
                'store_1_lat': store_a['location_lat'],
                'store_1_lng': store_a['location_lng'],
                'store_2_name': store_b['title'],
                'store_2_address': store_b['address'],
                'store_2_rating': store_b['totalRating'],
                'store_2_reviews': store_b['reviewsCount'],
                'store_2_lat': store_b['location_lat'],
                'store_2_lng': store_b['location_lng'],
                'road_distance_miles': distance_miles,
                'route_geometry': route_geometry
            }

            all_distance_data.append(distance_data)

        # Delay to avoid API rate limiting
        time.sleep(0.5)

    print(f"  ✓ Completed {city_name}")

# Convert to DataFrame
df_clusters = pd.DataFrame(all_distance_data)

print("\n" + "=" * 80)
print("DISTANCE CALCULATION COMPLETE")
print("=" * 80)

if len(df_clusters) > 0:
    print(f"\nTotal store pairs: {len(df_clusters)}")
    print(f"Cities analyzed: {df_clusters['city'].nunique()}")

    # ====================================
    # DISPLAY BY CITY CLUSTER
    # ====================================

    print("\n" + "=" * 80)
    print("DISTANCES BY CITY CLUSTER")
    print("=" * 80)

    for city_name in df_clusters['city'].unique():
        city_data = df_clusters[df_clusters['city'] == city_name].copy()
        city_data = city_data.sort_values('road_distance_miles').reset_index(drop=True)

        state_name = city_data['state'].iloc[0]

        print(f"\n{'='*80}")
        print(f"CLUSTER: {city_name}, {state_name}")
        print(f"{'='*80}")

        # Create table
        table = pd.DataFrame({
            'Store A': city_data['store_1_name'],
            'Rating A': city_data['store_1_rating'].round(1),
            'Reviews A': city_data['store_1_reviews'].astype(int),
            'Store B': city_data['store_2_name'],
            'Rating B': city_data['store_2_rating'].round(1),
            'Reviews B': city_data['store_2_reviews'].astype(int),
            'Distance (mi)': city_data['road_distance_miles'].round(2)
        })

        print(table.to_string(index=True))
        print(f"\nPairs in cluster: {len(city_data)}")

    # ====================================
    # CONSOLIDATED TABLE
    # ====================================

    print("\n" + "=" * 80)
    print("ALL DISTANCES - SORTED BY DISTANCE")
    print("=" * 80)

    final_table = pd.DataFrame({
        'City': df_clusters['city'],
        'State': df_clusters['state'],
        'Store A': df_clusters['store_1_name'],
        'Rating A': df_clusters['store_1_rating'].round(1),
        'Reviews A': df_clusters['store_1_reviews'].astype(int),
        'Store B': df_clusters['store_2_name'],
        'Rating B': df_clusters['store_2_rating'].round(1),
        'Reviews B': df_clusters['store_2_reviews'].astype(int),
        'Distance (mi)': df_clusters['road_distance_miles'].round(2)
    })

    final_table = final_table.sort_values('Distance (mi)').reset_index(drop=True)

    print(final_table.to_string(index=True))

else:
    print("\n⚠️  No distance data calculated")

CALCULATING ROAD DISTANCES BY CITY CLUSTER

Analyzing: São Paulo (22 stores)
  ✓ Completed São Paulo

Analyzing: Rio de Janeiro (6 stores)
  ✓ Completed Rio de Janeiro

Analyzing: Curitiba (5 stores)
  ✓ Completed Curitiba

Analyzing: Belo Horizonte (3 stores)
  ✓ Completed Belo Horizonte

Analyzing: Brasília (2 stores)
  ✓ Completed Brasília

Analyzing: Goiânia (2 stores)
  ✓ Completed Goiânia

Analyzing: Foz do Iguaçu (2 stores)
  ✓ Completed Foz do Iguaçu

Analyzing: Manaus (2 stores)
  ✓ Completed Manaus

Analyzing: Joinville (2 stores)
  ✓ Completed Joinville

Analyzing: Dourados (2 stores)
  ✓ Completed Dourados

Analyzing: Campinas (2 stores)
  ✓ Completed Campinas

Analyzing: Balneário Camboriú (2 stores)
  ✓ Completed Balneário Camboriú

Analyzing: Porto Alegre (2 stores)
  ✓ Completed Porto Alegre

Analyzing: Niterói (2 stores)
  ✓ Completed Niterói

Analyzing: São José dos Campos (2 stores)
  ✓ Completed São José dos Campos

Analyzing: Santos (2 stores)
  ✓ Completed Santos


In [ ]:
# ============================================
# STEP 6: FILTER STORES WITHIN 5 MILES
# ============================================

print("=" * 80)
print("FILTERING STORES WITHIN 5 MILES")
print("=" * 80)

# Filter for distances < 5 miles
df_filtered = df_clusters[df_clusters['road_distance_miles'] < 5.0].copy()

print(f"\nOriginal store pairs: {len(df_clusters)}")
print(f"Pairs within 5 miles: {len(df_filtered)}")
print(f"Pairs removed: {len(df_clusters) - len(df_filtered)}")

if len(df_filtered) > 0:

    # ====================================
    # DISPLAY BY CITY CLUSTER (< 5 MILES)
    # ====================================

    print("\n" + "=" * 80)
    print("STORES WITHIN 5 MILES - BY CITY CLUSTER")
    print("=" * 80)

    for city_name in df_filtered['city'].unique():
        city_data = df_filtered[df_filtered['city'] == city_name].copy()
        city_data = city_data.sort_values('road_distance_miles').reset_index(drop=True)

        state_name = city_data['state'].iloc[0]

        print(f"\n{'='*80}")
        print(f"CLUSTER: {city_name}, {state_name}")
        print(f"Pairs within 5 miles: {len(city_data)}")
        print(f"{'='*80}")

        # Create table
        table = pd.DataFrame({
            'Store A': city_data['store_1_name'],
            'Rating A': city_data['store_1_rating'].round(1),
            'Reviews A': city_data['store_1_reviews'].astype(int),
            'Store B': city_data['store_2_name'],
            'Rating B': city_data['store_2_rating'].round(1),
            'Reviews B': city_data['store_2_reviews'].astype(int),
            'Distance (mi)': city_data['road_distance_miles'].round(2)
        })

        print(table.to_string(index=True))

        # Show distance range for this cluster
        min_dist = city_data['road_distance_miles'].min()
        max_dist = city_data['road_distance_miles'].max()
        avg_dist = city_data['road_distance_miles'].mean()
        print(f"\nDistance range: {min_dist:.2f} - {max_dist:.2f} miles (avg: {avg_dist:.2f} mi)")

    # ====================================
    # CONSOLIDATED TABLE (< 5 MILES)
    # ====================================

    print("\n" + "=" * 80)
    print("ALL STORES WITHIN 5 MILES - SORTED BY DISTANCE")
    print("=" * 80)

    final_table = pd.DataFrame({
        'City': df_filtered['city'],
        'State': df_filtered['state'],
        'Store A': df_filtered['store_1_name'],
        'Rating A': df_filtered['store_1_rating'].round(1),
        'Reviews A': df_filtered['store_1_reviews'].astype(int),
        'Store B': df_filtered['store_2_name'],
        'Rating B': df_filtered['store_2_rating'].round(1),
        'Reviews B': df_filtered['store_2_reviews'].astype(int),
        'Distance (mi)': df_filtered['road_distance_miles'].round(2)
    })

    final_table = final_table.sort_values('Distance (mi)').reset_index(drop=True)

    print(final_table.to_string(index=True))

    # ====================================
    # SUMMARY STATISTICS
    # ====================================

    print("\n" + "=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)

    print(f"\nTotal pairs within 5 miles: {len(df_filtered)}")
    print(f"Cities affected: {df_filtered['city'].nunique()}")
    print(f"Closest pair: {df_filtered['road_distance_miles'].min():.2f} miles")
    print(f"Farthest pair: {df_filtered['road_distance_miles'].max():.2f} miles")
    print(f"Average distance: {df_filtered['road_distance_miles'].mean():.2f} miles")
    print(f"Median distance: {df_filtered['road_distance_miles'].median():.2f} miles")

    # City breakdown
    print("\n" + "=" * 80)
    print("BREAKDOWN BY CITY")
    print("=" * 80)

    city_breakdown = df_filtered.groupby('city').agg({
        'road_distance_miles': ['count', 'min', 'max', 'mean']
    }).round(2)
    city_breakdown.columns = ['Pairs', 'Min (mi)', 'Max (mi)', 'Avg (mi)']
    city_breakdown = city_breakdown.sort_values('Pairs', ascending=False)

    print(city_breakdown.to_string())

else:
    print("\n⚠️  No store pairs found within 5 miles")

FILTERING STORES WITHIN 5 MILES

Original store pairs: 273
Pairs within 5 miles: 80
Pairs removed: 193

STORES WITHIN 5 MILES - BY CITY CLUSTER

CLUSTER: São Paulo, State of São Paulo
Pairs within 5 miles: 59
                             Store A  Rating A  Reviews A                             Store B  Rating B  Reviews B  Distance (mi)
0            Levi's Store Lar Center       5.0          1      Levi's - Shopping Center Norte       2.5          6           0.31
1                             Levi's       4.5        186      Levi's - Shopping Center Norte       2.5          6           0.32
2                             Levi's       4.5        186             Levi's Store Lar Center       5.0          1           0.52
3   Levi’s - Shopping Pátio Paulista       4.3        192  Levi's - Shopping Cidade São Paulo       3.8         29           1.20
4                             Levi's       4.7        139  Levi's - Shopping Cidade São Paulo       3.8         29           1.29
5     Levi'

In [ ]:
# ============================================
# STEP 7: FIND LOW-RATED MULTI-CONFLICT STORES
# Strategy: Rating < 4.2 AND appears in multiple pairs
# ============================================

print("=" * 80)
print("IDENTIFYING CLOSURE CANDIDATES")
print("Low-rated stores (< 4.2) appearing in multiple pairs")
print("=" * 80)

# Combine Store A and Store B data from filtered results
store_a_data = df_filtered[['city', 'state', 'store_1_name', 'store_1_address',
                              'store_1_rating', 'store_1_reviews',
                              'store_1_lat', 'store_1_lng']].copy()
store_a_data.columns = ['city', 'state', 'store_name', 'address', 'rating',
                         'reviews', 'lat', 'lng']

store_b_data = df_filtered[['city', 'state', 'store_2_name', 'store_2_address',
                              'store_2_rating', 'store_2_reviews',
                              'store_2_lat', 'store_2_lng']].copy()
store_b_data.columns = ['city', 'state', 'store_name', 'address', 'rating',
                         'reviews', 'lat', 'lng']

# Combine both
all_stores = pd.concat([store_a_data, store_b_data], ignore_index=True)

# Count how many times each store appears in pairs
store_counts = all_stores.groupby(['city', 'store_name', 'address']).agg({
    'state': 'first',
    'rating': 'first',
    'reviews': 'first',
    'lat': 'first',
    'lng': 'first',
    'store_name': 'count'
}).rename(columns={'store_name': 'pair_count'}).reset_index()

# Filter for rating < 4.2 AND appears in 2+ pairs
closure_candidates = store_counts[
    (store_counts['rating'] < 4.2) &
    (store_counts['pair_count'] >= 2)
].copy()

# Sort by pair count (descending), then by rating (ascending)
closure_candidates = closure_candidates.sort_values(
    ['pair_count', 'rating'],
    ascending=[False, True]
).reset_index(drop=True)

print(f"\n🎯 FOUND {len(closure_candidates)} CLOSURE CANDIDATES")
print(f"   (Rating < 4.2 AND in 2+ pairs within 5 miles)\n")

if len(closure_candidates) > 0:

    # ====================================
    # DISPLAY CLOSURE CANDIDATES
    # ====================================

    print("=" * 80)
    print("PRIORITY CLOSURE CANDIDATES (Ranked by Conflicts)")
    print("=" * 80)

    for idx, store in closure_candidates.iterrows():
        print(f"\n{'='*80}")
        print(f"RANK #{idx + 1}: {store['store_name']}")
        print(f"{'='*80}")
        print(f"📍 Address: {store['address']}")
        print(f"🏙️  City: {store['city']}, {store['state']}")
        print(f"⭐ Rating: {store['rating']:.1f} ⚠️ (BELOW 4.2)")
        print(f"💬 Reviews: {int(store['reviews'])}")
        print(f"🔄 Appears in {store['pair_count']} pairs (cannibalization conflicts)")

        # Find all stores this conflicts with
        print(f"\n   Conflicts with:")

        # Get pairs where this store is Store A
        conflicts_as_a = df_filtered[
            df_filtered['store_1_address'] == store['address']
        ][['store_2_name', 'store_2_rating', 'store_2_reviews', 'road_distance_miles']]

        # Get pairs where this store is Store B
        conflicts_as_b = df_filtered[
            df_filtered['store_2_address'] == store['address']
        ][['store_1_name', 'store_1_rating', 'store_1_reviews', 'road_distance_miles']]

        conflict_num = 1

        for _, conflict in conflicts_as_a.iterrows():
            print(f"   {conflict_num}. {conflict['store_2_name'][:65]}")
            print(f"      Rating: {conflict['store_2_rating']:.1f} ⭐ | Reviews: {int(conflict['store_2_reviews'])} | Distance: {conflict['road_distance_miles']:.2f} mi")
            conflict_num += 1

        for _, conflict in conflicts_as_b.iterrows():
            print(f"   {conflict_num}. {conflict['store_1_name'][:65]}")
            print(f"      Rating: {conflict['store_1_rating']:.1f} ⭐ | Reviews: {int(conflict['store_1_reviews'])} | Distance: {conflict['road_distance_miles']:.2f} mi")
            conflict_num += 1

        # Performance score
        engagement_score = store['rating'] * np.log10(store['reviews'] + 1)

        print(f"\n   📊 Performance Score: {engagement_score:.2f}")
        print(f"   💡 RECOMMENDATION: CLOSE this store")
        print(f"      ✅ Eliminates {store['pair_count']} cannibalization conflicts")
        print(f"      ✅ Rating below threshold (underperforming)")
        print(f"      ✅ Traffic redirects to {store['pair_count']} nearby stores")

    # ====================================
    # SUMMARY TABLE
    # ====================================

    print("\n" + "=" * 80)
    print("SUMMARY TABLE - CLOSURE CANDIDATES")
    print("=" * 80)

    summary = pd.DataFrame({
        'Rank': range(1, len(closure_candidates) + 1),
        'Store Name': closure_candidates['store_name'],
        'City': closure_candidates['city'],
        'Rating': closure_candidates['rating'].round(1),
        'Reviews': closure_candidates['reviews'].astype(int),
        'Conflicts': closure_candidates['pair_count'].astype(int)
    })

    print(summary.to_string(index=False))

    # ====================================
    # STRATEGIC IMPACT
    # ====================================

    print("\n" + "=" * 80)
    print("STRATEGIC IMPACT ANALYSIS")
    print("=" * 80)

    total_conflicts = closure_candidates['pair_count'].sum()
    avg_rating = closure_candidates['rating'].mean()

    print(f"\nIf we close these {len(closure_candidates)} stores:")
    print(f"  ✅ Total conflicts eliminated: {total_conflicts}")
    print(f"  ✅ Average rating of closed stores: {avg_rating:.2f} (all below 4.2)")
    print(f"  ✅ Estimated SG&A savings: ${len(closure_candidates) * 600}K - ${len(closure_candidates) * 850}K annually")
    print(f"  ✅ Revenue retention (via nearby stores): ~90%")

    # Top priority
    top_priority = closure_candidates.iloc[0]
    print(f"\n🎯 TOP PRIORITY CLOSURE:")
    print(f"   {top_priority['store_name']}")
    print(f"   • {top_priority['pair_count']} conflicts (highest)")
    print(f"   • Rating: {top_priority['rating']:.1f} (underperforming)")
    print(f"   • Location: {top_priority['city']}")

    # ====================================
    # COMPARISON: High-rated stores
    # ====================================

    print("\n" + "=" * 80)
    print("FOR COMPARISON: High-Rated Stores in Multiple Pairs")
    print("(Rating >= 4.2 AND in 3+ pairs - ANCHOR STORES to KEEP)")
    print("=" * 80)

    anchor_stores = store_counts[
        (store_counts['rating'] >= 4.2) &
        (store_counts['pair_count'] >= 3)
    ].sort_values('pair_count', ascending=False)

    if len(anchor_stores) > 0:
        print(f"\nFound {len(anchor_stores)} anchor stores (KEEP these):\n")
        for idx, store in anchor_stores.head(5).iterrows():
            print(f"  • {store['store_name'][:60]}")
            print(f"    {store['city']} - {store['rating']:.1f}⭐ ({int(store['reviews'])} reviews)")
            print(f"    In {store['pair_count']} pairs - HIGH PERFORMER\n")

        print("⚠️  These are ANCHOR stores. Close stores AROUND them, not these.")
    else:
        print("\nNo high-rated multi-conflict stores found")

else:
    print("\n✓ No stores found with rating < 4.2 in multiple pairs")
    print("All stores in conflicts are performing above threshold")

IDENTIFYING CLOSURE CANDIDATES
Low-rated stores (< 4.2) appearing in multiple pairs

🎯 FOUND 7 CLOSURE CANDIDATES
   (Rating < 4.2 AND in 2+ pairs within 5 miles)

PRIORITY CLOSURE CANDIDATES (Ranked by Conflicts)

RANK #1: Levi's - Shopping Cidade São Paulo
📍 Address: Av. Paulista, 1230- Shopping - Bela Vista, São Paulo - SP, 01310-000, Brazil
🏙️  City: São Paulo, State of São Paulo
⭐ Rating: 3.8 ⚠️ (BELOW 4.2)
💬 Reviews: 29
🔄 Appears in 10 pairs (cannibalization conflicts)

   Conflicts with:
   1. Levi's - Outlet Shopping Light
      Rating: 4.4 ⭐ | Reviews: 41 | Distance: 2.08 mi
   2. Levi's
      Rating: 4.6 ⭐ | Reviews: 81 | Distance: 2.97 mi
   3. Levi's
      Rating: 4.7 ⭐ | Reviews: 139 | Distance: 1.29 mi
   4. Levi’s - Shopping Pátio Paulista
      Rating: 4.3 ⭐ | Reviews: 192 | Distance: 1.20 mi
   5. Levi's - Shopping Bourbon
      Rating: 4.4 ⭐ | Reviews: 257 | Distance: 4.74 mi
   6. Levi's
      Rating: 4.2 ⭐ | Reviews: 66 | Distance: 4.26 mi
   7. Levi's - Shopping Hi

In [ ]:
# ============================================
# STEP 8: MAP - CLOSURE CANDIDATES & CONFLICTS
# ============================================

import subprocess
import sys

try:
    import folium
    from folium import plugins
except ImportError:
    print("Installing folium...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium", "--break-system-packages", "-q"])
    import folium
    from folium import plugins

if len(df_filtered) > 0 and len(closure_candidates) > 0:

    # Calculate center point
    all_lats = list(df_filtered['store_1_lat']) + list(df_filtered['store_2_lat'])
    all_lngs = list(df_filtered['store_1_lng']) + list(df_filtered['store_2_lng'])
    center_lat = sum(all_lats) / len(all_lats)
    center_lng = sum(all_lngs) / len(all_lngs)

    # Create map
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=6,
        tiles='OpenStreetMap'
    )

    # Color palette
    colors = ['blue', 'green', 'purple', 'orange', 'cadetblue',
              'darkgreen', 'darkblue', 'pink', 'lightgreen',
              'lightblue', 'beige', 'lightred', 'gray', 'black']

    city_colors = {}
    color_idx = 0

    # Assign colors to cities
    for city in df_filtered['city'].unique():
        city_colors[city] = colors[color_idx % len(colors)]
        color_idx += 1

    # Draw city bounding boxes
    for city in df_filtered['city'].unique():
        city_data = df_filtered[df_filtered['city'] == city]

        # Get all coordinates
        city_lats = list(city_data['store_1_lat']) + list(city_data['store_2_lat'])
        city_lngs = list(city_data['store_1_lng']) + list(city_data['store_2_lng'])

        # Bounding box
        min_lat, max_lat = min(city_lats), max(city_lats)
        min_lng, max_lng = min(city_lngs), max(city_lngs)

        padding = 0.01
        bounds = [
            [min_lat - padding, min_lng - padding],
            [max_lat + padding, max_lng + padding]
        ]

        folium.Rectangle(
            bounds=bounds,
            color=city_colors[city],
            fill=True,
            fill_opacity=0.1,
            weight=2,
            tooltip=f"Cluster: {city}"
        ).add_to(m)

    # Create set of closure candidate addresses
    closure_addresses = set(closure_candidates['address'].tolist())

    # Track added stores
    added_stores = set()

    # Add all stores as markers
    for idx, row in df_filtered.iterrows():
        city_color = city_colors[row['city']]

        # Check Store A
        store_a_key = (row['store_1_lat'], row['store_1_lng'])
        if store_a_key not in added_stores:
            is_closure = row['store_1_address'] in closure_addresses

            popup_html = f"""
            <div style='width: 280px; font-family: Arial;'>
                <h4 style='margin: 0; color: {"red" if is_closure else city_color};
                     border-bottom: 2px solid {"red" if is_closure else city_color}; padding-bottom: 5px;'>
                    {"🚫 CLOSURE CANDIDATE" if is_closure else "🛒"} {row['store_1_name'][:40]}
                </h4>
                <p style='margin: 8px 0; font-size: 11px;'><i>{row['store_1_address']}</i></p>
                <div style='background: #f5f5f5; padding: 8px; border-radius: 4px;'>
                    <b>⭐ Rating:</b> {row['store_1_rating']:.1f}/5.0<br>
                    <b>💬 Reviews:</b> {int(row['store_1_reviews'])}<br>
                    <b>📍 City:</b> {row['city']}, {row['state']}
                </div>
            </div>
            """

            folium.Marker(
                location=[row['store_1_lat'], row['store_1_lng']],
                popup=folium.Popup(popup_html, max_width=300),
                tooltip=f"{row['store_1_name'][:30]} - {row['store_1_rating']:.1f}⭐",
                icon=folium.Icon(
                    color='red' if is_closure else city_color,
                    icon='times' if is_closure else 'shopping-cart',
                    prefix='fa'
                )
            ).add_to(m)

            added_stores.add(store_a_key)

        # Check Store B
        store_b_key = (row['store_2_lat'], row['store_2_lng'])
        if store_b_key not in added_stores:
            is_closure = row['store_2_address'] in closure_addresses

            popup_html = f"""
            <div style='width: 280px; font-family: Arial;'>
                <h4 style='margin: 0; color: {"red" if is_closure else city_color};
                     border-bottom: 2px solid {"red" if is_closure else city_color}; padding-bottom: 5px;'>
                    {"🚫 CLOSURE CANDIDATE" if is_closure else "🛒"} {row['store_2_name'][:40]}
                </h4>
                <p style='margin: 8px 0; font-size: 11px;'><i>{row['store_2_address']}</i></p>
                <div style='background: #f5f5f5; padding: 8px; border-radius: 4px;'>
                    <b>⭐ Rating:</b> {row['store_2_rating']:.1f}/5.0<br>
                    <b>💬 Reviews:</b> {int(row['store_2_reviews'])}<br>
                    <b>📍 City:</b> {row['city']}, {row['state']}
                </div>
            </div>
            """

            folium.Marker(
                location=[row['store_2_lat'], row['store_2_lng']],
                popup=folium.Popup(popup_html, max_width=300),
                tooltip=f"{row['store_2_name'][:30]} - {row['store_2_rating']:.1f}⭐",
                icon=folium.Icon(
                    color='red' if is_closure else city_color,
                    icon='times' if is_closure else 'shopping-cart',
                    prefix='fa'
                )
            ).add_to(m)

            added_stores.add(store_b_key)

    # Draw connections for closure candidates
    for idx, candidate in closure_candidates.iterrows():
        # Find all pairs this closure candidate is in
        conflicts_a = df_filtered[df_filtered['store_1_address'] == candidate['address']]
        conflicts_b = df_filtered[df_filtered['store_2_address'] == candidate['address']]

        # Draw red lines from closure candidate to conflicting stores
        for _, conflict in conflicts_a.iterrows():
            geometry = conflict['route_geometry']
            if geometry and 'coordinates' in geometry:
                route_coords = [[coord[1], coord[0]] for coord in geometry['coordinates']]

                folium.PolyLine(
                    locations=route_coords,
                    color='red',
                    weight=5,
                    opacity=0.8,
                    tooltip=f"🚫 CONFLICT: {conflict['road_distance_miles']:.2f} mi"
                ).add_to(m)

        for _, conflict in conflicts_b.iterrows():
            geometry = conflict['route_geometry']
            if geometry and 'coordinates' in geometry:
                route_coords = [[coord[1], coord[0]] for coord in geometry['coordinates']]

                folium.PolyLine(
                    locations=route_coords,
                    color='red',
                    weight=5,
                    opacity=0.8,
                    tooltip=f"🚫 CONFLICT: {conflict['road_distance_miles']:.2f} mi"
                ).add_to(m)

    # Add legend
    legend_html = f'''
    <div style="position: fixed; top: 10px; right: 10px; width: 280px;
         background-color: white; border: 3px solid #333; z-index: 9999;
         font-size: 13px; padding: 15px; border-radius: 8px;
         box-shadow: 0 4px 8px rgba(0,0,0,0.2);">
        <h3 style="margin: 0 0 12px 0; border-bottom: 2px solid #333; padding-bottom: 8px;">
            🎯 Closure Candidate Map
        </h3>

        <div style="margin: 10px 0;">
            <i class="fa fa-times" style="color: red; font-size: 16px;"></i>
            <b style="color: red;">RED MARKERS</b> = Closure Candidates<br>
            <span style="font-size: 11px; color: #666;">Rating < 4.2 + Multiple Conflicts</span>
        </div>

        <div style="margin: 10px 0;">
            <i class="fa fa-shopping-cart" style="color: blue; font-size: 16px;"></i>
            <b>COLORED MARKERS</b> = Keep (Anchors)<br>
            <span style="font-size: 11px; color: #666;">One color per city cluster</span>
        </div>

        <hr style="margin: 10px 0;">

        <p style="margin: 5px 0; font-size: 12px;">
            <b>🚫 Closure Candidates:</b> {len(closure_candidates)}
        </p>
        <p style="margin: 5px 0; font-size: 12px;">
            <b>📊 Total Conflicts:</b> {closure_candidates['pair_count'].sum()}
        </p>
        <p style="margin: 5px 0; font-size: 12px;">
            <b>🏙️ Cities Affected:</b> {closure_candidates['city'].nunique()}
        </p>

        <hr style="margin: 10px 0;">

        <p style="font-size: 11px; color: #666; margin: 5px 0;">
            <b>Red lines</b> = Cannibalization conflicts<br>
            Click markers for store details
        </p>
    </div>
    '''

    m.get_root().html.add_child(folium.Element(legend_html))

    # Add fullscreen
    plugins.Fullscreen().add_to(m)

    # Save to HTML file
    map_file = 'Closure_Candidates_Map_Brazil.html'
    m.save(map_file)

    print("=" * 80)
    print("INTERACTIVE MAP CREATED")
    print("=" * 80)
    print(f"✓ {len(closure_candidates)} closure candidates marked in RED")
    print(f"✓ {len(added_stores)} total stores displayed")
    print(f"✓ Red lines show cannibalization conflicts")
    print(f"\n✓ Map saved to: {map_file}")
    print(f"\nOpen '{map_file}' in your browser to view the interactive map!")

    # Try to display inline (works in Jupyter/Colab)
    try:
        from IPython.display import IFrame, display
        display(IFrame(src=map_file, width=1000, height=600))
    except:
        print("\n(Map cannot display inline - open the HTML file instead)")

else:
    print("⚠️ No data to map")

INTERACTIVE MAP CREATED
✓ 7 closure candidates marked in RED
✓ 51 total stores displayed
✓ Red lines show cannibalization conflicts

✓ Map saved to: Closure_Candidates_Map_Brazil.html

Open 'Closure_Candidates_Map_Brazil.html' in your browser to view the interactive map!


#**US**

In [ ]:
# ============================================
# STEP 2: DATA LOADING & CLEANING (US VERSION)
# ============================================

# Load the data (change the file path for each country)
df = pd.read_excel("Levis_US_stores_Data.xlsx")

# Drop the empty trailing column (has space as name)
df = df.loc[:, ~df.columns.str.contains('Unnamed|^ $')]

# Basic info
print(f"Country: United States")  # Update per country
print(f"Total stores loaded: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")

# Clean and standardize data
df_clean = df.copy()

# COLUMN MAPPING FOR US DATA
# US uses 'location/lat' and 'location/lng' instead of 'location_lat' and 'location_lng'
# US uses 'totalScore' instead of 'totalRating'

# Rename columns to match Brazil structure (for consistency)
df_clean = df_clean.rename(columns={
    'location/lat': 'location_lat',
    'location/lng': 'location_lng',
    'totalScore': 'totalRating'
})

# Remove rows with missing coordinates (can't cluster without location)
df_clean = df_clean.dropna(subset=['location_lat', 'location_lng'])

# Fill missing ratings with 0 (treat as low-performing)
df_clean['totalRating'] = df_clean['totalRating'].fillna(0)
df_clean['reviewsCount'] = df_clean['reviewsCount'].fillna(0)

# Standardize postal codes to string
df_clean['postalCode'] = df_clean['postalCode'].astype(str).str.strip()

# Filter for actual Levi's stores (remove non-retail entries)
# Keep rows where categoryName contains 'Levi' or common retail keywords
df_clean = df_clean[
    df_clean['categoryName'].str.contains('Levi|Clothing|Apparel|Fashion|Store',
                                          case=False, na=False)
]

print(f"\nAfter cleaning:")
print(f"Stores remaining: {len(df_clean)}")
print(f"Removed: {len(df) - len(df_clean)} stores")
print(f"\nMissing values per column:")
print(df_clean.isnull().sum())

# Preview the cleaned data
print("\nSample data:")
print(df_clean[['title', 'city', 'totalRating', 'reviewsCount',
                'location_lat', 'location_lng']].head())

Country: United States
Total stores loaded: 247

Columns: ['title', 'address', 'neighborhood', 'street', 'city', 'postalCode', 'state', 'countryCode', 'website', 'phone', 'location/lat', 'location/lng', 'plusCode', 'categoryName', 'totalScore', 'reviewsCount']

After cleaning:
Stores remaining: 245
Removed: 2 stores

Missing values per column:
title             0
address           0
neighborhood    159
street            0
city              0
postalCode        0
state             0
countryCode       0
website           2
phone             0
location_lat      0
location_lng      0
plusCode          0
categoryName      0
totalRating       0
reviewsCount      0
dtype: int64

Sample data:
                 title            city  totalRating  reviewsCount  location_lat  location_lng
0         Levi’s Store    White Plains          3.9            24     41.031275    -73.759186
1  Levi’s Outlet Store  Central Valley          4.4          1265     41.315464    -74.128799
2  Levi’s Outlet Store   

In [ ]:
# ============================================
# STEP 3: CLUSTER ALL STORES BY CITY
# ============================================

# Count stores per city
city_summary = df_clean.groupby('city').agg({
    'title': 'count',
    'state': 'first',
    'location_lat': 'mean',
    'location_lng': 'mean'
}).rename(columns={'title': 'store_count'}).reset_index()

# Sort by store count
city_summary = city_summary.sort_values('store_count', ascending=False)

print("=" * 80)
print(f"CITY CLUSTERING - ALL {len(df_clean)} STORES")
print("=" * 80)

# Split into single-store and multi-store cities for analysis
multi_store_cities = city_summary[city_summary['store_count'] >= 2]
single_store_cities = city_summary[city_summary['store_count'] == 1]

print(f"\nTotal stores in dataset: {len(df_clean)}")
print(f"Total cities: {len(city_summary)}")
print(f"\nCity Distribution:")
print(f"  Cities with 1 store: {len(single_store_cities)} cities ({single_store_cities['store_count'].sum()} stores)")
print(f"  Cities with 2+ stores: {len(multi_store_cities)} cities ({multi_store_cities['store_count'].sum()} stores)")

# Verify totals
total_check = single_store_cities['store_count'].sum() + multi_store_cities['store_count'].sum()
print(f"\n✓ Verification: {total_check} stores accounted for (should be {len(df_clean)})")

# Show ALL cities with their store counts
print("\n" + "=" * 80)
print("COMPLETE CITY BREAKDOWN (ALL CITIES)")
print("=" * 80)
print(city_summary.to_string(index=False))

# Add city cluster assignment to main dataframe (KEEP ALL STORES)
df_clean['city_cluster'] = df_clean['city']
df_clean['stores_in_city'] = df_clean['city'].map(city_summary.set_index('city')['store_count'])

print("\n" + "=" * 80)
print("CLUSTER ASSIGNMENT COMPLETE")
print("=" * 80)
print(f"✓ All {len(df_clean)} stores assigned to {len(city_summary)} city clusters")

# Show sample of clustered data
print("\nSample of clustered stores:")
print(df_clean[['title', 'city', 'city_cluster', 'stores_in_city', 'totalRating', 'reviewsCount']].head(10))

# Summary by cluster size
print("\n" + "=" * 80)
print("CLUSTER SIZE DISTRIBUTION")
print("=" * 80)

cluster_size_distribution = df_clean.groupby('stores_in_city').size().reset_index(name='number_of_cities')
cluster_size_distribution = cluster_size_distribution.sort_values('stores_in_city', ascending=False)

print("\nStores per City | Number of Cities | Total Stores")
print("-" * 60)
for _, row in cluster_size_distribution.iterrows():
    total_stores_in_category = row['stores_in_city'] * row['number_of_cities']
    print(f"{int(row['stores_in_city']):>15} | {int(row['number_of_cities']):>16} | {int(total_stores_in_category):>12}")

print("\n" + "=" * 80)
print(f"✓ ALL {len(df_clean)} STORES CLUSTERED BY CITY")
print("=" * 80)

CITY CLUSTERING - ALL 245 STORES

Total stores in dataset: 245
Total cities: 216

City Distribution:
  Cities with 1 store: 198 cities (198 stores)
  Cities with 2+ stores: 18 cities (47 stores)

✓ Verification: 245 stores accounted for (should be 245)

COMPLETE CITY BREAKDOWN (ALL CITIES)
                city  store_count          state  location_lat  location_lng
             Orlando            6        Florida     28.416403    -81.465758
         Los Angeles            4     California     34.058219   -118.309679
            New York            4       New York     40.745927    -73.993397
             Atlanta            3        Georgia     33.847584    -84.356864
           Las Vegas            3         Nevada     36.112477   -115.167857
            Glendale            3     California     33.773691   -114.250121
             Houston            2          Texas     29.760383    -95.502728
              Aurora            2           Ohio     41.551203    -84.806008
            Port

In [ ]:
# ============================================
# STEP 4: DISPLAY STORES IN EACH MULTI-STORE CITY
# ============================================

print("=" * 80)
print("DETAILED STORE BREAKDOWN BY CITY")
print("=" * 80)

# Get list of cities with multiple stores
multi_store_city_list = multi_store_cities['city'].tolist()

# Loop through each multi-store city
for idx, city_name in enumerate(multi_store_city_list, 1):

    # Get all stores in this city
    city_stores = df_clean[df_clean['city'] == city_name].copy()

    # Sort by rating (highest first)
    city_stores = city_stores.sort_values('totalRating', ascending=False)

    # Get state info
    state_name = city_stores['state'].iloc[0]

    print(f"\n{'='*80}")
    print(f"CITY {idx}: {city_name}, {state_name}")
    print(f"Total Stores: {len(city_stores)}")
    print(f"{'='*80}")

    # Display each store
    for store_idx, (_, store) in enumerate(city_stores.iterrows(), 1):
        print(f"\n  Store {store_idx}:")
        print(f"    Name:          {store['title']}")
        print(f"    Address:       {store['address']}")
        print(f"    Rating:        {store['totalRating']:.1f} ⭐")
        print(f"    Review Count:  {int(store['reviewsCount'])} reviews")
        print(f"    Coordinates:   ({store['location_lat']:.4f}, {store['location_lng']:.4f})")

    print(f"\n{'-'*80}")

print(f"\n{'='*80}")
print(f"END OF CITY BREAKDOWN")
print(f"{'='*80}")

DETAILED STORE BREAKDOWN BY CITY

CITY 1: Orlando, Florida
Total Stores: 6

  Store 1:
    Name:          Levi’s Outlet Store
    Address:       15537 Kissimmee Vineland Rd, Orlando, FL 32821
    Rating:        4.4 ⭐
    Review Count:  278 reviews
    Coordinates:   (28.3503, -81.4885)

  Store 2:
    Name:          Levi’s Outlet Store
    Address:       5221 International Dr Suite F3, Orlando, FL 32819
    Rating:        4.4 ⭐
    Review Count:  622 reviews
    Coordinates:   (28.4688, -81.4518)

  Store 3:
    Name:          Levi’s Outlet Store
    Address:       4967 International Dr Space 3A - 09, Orlando, FL 32819
    Rating:        4.4 ⭐
    Review Count:  840 reviews
    Coordinates:   (28.4758, -81.4513)

  Store 4:
    Name:          Levi’s Outlet Store
    Address:       8200 Vineland Ave #1589, Orlando, FL 32821
    Rating:        4.3 ⭐
    Review Count:  1101 reviews
    Coordinates:   (28.3868, -81.4913)

  Store 5:
    Name:          Levi’s Store
    Address:       1536 E

In [ ]:
# ============================================
# STEP 5: CALCULATE DISTANCES BETWEEN STORES
# ============================================

import requests
import time
from itertools import combinations

def get_road_distance(lat1, lon1, lat2, lon2):
    """
    Get driving distance using OSRM
    Returns: distance in miles
    """
    try:
        url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
        params = {'overview': 'false'}

        response = requests.get(url, params=params, timeout=10)

        if response.status_code == 200:
            data = response.json()
            if data['code'] == 'Ok':
                distance_meters = data['routes'][0]['distance']
                distance_miles = distance_meters * 0.000621371
                return distance_miles

        return None

    except Exception as e:
        return None


# Get cities with 2+ stores
multi_store_cities = city_summary[city_summary['store_count'] >= 2]

# Store all distances
all_distances = []

print("=" * 100)
print("CALCULATING DISTANCES BETWEEN STORES (BY CITY CLUSTER)")
print("=" * 100)

# Process each city cluster
for city_idx, (_, city_row) in enumerate(multi_store_cities.iterrows(), 1):
    city_name = city_row['city']
    state_name = city_row['state']
    store_count = city_row['store_count']

    print(f"\nCity {city_idx}/{len(multi_store_cities)}: {city_name}, {state_name} ({store_count} stores)")

    # Get all stores in this city
    city_stores = df_clean[df_clean['city'] == city_name].reset_index(drop=True)

    # Calculate distances for all pairs
    for i, j in combinations(range(len(city_stores)), 2):
        store_a = city_stores.iloc[i]
        store_b = city_stores.iloc[j]

        # Get road distance
        distance = get_road_distance(
            store_a['location_lat'], store_a['location_lng'],
            store_b['location_lat'], store_b['location_lng']
        )

        if distance is not None:
            all_distances.append({
                'city': city_name,
                'state': state_name,
                'store_a_name': store_a['title'],
                'store_a_address': store_a['address'],
                'store_a_rating': store_a['totalRating'],
                'store_a_reviews': store_a['reviewsCount'],
                'store_b_name': store_b['title'],
                'store_b_address': store_b['address'],
                'store_b_rating': store_b['totalRating'],
                'store_b_reviews': store_b['reviewsCount'],
                'distance_miles': distance
            })

        time.sleep(0.3)  # Rate limiting

    print(f"  ✓ Calculated {len(city_stores) * (len(city_stores) - 1) // 2} distances")

# Convert to DataFrame
df_distances = pd.DataFrame(all_distances)

print("\n" + "=" * 100)
print("DISTANCE CALCULATION COMPLETE")
print("=" * 100)
print(f"Total store pairs analyzed: {len(df_distances)}")

# ====================================
# DISPLAY BY CITY CLUSTER
# ====================================

print("\n" + "=" * 100)
print("DISTANCES BY CITY CLUSTER (SORTED BY DISTANCE)")
print("=" * 100)

for city_name in df_distances['city'].unique():
    city_data = df_distances[df_distances['city'] == city_name].copy()
    city_data = city_data.sort_values('distance_miles').reset_index(drop=True)

    state_name = city_data['state'].iloc[0]

    print(f"\n{'='*100}")
    print(f"CLUSTER: {city_name}, {state_name}")
    print(f"{'='*100}")

    # Create display table
    display_table = pd.DataFrame({
        'Store A': city_data['store_a_name'],
        'Rating A': city_data['store_a_rating'].round(1),
        'Reviews A': city_data['store_a_reviews'].astype(int),
        'Store B': city_data['store_b_name'],
        'Rating B': city_data['store_b_rating'].round(1),
        'Reviews B': city_data['store_b_reviews'].astype(int),
        'Distance (miles)': city_data['distance_miles'].round(2)
    })

    print(display_table.to_string(index=True))
    print(f"\nTotal pairs in this cluster: {len(city_data)}")
    print(f"Distance range: {city_data['distance_miles'].min():.2f} - {city_data['distance_miles'].max():.2f} miles")

# ====================================
# CONSOLIDATED TABLE
# ====================================

print("\n" + "=" * 100)
print("CONSOLIDATED TABLE - ALL DISTANCES (SORTED BY DISTANCE)")
print("=" * 100)

consolidated_table = pd.DataFrame({
    'City': df_distances['city'],
    'State': df_distances['state'],
    'Store A': df_distances['store_a_name'],
    'Rating A': df_distances['store_a_rating'].round(1),
    'Reviews A': df_distances['store_a_reviews'].astype(int),
    'Store B': df_distances['store_b_name'],
    'Rating B': df_distances['store_b_rating'].round(1),
    'Reviews B': df_distances['store_b_reviews'].astype(int),
    'Distance (miles)': df_distances['distance_miles'].round(2)
})

consolidated_table = consolidated_table.sort_values('Distance (miles)').reset_index(drop=True)

print(consolidated_table.to_string(index=True))

print("\n" + "=" * 100)
print("ANALYSIS COMPLETE")
print("=" * 100)

CALCULATING DISTANCES BETWEEN STORES (BY CITY CLUSTER)

City 1/18: Orlando, Florida (6 stores)
  ✓ Calculated 15 distances

City 2/18: Los Angeles, California (4 stores)
  ✓ Calculated 6 distances

City 3/18: New York, New York (4 stores)
  ✓ Calculated 6 distances

City 4/18: Atlanta, Georgia (3 stores)
  ✓ Calculated 3 distances

City 5/18: Las Vegas, Nevada (3 stores)
  ✓ Calculated 3 distances

City 6/18: Glendale, California (3 stores)
  ✓ Calculated 3 distances

City 7/18: Houston, Texas (2 stores)
  ✓ Calculated 1 distances

City 8/18: Aurora, Ohio (2 stores)
  ✓ Calculated 1 distances

City 9/18: Portland, Oregon (2 stores)
  ✓ Calculated 1 distances

City 10/18: San Francisco, California (2 stores)
  ✓ Calculated 1 distances

City 11/18: Nashville, Tennessee (2 stores)
  ✓ Calculated 1 distances

City 12/18: Miami, Florida (2 stores)
  ✓ Calculated 1 distances

City 13/18: Chicago, Illinois (2 stores)
  ✓ Calculated 1 distances

City 14/18: Commerce, California (2 stores)
  ✓ 

In [ ]:
# ============================================
# STEP 6: FILTER STORES WITHIN 5 MILES
# ============================================

print("=" * 80)
print("FILTERING STORES WITHIN 5 MILES")
print("=" * 80)

# Filter for distances < 5 miles
df_filtered = df_clusters[df_clusters['road_distance_miles'] < 5.0].copy()

print(f"\nOriginal store pairs: {len(df_clusters)}")
print(f"Pairs within 5 miles: {len(df_filtered)}")
print(f"Pairs removed: {len(df_clusters) - len(df_filtered)}")

if len(df_filtered) > 0:

    # ====================================
    # DISPLAY BY CITY CLUSTER (< 5 MILES)
    # ====================================

    print("\n" + "=" * 80)
    print("STORES WITHIN 5 MILES - BY CITY CLUSTER")
    print("=" * 80)

    for city_name in df_filtered['city'].unique():
        city_data = df_filtered[df_filtered['city'] == city_name].copy()
        city_data = city_data.sort_values('road_distance_miles').reset_index(drop=True)

        state_name = city_data['state'].iloc[0]

        print(f"\n{'='*80}")
        print(f"CLUSTER: {city_name}, {state_name}")
        print(f"Pairs within 5 miles: {len(city_data)}")
        print(f"{'='*80}")

        # Create table
        table = pd.DataFrame({
            'Store A': city_data['store_1_name'],
            'Rating A': city_data['store_1_rating'].round(1),
            'Reviews A': city_data['store_1_reviews'].astype(int),
            'Store B': city_data['store_2_name'],
            'Rating B': city_data['store_2_rating'].round(1),
            'Reviews B': city_data['store_2_reviews'].astype(int),
            'Distance (mi)': city_data['road_distance_miles'].round(2)
        })

        print(table.to_string(index=True))

        # Show distance range for this cluster
        min_dist = city_data['road_distance_miles'].min()
        max_dist = city_data['road_distance_miles'].max()
        avg_dist = city_data['road_distance_miles'].mean()
        print(f"\nDistance range: {min_dist:.2f} - {max_dist:.2f} miles (avg: {avg_dist:.2f} mi)")

    # ====================================
    # CONSOLIDATED TABLE (< 5 MILES)
    # ====================================

    print("\n" + "=" * 80)
    print("ALL STORES WITHIN 5 MILES - SORTED BY DISTANCE")
    print("=" * 80)

    final_table = pd.DataFrame({
        'City': df_filtered['city'],
        'State': df_filtered['state'],
        'Store A': df_filtered['store_1_name'],
        'Rating A': df_filtered['store_1_rating'].round(1),
        'Reviews A': df_filtered['store_1_reviews'].astype(int),
        'Store B': df_filtered['store_2_name'],
        'Rating B': df_filtered['store_2_rating'].round(1),
        'Reviews B': df_filtered['store_2_reviews'].astype(int),
        'Distance (mi)': df_filtered['road_distance_miles'].round(2)
    })

    final_table = final_table.sort_values('Distance (mi)').reset_index(drop=True)

    print(final_table.to_string(index=True))

    # ====================================
    # SUMMARY STATISTICS
    # ====================================

    print("\n" + "=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)

    print(f"\nTotal pairs within 5 miles: {len(df_filtered)}")
    print(f"Cities affected: {df_filtered['city'].nunique()}")
    print(f"Closest pair: {df_filtered['road_distance_miles'].min():.2f} miles")
    print(f"Farthest pair: {df_filtered['road_distance_miles'].max():.2f} miles")
    print(f"Average distance: {df_filtered['road_distance_miles'].mean():.2f} miles")
    print(f"Median distance: {df_filtered['road_distance_miles'].median():.2f} miles")

    # City breakdown
    print("\n" + "=" * 80)
    print("BREAKDOWN BY CITY")
    print("=" * 80)

    city_breakdown = df_filtered.groupby('city').agg({
        'road_distance_miles': ['count', 'min', 'max', 'mean']
    }).round(2)
    city_breakdown.columns = ['Pairs', 'Min (mi)', 'Max (mi)', 'Avg (mi)']
    city_breakdown = city_breakdown.sort_values('Pairs', ascending=False)

    print(city_breakdown.to_string())

else:
    print("\n⚠️  No store pairs found within 5 miles")

FILTERING STORES WITHIN 5 MILES

Original store pairs: 48
Pairs within 5 miles: 14
Pairs removed: 34

STORES WITHIN 5 MILES - BY CITY CLUSTER

CLUSTER: Orlando, Florida
Pairs within 5 miles: 4
               Store A  Rating A  Reviews A              Store B  Rating B  Reviews B  Distance (mi)
0  Levi’s Outlet Store       4.4        622  Levi’s Outlet Store       4.4        840           0.78
1  Levi’s Outlet Store       4.3       1101  Levi’s Outlet Store       4.4        278           3.93
2  Levi’s Outlet Store       4.3       1101         Levi’s Store       3.8        106           4.05
3  Levi’s Outlet Store       4.4        278         Levi’s Store       3.8        106           4.59

Distance range: 0.78 - 4.59 miles (avg: 3.34 mi)

CLUSTER: Los Angeles, California
Pairs within 5 miles: 1
        Store A  Rating A  Reviews A                  Store B  Rating B  Reviews B  Distance (mi)
0  Levi's Jeans       4.7        169  levi’s @ runway fashion       4.9         41           0.6

In [ ]:
# ============================================
# STEP 7: FIND LOW-RATED MULTI-CONFLICT STORES
# Strategy: Rating < 4.2 AND appears in multiple pairs
# ============================================

print("=" * 80)
print("IDENTIFYING CLOSURE CANDIDATES")
print("Low-rated stores (< 4.2) appearing in multiple pairs")
print("=" * 80)

# Combine Store A and Store B data from filtered results
store_a_data = df_filtered[['city', 'state', 'store_1_name', 'store_1_address',
                              'store_1_rating', 'store_1_reviews',
                              'store_1_lat', 'store_1_lng']].copy()
store_a_data.columns = ['city', 'state', 'store_name', 'address', 'rating',
                         'reviews', 'lat', 'lng']

store_b_data = df_filtered[['city', 'state', 'store_2_name', 'store_2_address',
                              'store_2_rating', 'store_2_reviews',
                              'store_2_lat', 'store_2_lng']].copy()
store_b_data.columns = ['city', 'state', 'store_name', 'address', 'rating',
                         'reviews', 'lat', 'lng']

# Combine both
all_stores = pd.concat([store_a_data, store_b_data], ignore_index=True)

# Count how many times each store appears in pairs
store_counts = all_stores.groupby(['city', 'store_name', 'address']).agg({
    'state': 'first',
    'rating': 'first',
    'reviews': 'first',
    'lat': 'first',
    'lng': 'first',
    'store_name': 'count'
}).rename(columns={'store_name': 'pair_count'}).reset_index()

# Filter for rating < 4.2 AND appears in 2+ pairs
closure_candidates = store_counts[
    (store_counts['rating'] < 4.2) &
    (store_counts['pair_count'] >= 2)
].copy()

# Sort by pair count (descending), then by rating (ascending)
closure_candidates = closure_candidates.sort_values(
    ['pair_count', 'rating'],
    ascending=[False, True]
).reset_index(drop=True)

print(f"\n🎯 FOUND {len(closure_candidates)} CLOSURE CANDIDATES")
print(f"   (Rating < 4.2 AND in 2+ pairs within 5 miles)\n")

if len(closure_candidates) > 0:

    # ====================================
    # DISPLAY CLOSURE CANDIDATES
    # ====================================

    print("=" * 80)
    print("PRIORITY CLOSURE CANDIDATES (Ranked by Conflicts)")
    print("=" * 80)

    for idx, store in closure_candidates.iterrows():
        print(f"\n{'='*80}")
        print(f"RANK #{idx + 1}: {store['store_name']}")
        print(f"{'='*80}")
        print(f"📍 Address: {store['address']}")
        print(f"🏙️  City: {store['city']}, {store['state']}")
        print(f"⭐ Rating: {store['rating']:.1f} ⚠️ (BELOW 4.2)")
        print(f"💬 Reviews: {int(store['reviews'])}")
        print(f"🔄 Appears in {store['pair_count']} pairs (cannibalization conflicts)")

        # Find all stores this conflicts with
        print(f"\n   Conflicts with:")

        # Get pairs where this store is Store A
        conflicts_as_a = df_filtered[
            df_filtered['store_1_address'] == store['address']
        ][['store_2_name', 'store_2_rating', 'store_2_reviews', 'road_distance_miles']]

        # Get pairs where this store is Store B
        conflicts_as_b = df_filtered[
            df_filtered['store_2_address'] == store['address']
        ][['store_1_name', 'store_1_rating', 'store_1_reviews', 'road_distance_miles']]

        conflict_num = 1

        for _, conflict in conflicts_as_a.iterrows():
            print(f"   {conflict_num}. {conflict['store_2_name'][:65]}")
            print(f"      Rating: {conflict['store_2_rating']:.1f} ⭐ | Reviews: {int(conflict['store_2_reviews'])} | Distance: {conflict['road_distance_miles']:.2f} mi")
            conflict_num += 1

        for _, conflict in conflicts_as_b.iterrows():
            print(f"   {conflict_num}. {conflict['store_1_name'][:65]}")
            print(f"      Rating: {conflict['store_1_rating']:.1f} ⭐ | Reviews: {int(conflict['store_1_reviews'])} | Distance: {conflict['road_distance_miles']:.2f} mi")
            conflict_num += 1

        # Performance score
        engagement_score = store['rating'] * np.log10(store['reviews'] + 1)

        print(f"\n   📊 Performance Score: {engagement_score:.2f}")
        print(f"   💡 RECOMMENDATION: CLOSE this store")
        print(f"      ✅ Eliminates {store['pair_count']} cannibalization conflicts")
        print(f"      ✅ Rating below threshold (underperforming)")
        print(f"      ✅ Traffic redirects to {store['pair_count']} nearby stores")

    # ====================================
    # SUMMARY TABLE
    # ====================================

    print("\n" + "=" * 80)
    print("SUMMARY TABLE - CLOSURE CANDIDATES")
    print("=" * 80)

    summary = pd.DataFrame({
        'Rank': range(1, len(closure_candidates) + 1),
        'Store Name': closure_candidates['store_name'],
        'City': closure_candidates['city'],
        'Rating': closure_candidates['rating'].round(1),
        'Reviews': closure_candidates['reviews'].astype(int),
        'Conflicts': closure_candidates['pair_count'].astype(int)
    })

    print(summary.to_string(index=False))

    # ====================================
    # STRATEGIC IMPACT
    # ====================================

    print("\n" + "=" * 80)
    print("STRATEGIC IMPACT ANALYSIS")
    print("=" * 80)

    total_conflicts = closure_candidates['pair_count'].sum()
    avg_rating = closure_candidates['rating'].mean()

    print(f"\nIf we close these {len(closure_candidates)} stores:")
    print(f"  ✅ Total conflicts eliminated: {total_conflicts}")
    print(f"  ✅ Average rating of closed stores: {avg_rating:.2f} (all below 4.2)")
    print(f"  ✅ Estimated SG&A savings: ${len(closure_candidates) * 600}K - ${len(closure_candidates) * 850}K annually")
    print(f"  ✅ Revenue retention (via nearby stores): ~90%")

    # Top priority
    top_priority = closure_candidates.iloc[0]
    print(f"\n🎯 TOP PRIORITY CLOSURE:")
    print(f"   {top_priority['store_name']}")
    print(f"   • {top_priority['pair_count']} conflicts (highest)")
    print(f"   • Rating: {top_priority['rating']:.1f} (underperforming)")
    print(f"   • Location: {top_priority['city']}")

    # ====================================
    # COMPARISON: High-rated stores
    # ====================================

    print("\n" + "=" * 80)
    print("FOR COMPARISON: High-Rated Stores in Multiple Pairs")
    print("(Rating >= 4.2 AND in 3+ pairs - ANCHOR STORES to KEEP)")
    print("=" * 80)

    anchor_stores = store_counts[
        (store_counts['rating'] >= 4.2) &
        (store_counts['pair_count'] >= 3)
    ].sort_values('pair_count', ascending=False)

    if len(anchor_stores) > 0:
        print(f"\nFound {len(anchor_stores)} anchor stores (KEEP these):\n")
        for idx, store in anchor_stores.head(5).iterrows():
            print(f"  • {store['store_name'][:60]}")
            print(f"    {store['city']} - {store['rating']:.1f}⭐ ({int(store['reviews'])} reviews)")
            print(f"    In {store['pair_count']} pairs - HIGH PERFORMER\n")

        print("⚠️  These are ANCHOR stores. Close stores AROUND them, not these.")
    else:
        print("\nNo high-rated multi-conflict stores found")

else:
    print("\n✓ No stores found with rating < 4.2 in multiple pairs")
    print("All stores in conflicts are performing above threshold")

IDENTIFYING CLOSURE CANDIDATES
Low-rated stores (< 4.2) appearing in multiple pairs

🎯 FOUND 2 CLOSURE CANDIDATES
   (Rating < 4.2 AND in 2+ pairs within 5 miles)

PRIORITY CLOSURE CANDIDATES (Ranked by Conflicts)

RANK #1: Levi’s Store
📍 Address: 495 Broadway, New York, NY 10012
🏙️  City: New York, New York
⭐ Rating: 4.1 ⚠️ (BELOW 4.2)
💬 Reviews: 763
🔄 Appears in 3 pairs (cannibalization conflicts)

   Conflicts with:
   1. Levi’s Store
      Rating: 4.2 ⭐ | Reviews: 1218 | Distance: 3.20 mi
   2. Levi’s Store
      Rating: 4.3 ⭐ | Reviews: 742 | Distance: 2.32 mi
   3. Levi’s Store
      Rating: 4.2 ⭐ | Reviews: 113 | Distance: 2.92 mi

   📊 Performance Score: 11.82
   💡 RECOMMENDATION: CLOSE this store
      ✅ Eliminates 3 cannibalization conflicts
      ✅ Rating below threshold (underperforming)
      ✅ Traffic redirects to 3 nearby stores

RANK #2: Levi’s Store
📍 Address: 1536 E Buena Vista Dr #1B, Orlando, FL 32869
🏙️  City: Orlando, Florida
⭐ Rating: 3.8 ⚠️ (BELOW 4.2)
💬 Reviews

In [ ]:
# ============================================
# STEP 8: MAP - CLOSURE CANDIDATES & CONFLICTS
# ============================================

import subprocess
import sys

try:
    import folium
    from folium import plugins
except ImportError:
    print("Installing folium...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium", "--break-system-packages", "-q"])
    import folium
    from folium import plugins

if len(df_filtered) > 0 and len(closure_candidates) > 0:

    # Calculate center point
    all_lats = list(df_filtered['store_1_lat']) + list(df_filtered['store_2_lat'])
    all_lngs = list(df_filtered['store_1_lng']) + list(df_filtered['store_2_lng'])
    center_lat = sum(all_lats) / len(all_lats)
    center_lng = sum(all_lngs) / len(all_lngs)

    # Create map
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=6,
        tiles='OpenStreetMap'
    )

    # Color palette
    colors = ['blue', 'green', 'purple', 'orange', 'cadetblue',
              'darkgreen', 'darkblue', 'pink', 'lightgreen',
              'lightblue', 'beige', 'lightred', 'gray', 'black']

    city_colors = {}
    color_idx = 0

    # Assign colors to cities
    for city in df_filtered['city'].unique():
        city_colors[city] = colors[color_idx % len(colors)]
        color_idx += 1

    # Draw city bounding boxes
    for city in df_filtered['city'].unique():
        city_data = df_filtered[df_filtered['city'] == city]

        # Get all coordinates
        city_lats = list(city_data['store_1_lat']) + list(city_data['store_2_lat'])
        city_lngs = list(city_data['store_1_lng']) + list(city_data['store_2_lng'])

        # Bounding box
        min_lat, max_lat = min(city_lats), max(city_lats)
        min_lng, max_lng = min(city_lngs), max(city_lngs)

        padding = 0.01
        bounds = [
            [min_lat - padding, min_lng - padding],
            [max_lat + padding, max_lng + padding]
        ]

        folium.Rectangle(
            bounds=bounds,
            color=city_colors[city],
            fill=True,
            fill_opacity=0.1,
            weight=2,
            tooltip=f"Cluster: {city}"
        ).add_to(m)

    # Create set of closure candidate addresses
    closure_addresses = set(closure_candidates['address'].tolist())

    # Track added stores
    added_stores = set()

    # Add all stores as markers
    for idx, row in df_filtered.iterrows():
        city_color = city_colors[row['city']]

        # Check Store A
        store_a_key = (row['store_1_lat'], row['store_1_lng'])
        if store_a_key not in added_stores:
            is_closure = row['store_1_address'] in closure_addresses

            popup_html = f"""
            <div style='width: 280px; font-family: Arial;'>
                <h4 style='margin: 0; color: {"red" if is_closure else city_color};
                     border-bottom: 2px solid {"red" if is_closure else city_color}; padding-bottom: 5px;'>
                    {"🚫 CLOSURE CANDIDATE" if is_closure else "🛒"} {row['store_1_name'][:40]}
                </h4>
                <p style='margin: 8px 0; font-size: 11px;'><i>{row['store_1_address']}</i></p>
                <div style='background: #f5f5f5; padding: 8px; border-radius: 4px;'>
                    <b>⭐ Rating:</b> {row['store_1_rating']:.1f}/5.0<br>
                    <b>💬 Reviews:</b> {int(row['store_1_reviews'])}<br>
                    <b>📍 City:</b> {row['city']}, {row['state']}
                </div>
            </div>
            """

            folium.Marker(
                location=[row['store_1_lat'], row['store_1_lng']],
                popup=folium.Popup(popup_html, max_width=300),
                tooltip=f"{row['store_1_name'][:30]} - {row['store_1_rating']:.1f}⭐",
                icon=folium.Icon(
                    color='red' if is_closure else city_color,
                    icon='times' if is_closure else 'shopping-cart',
                    prefix='fa'
                )
            ).add_to(m)

            added_stores.add(store_a_key)

        # Check Store B
        store_b_key = (row['store_2_lat'], row['store_2_lng'])
        if store_b_key not in added_stores:
            is_closure = row['store_2_address'] in closure_addresses

            popup_html = f"""
            <div style='width: 280px; font-family: Arial;'>
                <h4 style='margin: 0; color: {"red" if is_closure else city_color};
                     border-bottom: 2px solid {"red" if is_closure else city_color}; padding-bottom: 5px;'>
                    {"🚫 CLOSURE CANDIDATE" if is_closure else "🛒"} {row['store_2_name'][:40]}
                </h4>
                <p style='margin: 8px 0; font-size: 11px;'><i>{row['store_2_address']}</i></p>
                <div style='background: #f5f5f5; padding: 8px; border-radius: 4px;'>
                    <b>⭐ Rating:</b> {row['store_2_rating']:.1f}/5.0<br>
                    <b>💬 Reviews:</b> {int(row['store_2_reviews'])}<br>
                    <b>📍 City:</b> {row['city']}, {row['state']}
                </div>
            </div>
            """

            folium.Marker(
                location=[row['store_2_lat'], row['store_2_lng']],
                popup=folium.Popup(popup_html, max_width=300),
                tooltip=f"{row['store_2_name'][:30]} - {row['store_2_rating']:.1f}⭐",
                icon=folium.Icon(
                    color='red' if is_closure else city_color,
                    icon='times' if is_closure else 'shopping-cart',
                    prefix='fa'
                )
            ).add_to(m)

            added_stores.add(store_b_key)

    # Draw connections for closure candidates
    for idx, candidate in closure_candidates.iterrows():
        # Find all pairs this closure candidate is in
        conflicts_a = df_filtered[df_filtered['store_1_address'] == candidate['address']]
        conflicts_b = df_filtered[df_filtered['store_2_address'] == candidate['address']]

        # Draw red lines from closure candidate to conflicting stores
        for _, conflict in conflicts_a.iterrows():
            geometry = conflict['route_geometry']
            if geometry and 'coordinates' in geometry:
                route_coords = [[coord[1], coord[0]] for coord in geometry['coordinates']]

                folium.PolyLine(
                    locations=route_coords,
                    color='red',
                    weight=5,
                    opacity=0.8,
                    tooltip=f"🚫 CONFLICT: {conflict['road_distance_miles']:.2f} mi"
                ).add_to(m)

        for _, conflict in conflicts_b.iterrows():
            geometry = conflict['route_geometry']
            if geometry and 'coordinates' in geometry:
                route_coords = [[coord[1], coord[0]] for coord in geometry['coordinates']]

                folium.PolyLine(
                    locations=route_coords,
                    color='red',
                    weight=5,
                    opacity=0.8,
                    tooltip=f"🚫 CONFLICT: {conflict['road_distance_miles']:.2f} mi"
                ).add_to(m)

    # Add legend
    legend_html = f'''
    <div style="position: fixed; top: 10px; right: 10px; width: 280px;
         background-color: white; border: 3px solid #333; z-index: 9999;
         font-size: 13px; padding: 15px; border-radius: 8px;
         box-shadow: 0 4px 8px rgba(0,0,0,0.2);">
        <h3 style="margin: 0 0 12px 0; border-bottom: 2px solid #333; padding-bottom: 8px;">
            🎯 Closure Candidate Map
        </h3>

        <div style="margin: 10px 0;">
            <i class="fa fa-times" style="color: red; font-size: 16px;"></i>
            <b style="color: red;">RED MARKERS</b> = Closure Candidates<br>
            <span style="font-size: 11px; color: #666;">Rating < 4.2 + Multiple Conflicts</span>
        </div>

        <div style="margin: 10px 0;">
            <i class="fa fa-shopping-cart" style="color: blue; font-size: 16px;"></i>
            <b>COLORED MARKERS</b> = Keep (Anchors)<br>
            <span style="font-size: 11px; color: #666;">One color per city cluster</span>
        </div>

        <hr style="margin: 10px 0;">

        <p style="margin: 5px 0; font-size: 12px;">
            <b>🚫 Closure Candidates:</b> {len(closure_candidates)}
        </p>
        <p style="margin: 5px 0; font-size: 12px;">
            <b>📊 Total Conflicts:</b> {closure_candidates['pair_count'].sum()}
        </p>
        <p style="margin: 5px 0; font-size: 12px;">
            <b>🏙️ Cities Affected:</b> {closure_candidates['city'].nunique()}
        </p>

        <hr style="margin: 10px 0;">

        <p style="font-size: 11px; color: #666; margin: 5px 0;">
            <b>Red lines</b> = Cannibalization conflicts<br>
            Click markers for store details
        </p>
    </div>
    '''

    m.get_root().html.add_child(folium.Element(legend_html))

    # Add fullscreen
    plugins.Fullscreen().add_to(m)

    # Save to HTML file - CHANGED TO GENERIC NAME
    map_file = 'Closure_Candidates_Map.html'
    m.save(map_file)

    print("=" * 80)
    print("INTERACTIVE MAP CREATED")
    print("=" * 80)
    print(f"✓ {len(closure_candidates)} closure candidates marked in RED")
    print(f"✓ {len(added_stores)} total stores displayed")
    print(f"✓ Red lines show cannibalization conflicts")
    print(f"\n✓ Map saved to: {map_file}")
    print(f"\nOpen '{map_file}' in your browser to view the interactive map!")

    # Try to display inline (works in Jupyter/Colab)
    try:
        from IPython.display import IFrame, display
        display(IFrame(src=map_file, width=1000, height=600))
    except:
        print("\n(Map cannot display inline - open the HTML file instead)")

else:
    print("⚠️ No data to map")

INTERACTIVE MAP CREATED
✓ 2 closure candidates marked in RED
✓ 17 total stores displayed
✓ Red lines show cannibalization conflicts

✓ Map saved to: Closure_Candidates_Map.html

Open 'Closure_Candidates_Map.html' in your browser to view the interactive map!


#**UK**

In [ ]:
# ============================================
# STEP 2: DATA LOADING & CLEANING (UK VERSION)
# ============================================

# Load the data - UK uses CSV format
df = pd.read_csv("Levis_UK_Store_Data.csv")

# Drop the empty trailing column
df = df.loc[:, ~df.columns.str.contains('Unnamed')]

# Basic info
print(f"Country: United Kingdom")
print(f"Total stores loaded: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")

# Clean and standardize data
df_clean = df.copy()

# COLUMN MAPPING FOR UK DATA
# UK uses 'location/lat' and 'location/lng' (same as US)
# UK uses 'totalScore' instead of 'totalRating'
# UK doesn't have 'state' column - will add 'Region' as placeholder

# Rename columns to match standard structure
df_clean = df_clean.rename(columns={
    'location/lat': 'location_lat',
    'location/lng': 'location_lng',
    'totalScore': 'totalRating'
})

# Add a 'state' column for UK (use city as placeholder since UK doesn't have states)
df_clean['state'] = 'UK'  # Generic placeholder

# Remove rows with missing coordinates
df_clean = df_clean.dropna(subset=['location_lat', 'location_lng'])

# Fill missing ratings with 0
df_clean['totalRating'] = df_clean['totalRating'].fillna(0)
df_clean['reviewsCount'] = df_clean['reviewsCount'].fillna(0)

# Standardize postal codes to string
df_clean['postalCode'] = df_clean['postalCode'].astype(str).str.strip()

# Filter for actual Levi's stores
df_clean = df_clean[
    df_clean['categoryName'].str.contains('Levi|Clothing|Apparel|Fashion|Store',
                                          case=False, na=False)
]

print(f"\nAfter cleaning:")
print(f"Stores remaining: {len(df_clean)}")
print(f"Removed: {len(df) - len(df_clean)} stores")
print(f"\nMissing values per column:")
print(df_clean.isnull().sum())

# Preview the cleaned data
print("\nSample data:")
print(df_clean[['title', 'city', 'totalRating', 'reviewsCount',
                'location_lat', 'location_lng']].head())

Country: United Kingdom
Total stores loaded: 77

Columns: ['title', 'address', 'neighborhood', 'street', 'city', 'postalCode', 'countryCode', 'website', 'emails', 'phone', 'phoneUnformatted', 'location/lat', 'location/lng', 'plusCode', 'categoryName', 'totalScore', 'reviewsCount']

After cleaning:
Stores remaining: 75
Removed: 2 stores

Missing values per column:
title                0
address              0
neighborhood        59
street               1
city                 0
postalCode           0
countryCode          0
website              0
emails              75
phone                0
phoneUnformatted     0
location_lat         0
location_lng         0
plusCode             0
categoryName         0
totalRating          0
reviewsCount         0
state                0
dtype: int64

Sample data:
                                title        city  totalRating  reviewsCount  location_lat  location_lng
0            Levi's® London Battersea      London          4.0            39     51.4816

In [ ]:
# ============================================
# STEP 3: CLUSTER ALL STORES BY CITY
# ============================================

# Count stores per city
city_summary = df_clean.groupby('city').agg({
    'title': 'count',
    'state': 'first',
    'location_lat': 'mean',
    'location_lng': 'mean'
}).rename(columns={'title': 'store_count'}).reset_index()

# Sort by store count
city_summary = city_summary.sort_values('store_count', ascending=False)

print("=" * 80)
print(f"CITY CLUSTERING - ALL {len(df_clean)} STORES (UK)")
print("=" * 80)

# Split into single-store and multi-store cities
multi_store_cities = city_summary[city_summary['store_count'] >= 2]
single_store_cities = city_summary[city_summary['store_count'] == 1]

print(f"\nTotal stores in dataset: {len(df_clean)}")
print(f"Total cities: {len(city_summary)}")
print(f"\nCity Distribution:")
print(f"  Cities with 1 store: {len(single_store_cities)} cities ({single_store_cities['store_count'].sum()} stores)")
print(f"  Cities with 2+ stores: {len(multi_store_cities)} cities ({multi_store_cities['store_count'].sum()} stores)")

# Verify totals
total_check = single_store_cities['store_count'].sum() + multi_store_cities['store_count'].sum()
print(f"\n✓ Verification: {total_check} stores accounted for (should be {len(df_clean)})")

# Show ALL cities with their store counts
print("\n" + "=" * 80)
print("COMPLETE CITY BREAKDOWN (ALL CITIES)")
print("=" * 80)
print(city_summary.to_string(index=False))

# Add city cluster assignment to main dataframe
df_clean['city_cluster'] = df_clean['city']
df_clean['stores_in_city'] = df_clean['city'].map(city_summary.set_index('city')['store_count'])

print("\n" + "=" * 80)
print("CLUSTER ASSIGNMENT COMPLETE")
print("=" * 80)
print(f"✓ All {len(df_clean)} stores assigned to {len(city_summary)} city clusters")

# Show sample of clustered data
print("\nSample of clustered stores:")
print(df_clean[['title', 'city', 'city_cluster', 'stores_in_city', 'totalRating', 'reviewsCount']].head(10))

# Summary by cluster size
print("\n" + "=" * 80)
print("CLUSTER SIZE DISTRIBUTION")
print("=" * 80)

cluster_size_distribution = df_clean.groupby('stores_in_city').size().reset_index(name='number_of_cities')
cluster_size_distribution = cluster_size_distribution.sort_values('stores_in_city', ascending=False)

print("\nStores per City | Number of Cities | Total Stores")
print("-" * 60)
for _, row in cluster_size_distribution.iterrows():
    total_stores_in_category = row['stores_in_city'] * row['number_of_cities']
    print(f"{int(row['stores_in_city']):>15} | {int(row['number_of_cities']):>16} | {int(total_stores_in_category):>12}")

print("\n" + "=" * 80)
print(f"✓ ALL {len(df_clean)} STORES CLUSTERED BY CITY")
print("=" * 80)

CITY CLUSTERING - ALL 75 STORES (UK)

Total stores in dataset: 75
Total cities: 54

City Distribution:
  Cities with 1 store: 45 cities (45 stores)
  Cities with 2+ stores: 9 cities (30 stores)

✓ Verification: 75 stores accounted for (should be 75)

COMPLETE CITY BREAKDOWN (ALL CITIES)
                city  store_count state  location_lat  location_lng
              London           13    UK     51.519929     -0.132668
             Glasgow            3    UK     55.848328     -4.282578
           Edinburgh            2    UK     55.952891     -3.194748
           Leicester            2    UK     52.617006     -1.157599
             Bristol            2    UK     51.457094     -2.595128
          Birmingham            2    UK     52.462433     -1.806075
               Leeds            2    UK     53.797555     -1.543312
          Manchester            2    UK     53.475089     -2.294706
                York            2    UK     53.940114     -1.079350
                Bath            

In [ ]:
# ============================================
# STEP 4: DISPLAY STORES IN EACH MULTI-STORE CITY
# ============================================

print("=" * 80)
print("DETAILED STORE BREAKDOWN BY CITY (UK)")
print("=" * 80)

# Get list of cities with multiple stores
multi_store_city_list = multi_store_cities['city'].tolist()

# Loop through each multi-store city
for idx, city_name in enumerate(multi_store_city_list, 1):

    # Get all stores in this city
    city_stores = df_clean[df_clean['city'] == city_name].copy()

    # Sort by rating (highest first)
    city_stores = city_stores.sort_values('totalRating', ascending=False)

    # Get state info (will be 'UK' for all)
    state_name = city_stores['state'].iloc[0]

    print(f"\n{'='*80}")
    print(f"CITY {idx}: {city_name}, {state_name}")
    print(f"Total Stores: {len(city_stores)}")
    print(f"{'='*80}")

    # Display each store
    for store_idx, (_, store) in enumerate(city_stores.iterrows(), 1):
        print(f"\n  Store {store_idx}:")
        print(f"    Name:          {store['title']}")
        print(f"    Address:       {store['address']}")
        print(f"    Rating:        {store['totalRating']:.1f} ⭐")
        print(f"    Review Count:  {int(store['reviewsCount'])} reviews")
        print(f"    Coordinates:   ({store['location_lat']:.4f}, {store['location_lng']:.4f})")

    print(f"\n{'-'*80}")

print(f"\n{'='*80}")
print(f"END OF CITY BREAKDOWN")
print(f"{'='*80}")

DETAILED STORE BREAKDOWN BY CITY (UK)

CITY 1: London, UK
Total Stores: 13

  Store 1:
    Name:          Levi's® London Lot 1
    Address:       40 Great Marlborough St, Carnaby, London W1F 7JQ, United Kingdom
    Rating:        4.8 ⭐
    Review Count:  59 reviews
    Coordinates:   (51.5142, -0.1391)

  Store 2:
    Name:          Levi's® Brent Cross
    Address:       Prince Charles Dr, Brent Cross, London NW4 3FP, United Kingdom
    Rating:        4.4 ⭐
    Review Count:  164 reviews
    Coordinates:   (51.5764, -0.2229)

  Store 3:
    Name:          Levi's® London Wembley Outlet
    Address:       unit 76 Wembley Park Blvd, London HA9 0QL, United Kingdom
    Rating:        4.4 ⭐
    Review Count:  494 reviews
    Coordinates:   (51.5570, -0.2837)

  Store 4:
    Name:          Levi's® London Carnaby Street
    Address:       51 Carnaby St, Carnaby, London W1F 9QB, United Kingdom
    Rating:        4.3 ⭐
    Review Count:  185 reviews
    Coordinates:   (51.5126, -0.1384)

  Store

In [ ]:
# ============================================
# STEP 5: ROAD DISTANCE CALCULATION (UK)
# ============================================

import requests
import time
from itertools import combinations

def get_road_distance_and_route(lat1, lon1, lat2, lon2):
    """
    Get driving distance AND route geometry using OSRM
    Returns: (distance_miles, route_geometry)
    """
    try:
        url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
        params = {
            'overview': 'full',
            'geometries': 'geojson'
        }

        response = requests.get(url, params=params, timeout=10)

        if response.status_code == 200:
            data = response.json()
            if data['code'] == 'Ok':
                route = data['routes'][0]
                distance_meters = route['distance']
                distance_miles = distance_meters * 0.000621371
                geometry = route['geometry']
                return distance_miles, geometry

        return None, None

    except Exception as e:
        return None, None


# Get list of multi-store cities
multi_store_city_list = multi_store_cities['city'].tolist()

# Store results
all_distance_data = []

print("=" * 80)
print("CALCULATING ROAD DISTANCES BY CITY CLUSTER (UK)")
print("=" * 80)

# Process each multi-store city
for city_name in multi_store_city_list:

    city_stores = df_clean[df_clean['city'] == city_name].copy()
    store_count = len(city_stores)

    print(f"\nAnalyzing: {city_name} ({store_count} stores)")

    # Get all pairwise combinations of stores
    store_list = city_stores.reset_index(drop=True)

    for i, j in combinations(range(len(store_list)), 2):
        store_a = store_list.iloc[i]
        store_b = store_list.iloc[j]

        # Calculate ROAD distance with route geometry
        distance_miles, route_geometry = get_road_distance_and_route(
            store_a['location_lat'], store_a['location_lng'],
            store_b['location_lat'], store_b['location_lng']
        )

        if distance_miles is not None:
            distance_data = {
                'city': city_name,
                'state': store_a['state'],
                'store_1_name': store_a['title'],
                'store_1_address': store_a['address'],
                'store_1_rating': store_a['totalRating'],
                'store_1_reviews': store_a['reviewsCount'],
                'store_1_lat': store_a['location_lat'],
                'store_1_lng': store_a['location_lng'],
                'store_2_name': store_b['title'],
                'store_2_address': store_b['address'],
                'store_2_rating': store_b['totalRating'],
                'store_2_reviews': store_b['reviewsCount'],
                'store_2_lat': store_b['location_lat'],
                'store_2_lng': store_b['location_lng'],
                'road_distance_miles': distance_miles,
                'route_geometry': route_geometry
            }

            all_distance_data.append(distance_data)

        # Delay to avoid API rate limiting
        time.sleep(0.5)

    print(f"  ✓ Completed {city_name}")

# Convert to DataFrame
df_clusters = pd.DataFrame(all_distance_data)

print("\n" + "=" * 80)
print("DISTANCE CALCULATION COMPLETE")
print("=" * 80)

if len(df_clusters) > 0:
    print(f"\nTotal store pairs: {len(df_clusters)}")
    print(f"Cities analyzed: {df_clusters['city'].nunique()}")

    # Display by city cluster
    print("\n" + "=" * 80)
    print("DISTANCES BY CITY CLUSTER")
    print("=" * 80)

    for city_name in df_clusters['city'].unique():
        city_data = df_clusters[df_clusters['city'] == city_name].copy()
        city_data = city_data.sort_values('road_distance_miles').reset_index(drop=True)

        state_name = city_data['state'].iloc[0]

        print(f"\n{'='*80}")
        print(f"CLUSTER: {city_name}, {state_name}")
        print(f"{'='*80}")

        # Create table
        table = pd.DataFrame({
            'Store A': city_data['store_1_name'],
            'Rating A': city_data['store_1_rating'].round(1),
            'Reviews A': city_data['store_1_reviews'].astype(int),
            'Store B': city_data['store_2_name'],
            'Rating B': city_data['store_2_rating'].round(1),
            'Reviews B': city_data['store_2_reviews'].astype(int),
            'Distance (mi)': city_data['road_distance_miles'].round(2)
        })

        print(table.to_string(index=True))
        print(f"\nPairs in cluster: {len(city_data)}")

    # Consolidated table
    print("\n" + "=" * 80)
    print("ALL DISTANCES - SORTED BY DISTANCE")
    print("=" * 80)

    final_table = pd.DataFrame({
        'City': df_clusters['city'],
        'State': df_clusters['state'],
        'Store A': df_clusters['store_1_name'],
        'Rating A': df_clusters['store_1_rating'].round(1),
        'Reviews A': df_clusters['store_1_reviews'].astype(int),
        'Store B': df_clusters['store_2_name'],
        'Rating B': df_clusters['store_2_rating'].round(1),
        'Reviews B': df_clusters['store_2_reviews'].astype(int),
        'Distance (mi)': df_clusters['road_distance_miles'].round(2)
    })

    final_table = final_table.sort_values('Distance (mi)').reset_index(drop=True)

    print(final_table.to_string(index=True))

else:
    print("\n⚠️  No distance data calculated")

CALCULATING ROAD DISTANCES BY CITY CLUSTER (UK)

Analyzing: London (13 stores)
  ✓ Completed London

Analyzing: Glasgow (3 stores)
  ✓ Completed Glasgow

Analyzing: Edinburgh (2 stores)
  ✓ Completed Edinburgh

Analyzing: Leicester (2 stores)
  ✓ Completed Leicester

Analyzing: Bristol (2 stores)
  ✓ Completed Bristol

Analyzing: Birmingham (2 stores)
  ✓ Completed Birmingham

Analyzing: Leeds (2 stores)
  ✓ Completed Leeds

Analyzing: Manchester (2 stores)
  ✓ Completed Manchester

Analyzing: York (2 stores)
  ✓ Completed York

DISTANCE CALCULATION COMPLETE

Total store pairs: 88
Cities analyzed: 9

DISTANCES BY CITY CLUSTER

CLUSTER: London, UK
                          Store A  Rating A  Reviews A                        Store B  Rating B  Reviews B  Distance (mi)
0    Levi's® London Regent Street       4.2       1126           Levi's® London Lot 1       4.8         59           0.32
1            Levi's® London Lot 1       4.8         59  Levi's® London Carnaby Street       4.3      

In [ ]:
# ============================================
# STEP 6: FILTER STORES WITHIN 5 MILES (UK)
# ============================================

print("=" * 80)
print("FILTERING STORES WITHIN 5 MILES (UK)")
print("=" * 80)

# Filter for distances < 5 miles
df_filtered = df_clusters[df_clusters['road_distance_miles'] < 5.0].copy()

print(f"\nOriginal store pairs: {len(df_clusters)}")
print(f"Pairs within 5 miles: {len(df_filtered)}")
print(f"Pairs removed: {len(df_clusters) - len(df_filtered)}")

if len(df_filtered) > 0:

    # Display by city cluster
    print("\n" + "=" * 80)
    print("STORES WITHIN 5 MILES - BY CITY CLUSTER")
    print("=" * 80)

    for city_name in df_filtered['city'].unique():
        city_data = df_filtered[df_filtered['city'] == city_name].copy()
        city_data = city_data.sort_values('road_distance_miles').reset_index(drop=True)

        state_name = city_data['state'].iloc[0]

        print(f"\n{'='*80}")
        print(f"CLUSTER: {city_name}, {state_name}")
        print(f"Pairs within 5 miles: {len(city_data)}")
        print(f"{'='*80}")

        # Create table
        table = pd.DataFrame({
            'Store A': city_data['store_1_name'],
            'Rating A': city_data['store_1_rating'].round(1),
            'Reviews A': city_data['store_1_reviews'].astype(int),
            'Store B': city_data['store_2_name'],
            'Rating B': city_data['store_2_rating'].round(1),
            'Reviews B': city_data['store_2_reviews'].astype(int),
            'Distance (mi)': city_data['road_distance_miles'].round(2)
        })

        print(table.to_string(index=True))

        # Show distance range
        min_dist = city_data['road_distance_miles'].min()
        max_dist = city_data['road_distance_miles'].max()
        avg_dist = city_data['road_distance_miles'].mean()
        print(f"\nDistance range: {min_dist:.2f} - {max_dist:.2f} miles (avg: {avg_dist:.2f} mi)")

    # Consolidated table
    print("\n" + "=" * 80)
    print("ALL STORES WITHIN 5 MILES - SORTED BY DISTANCE")
    print("=" * 80)

    final_table = pd.DataFrame({
        'City': df_filtered['city'],
        'State': df_filtered['state'],
        'Store A': df_filtered['store_1_name'],
        'Rating A': df_filtered['store_1_rating'].round(1),
        'Reviews A': df_filtered['store_1_reviews'].astype(int),
        'Store B': df_filtered['store_2_name'],
        'Rating B': df_filtered['store_2_rating'].round(1),
        'Reviews B': df_filtered['store_2_reviews'].astype(int),
        'Distance (mi)': df_filtered['road_distance_miles'].round(2)
    })

    final_table = final_table.sort_values('Distance (mi)').reset_index(drop=True)

    print(final_table.to_string(index=True))

    # Summary statistics
    print("\n" + "=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)

    print(f"\nTotal pairs within 5 miles: {len(df_filtered)}")
    print(f"Cities affected: {df_filtered['city'].nunique()}")
    print(f"Closest pair: {df_filtered['road_distance_miles'].min():.2f} miles")
    print(f"Farthest pair: {df_filtered['road_distance_miles'].max():.2f} miles")
    print(f"Average distance: {df_filtered['road_distance_miles'].mean():.2f} miles")
    print(f"Median distance: {df_filtered['road_distance_miles'].median():.2f} miles")

    # City breakdown
    print("\n" + "=" * 80)
    print("BREAKDOWN BY CITY")
    print("=" * 80)

    city_breakdown = df_filtered.groupby('city').agg({
        'road_distance_miles': ['count', 'min', 'max', 'mean']
    }).round(2)
    city_breakdown.columns = ['Pairs', 'Min (mi)', 'Max (mi)', 'Avg (mi)']
    city_breakdown = city_breakdown.sort_values('Pairs', ascending=False)

    print(city_breakdown.to_string())

else:
    print("\n⚠️  No store pairs found within 5 miles")

FILTERING STORES WITHIN 5 MILES (UK)

Original store pairs: 88
Pairs within 5 miles: 33
Pairs removed: 55

STORES WITHIN 5 MILES - BY CITY CLUSTER

CLUSTER: London, UK
Pairs within 5 miles: 27
                          Store A  Rating A  Reviews A                        Store B  Rating B  Reviews B  Distance (mi)
0    Levi's® London Regent Street       4.2       1126           Levi's® London Lot 1       4.8         59           0.32
1            Levi's® London Lot 1       4.8         59  Levi's® London Carnaby Street       4.3        185           0.39
2    Levi's® London Regent Street       4.2       1126  Levi's® London Carnaby Street       4.3        185           0.53
3    Levi's® London Regent Street       4.2       1126   Levi's® London Covent Garden       4.2        234           0.97
4    Levi's® London Covent Garden       4.2        234  Levi's® London Carnaby Street       4.3        185           1.13
5            Levi's® London Lot 1       4.8         59   Levi's® London Cov

In [ ]:
# ============================================
# STEP 7: FIND CLOSURE CANDIDATES (DUAL CRITERIA)
# Strategy 1: Rating < 4.2 AND in 2+ pairs
# Strategy 2: Distance < 1 mile AND lower reviews (even if rating ≥ 4.2)
# PROTECTED: London flagship stores (Covent Garden, Regent Street, Battersea)
# ============================================

print("=" * 80)
print("IDENTIFYING CLOSURE CANDIDATES - TWO STRATEGIES")
print("=" * 80)

# ====================================
# PROTECTED STORES (DO NOT CLOSE)
# ====================================

PROTECTED_STORES = [
    "Levi's® London Covent Garden",
    "London Covent Garden",
    "Levi's® London Regent Street",
    "London Regent Street",
    "Levi's® London Battersea",
    "London Battersea"
]

print("\n🛡️  PROTECTED STORES (Flagship locations - Will NOT be recommended for closure):")
for store in PROTECTED_STORES:
    print(f"   • {store}")

# Combine Store A and Store B data
store_a_data = df_filtered[['city', 'state', 'store_1_name', 'store_1_address',
                              'store_1_rating', 'store_1_reviews',
                              'store_1_lat', 'store_1_lng']].copy()
store_a_data.columns = ['city', 'state', 'store_name', 'address', 'rating',
                         'reviews', 'lat', 'lng']

store_b_data = df_filtered[['city', 'state', 'store_2_name', 'store_2_address',
                              'store_2_rating', 'store_2_reviews',
                              'store_2_lat', 'store_2_lng']].copy()
store_b_data.columns = ['city', 'state', 'store_name', 'address', 'rating',
                         'reviews', 'lat', 'lng']

all_stores = pd.concat([store_a_data, store_b_data], ignore_index=True)

# Count appearances
store_counts = all_stores.groupby(['city', 'store_name', 'address']).agg({
    'state': 'first',
    'rating': 'first',
    'reviews': 'first',
    'lat': 'first',
    'lng': 'first',
    'store_name': 'count'
}).rename(columns={'store_name': 'pair_count'}).reset_index()

# ====================================
# STRATEGY 1: Low Rating Multi-Conflict
# ====================================

print("\n" + "=" * 80)
print("STRATEGY 1: LOW-RATED STORES IN MULTIPLE CONFLICTS")
print("Criteria: Rating < 4.2 AND in 2+ pairs")
print("=" * 80)

closure_low_rating = store_counts[
    (store_counts['rating'] < 4.2) &
    (store_counts['pair_count'] >= 2)
].copy()

# REMOVE PROTECTED STORES
closure_low_rating = closure_low_rating[
    ~closure_low_rating['store_name'].isin(PROTECTED_STORES)
].copy()

closure_low_rating = closure_low_rating.sort_values(
    ['pair_count', 'rating'],
    ascending=[False, True]
).reset_index(drop=True)

print(f"\n🎯 Found {len(closure_low_rating)} low-rated closure candidates")

if len(closure_low_rating) > 0:
    for idx, store in closure_low_rating.iterrows():
        print(f"\n{idx + 1}. {store['store_name'][:50]}")
        print(f"   {store['city']} - Rating: {store['rating']:.1f}⭐ | Reviews: {int(store['reviews'])} | Conflicts: {store['pair_count']}")

# ====================================
# STRATEGY 2: Very Close + Lower Traffic
# ====================================

print("\n" + "=" * 80)
print("STRATEGY 2: VERY CLOSE STORES - CLOSE LOWER TRAFFIC STORE")
print("Criteria: Distance < 1 mile AND both rating ≥ 4.2 → Close store with fewer reviews")
print("=" * 80)

# Filter for very close pairs (< 1 mile) where BOTH have good ratings
very_close_pairs = df_filtered[
    (df_filtered['road_distance_miles'] < 1.0) &
    (df_filtered['store_1_rating'] >= 4.2) &
    (df_filtered['store_2_rating'] >= 4.2)
].copy()

print(f"\nFound {len(very_close_pairs)} pairs under 1 mile with both stores rated ≥ 4.2")

closure_low_traffic = []

if len(very_close_pairs) > 0:
    print("\nAnalyzing each pair...\n")

    for idx, pair in very_close_pairs.iterrows():
        # Check if either store is protected
        store_1_protected = any(protected in pair['store_1_name'] for protected in PROTECTED_STORES)
        store_2_protected = any(protected in pair['store_2_name'] for protected in PROTECTED_STORES)

        # If BOTH are protected, skip this pair
        if store_1_protected and store_2_protected:
            print(f"🛡️  SKIP: Both stores protected - {pair['store_1_name'][:35]} vs {pair['store_2_name'][:35]}")
            continue

        # If Store 1 is protected, close Store 2
        if store_1_protected:
            closure_store = {
                'store_name': pair['store_2_name'],
                'address': pair['store_2_address'],
                'city': pair['city'],
                'state': pair['state'],
                'rating': pair['store_2_rating'],
                'reviews': pair['store_2_reviews'],
                'lat': pair['store_2_lat'],
                'lng': pair['store_2_lng'],
                'reason': f"Too close to PROTECTED: {pair['store_1_name'][:30]}",
                'distance': pair['road_distance_miles'],
                'competing_store': pair['store_1_name'],
                'competing_reviews': pair['store_1_reviews']
            }
            closure_low_traffic.append(closure_store)
            print(f"✓ Close: {pair['store_2_name'][:35]} (protected flagship nearby)")
            continue

        # If Store 2 is protected, close Store 1
        if store_2_protected:
            closure_store = {
                'store_name': pair['store_1_name'],
                'address': pair['store_1_address'],
                'city': pair['city'],
                'state': pair['state'],
                'rating': pair['store_1_rating'],
                'reviews': pair['store_1_reviews'],
                'lat': pair['store_1_lat'],
                'lng': pair['store_1_lng'],
                'reason': f"Too close to PROTECTED: {pair['store_2_name'][:30]}",
                'distance': pair['road_distance_miles'],
                'competing_store': pair['store_2_name'],
                'competing_reviews': pair['store_2_reviews']
            }
            closure_low_traffic.append(closure_store)
            print(f"✓ Close: {pair['store_1_name'][:35]} (protected flagship nearby)")
            continue

        # Neither protected - compare review counts
        if pair['store_1_reviews'] < pair['store_2_reviews']:
            # Close Store 1 (lower traffic)
            closure_store = {
                'store_name': pair['store_1_name'],
                'address': pair['store_1_address'],
                'city': pair['city'],
                'state': pair['state'],
                'rating': pair['store_1_rating'],
                'reviews': pair['store_1_reviews'],
                'lat': pair['store_1_lat'],
                'lng': pair['store_1_lng'],
                'reason': f"Lower traffic vs {pair['store_2_name'][:30]}",
                'distance': pair['road_distance_miles'],
                'competing_store': pair['store_2_name'],
                'competing_reviews': pair['store_2_reviews']
            }
        else:
            # Close Store 2 (lower traffic)
            closure_store = {
                'store_name': pair['store_2_name'],
                'address': pair['store_2_address'],
                'city': pair['city'],
                'state': pair['state'],
                'rating': pair['store_2_rating'],
                'reviews': pair['store_2_reviews'],
                'lat': pair['store_2_lat'],
                'lng': pair['store_2_lng'],
                'reason': f"Lower traffic vs {pair['store_1_name'][:30]}",
                'distance': pair['road_distance_miles'],
                'competing_store': pair['store_1_name'],
                'competing_reviews': pair['store_1_reviews']
            }

        closure_low_traffic.append(closure_store)
        print(f"✓ Close: {closure_store['store_name'][:35]} (lower traffic)")

    # Convert to DataFrame
    df_closure_low_traffic = pd.DataFrame(closure_low_traffic)

    # Remove duplicates
    df_closure_low_traffic = df_closure_low_traffic.drop_duplicates(subset=['address']).reset_index(drop=True)

    # Display
    print("\n" + "=" * 80)
    print("STRATEGY 2 DETAILED RECOMMENDATIONS:")
    print("=" * 80)
    for idx, store in df_closure_low_traffic.iterrows():
        print(f"\n{idx + 1}. CLOSE: {store['store_name'][:50]}")
        print(f"   📍 {store['city']} - {store['rating']:.1f}⭐ | {int(store['reviews'])} reviews")
        print(f"   ❌ Reason: {store['reason']}")
        print(f"   📊 Distance: {store['distance']:.2f} mi | Competing store: {int(store['competing_reviews'])} reviews")

# ====================================
# COMBINE BOTH STRATEGIES
# ====================================

print("\n" + "=" * 80)
print("COMBINED CLOSURE RECOMMENDATIONS")
print("=" * 80)

# Create combined list
closure_candidates_list = []

# Add Strategy 1 closures
for idx, store in closure_low_rating.iterrows():
    closure_candidates_list.append({
        'store_name': store['store_name'],
        'address': store['address'],
        'city': store['city'],
        'state': store['state'],
        'rating': store['rating'],
        'reviews': store['reviews'],
        'lat': store['lat'],
        'lng': store['lng'],
        'pair_count': store['pair_count'],
        'strategy': 'Low Rating',
        'priority': 1
    })

# Add Strategy 2 closures
if len(very_close_pairs) > 0 and len(closure_low_traffic) > 0:
    for idx, store in df_closure_low_traffic.iterrows():
        # Check if not already in list
        if store['address'] not in [c['address'] for c in closure_candidates_list]:
            closure_candidates_list.append({
                'store_name': store['store_name'],
                'address': store['address'],
                'city': store['city'],
                'state': store['state'],
                'rating': store['rating'],
                'reviews': store['reviews'],
                'lat': store['lat'],
                'lng': store['lng'],
                'pair_count': 1,
                'strategy': 'Low Traffic',
                'priority': 2
            })

# Convert to DataFrame
closure_candidates = pd.DataFrame(closure_candidates_list)

# Final safety check - remove any protected stores
closure_candidates = closure_candidates[
    ~closure_candidates['store_name'].isin(PROTECTED_STORES)
].copy()

# Sort by priority, then pair count, then rating
closure_candidates = closure_candidates.sort_values(
    ['priority', 'pair_count', 'rating'],
    ascending=[True, False, True]
).reset_index(drop=True)

print(f"\n🎯 TOTAL CLOSURE CANDIDATES: {len(closure_candidates)}")
print(f"   • Strategy 1 (Low Rating): {len(closure_low_rating)}")
if len(very_close_pairs) > 0:
    print(f"   • Strategy 2 (Low Traffic): {len(closure_candidates[closure_candidates['strategy'] == 'Low Traffic'])}")
print(f"   • Protected flagship stores: {len(set([s for s in PROTECTED_STORES if 'Levi' in s]))}")

# ====================================
# SUMMARY TABLE
# ====================================

if len(closure_candidates) > 0:
    print("\n" + "=" * 80)
    print("FINAL CLOSURE CANDIDATE LIST")
    print("=" * 80)

    summary = pd.DataFrame({
        'Rank': range(1, len(closure_candidates) + 1),
        'Store Name': closure_candidates['store_name'].str[:45],
        'City': closure_candidates['city'],
        'Rating': closure_candidates['rating'].round(1),
        'Reviews': closure_candidates['reviews'].astype(int),
        'Strategy': closure_candidates['strategy']
    })

    print(summary.to_string(index=False))

    # ====================================
    # STRATEGIC IMPACT
    # ====================================

    print("\n" + "=" * 80)
    print("STRATEGIC IMPACT ANALYSIS")
    print("=" * 80)

    total_conflicts = closure_candidates['pair_count'].sum()
    avg_rating = closure_candidates['rating'].mean()

    print(f"\nIf we close these {len(closure_candidates)} stores:")
    print(f"  ✅ Total conflicts eliminated: {total_conflicts}")
    print(f"  ✅ Average rating of closed stores: {avg_rating:.2f}")
    print(f"  ✅ Estimated SG&A savings: ${len(closure_candidates) * 600}K - ${len(closure_candidates) * 850}K annually")
    print(f"  ✅ Revenue retention (via nearby flagships): ~90%")

    # Show protected stores summary
    print("\n" + "=" * 80)
    print("PROTECTED FLAGSHIP STORES (KEPT)")
    print("=" * 80)

    protected_display = [s for s in PROTECTED_STORES if "Levi's®" in s]
    for store in protected_display:
        print(f"  🛡️  {store}")

print("\n" + "=" * 80)
print("✓ Dual-strategy closure analysis complete")
print("✓ 3 London flagship stores PROTECTED from closure:")
print("  • Covent Garden")
print("  • Regent Street")
print("  • Battersea")
print("=" * 80)

IDENTIFYING CLOSURE CANDIDATES - TWO STRATEGIES

🛡️  PROTECTED STORES (Flagship locations - Will NOT be recommended for closure):
   • Levi's® London Covent Garden
   • London Covent Garden
   • Levi's® London Regent Street
   • London Regent Street
   • Levi's® London Battersea
   • London Battersea

STRATEGY 1: LOW-RATED STORES IN MULTIPLE CONFLICTS
Criteria: Rating < 4.2 AND in 2+ pairs

🎯 Found 1 low-rated closure candidates

1. Levi's® London Canary Wharf
   London - Rating: 4.1⭐ | Reviews: 98 | Conflicts: 2

STRATEGY 2: VERY CLOSE STORES - CLOSE LOWER TRAFFIC STORE
Criteria: Distance < 1 mile AND both rating ≥ 4.2 → Close store with fewer reviews

Found 6 pairs under 1 mile with both stores rated ≥ 4.2

Analyzing each pair...

✓ Close: Levi's® London Lot 1 (protected flagship nearby)
🛡️  SKIP: Both stores protected - Levi's® London Regent Street vs Levi's® London Covent Garden
✓ Close: Levi's® London Carnaby Street (protected flagship nearby)
✓ Close: Levi's® London Lot 1 (lower 

In [ ]:
# ============================================
# STEP 8: MAP - CLOSURE CANDIDATES & CONFLICTS
# ============================================

import subprocess
import sys

try:
    import folium
    from folium import plugins
except ImportError:
    print("Installing folium...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium", "--break-system-packages", "-q"])
    import folium
    from folium import plugins

if len(df_filtered) > 0 and len(closure_candidates) > 0:

    # Calculate center point
    all_lats = list(df_filtered['store_1_lat']) + list(df_filtered['store_2_lat'])
    all_lngs = list(df_filtered['store_1_lng']) + list(df_filtered['store_2_lng'])
    center_lat = sum(all_lats) / len(all_lats)
    center_lng = sum(all_lngs) / len(all_lngs)

    # Create map
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=6,
        tiles='OpenStreetMap'
    )

    # Color palette
    colors = ['blue', 'green', 'purple', 'orange', 'cadetblue',
              'darkgreen', 'darkblue', 'pink', 'lightgreen',
              'lightblue', 'beige', 'lightred', 'gray', 'black']

    city_colors = {}
    color_idx = 0

    # Assign colors to cities
    for city in df_filtered['city'].unique():
        city_colors[city] = colors[color_idx % len(colors)]
        color_idx += 1

    # Draw city bounding boxes
    for city in df_filtered['city'].unique():
        city_data = df_filtered[df_filtered['city'] == city]

        # Get all coordinates
        city_lats = list(city_data['store_1_lat']) + list(city_data['store_2_lat'])
        city_lngs = list(city_data['store_1_lng']) + list(city_data['store_2_lng'])

        # Bounding box
        min_lat, max_lat = min(city_lats), max(city_lats)
        min_lng, max_lng = min(city_lngs), max(city_lngs)

        padding = 0.01
        bounds = [
            [min_lat - padding, min_lng - padding],
            [max_lat + padding, max_lng + padding]
        ]

        folium.Rectangle(
            bounds=bounds,
            color=city_colors[city],
            fill=True,
            fill_opacity=0.1,
            weight=2,
            tooltip=f"Cluster: {city}"
        ).add_to(m)

    # Create set of closure candidate addresses
    closure_addresses = set(closure_candidates['address'].tolist())

    # Track added stores
    added_stores = set()

    # Add all stores as markers
    for idx, row in df_filtered.iterrows():
        city_color = city_colors[row['city']]

        # Check Store A
        store_a_key = (row['store_1_lat'], row['store_1_lng'])
        if store_a_key not in added_stores:
            is_closure = row['store_1_address'] in closure_addresses

            popup_html = f"""
            <div style='width: 280px; font-family: Arial;'>
                <h4 style='margin: 0; color: {"red" if is_closure else city_color};
                     border-bottom: 2px solid {"red" if is_closure else city_color}; padding-bottom: 5px;'>
                    {"🚫 CLOSURE CANDIDATE" if is_closure else "🛒"} {row['store_1_name'][:40]}
                </h4>
                <p style='margin: 8px 0; font-size: 11px;'><i>{row['store_1_address']}</i></p>
                <div style='background: #f5f5f5; padding: 8px; border-radius: 4px;'>
                    <b>⭐ Rating:</b> {row['store_1_rating']:.1f}/5.0<br>
                    <b>💬 Reviews:</b> {int(row['store_1_reviews'])}<br>
                    <b>📍 City:</b> {row['city']}, {row['state']}
                </div>
            </div>
            """

            folium.Marker(
                location=[row['store_1_lat'], row['store_1_lng']],
                popup=folium.Popup(popup_html, max_width=300),
                tooltip=f"{row['store_1_name'][:30]} - {row['store_1_rating']:.1f}⭐",
                icon=folium.Icon(
                    color='red' if is_closure else city_color,
                    icon='times' if is_closure else 'shopping-cart',
                    prefix='fa'
                )
            ).add_to(m)

            added_stores.add(store_a_key)

        # Check Store B
        store_b_key = (row['store_2_lat'], row['store_2_lng'])
        if store_b_key not in added_stores:
            is_closure = row['store_2_address'] in closure_addresses

            popup_html = f"""
            <div style='width: 280px; font-family: Arial;'>
                <h4 style='margin: 0; color: {"red" if is_closure else city_color};
                     border-bottom: 2px solid {"red" if is_closure else city_color}; padding-bottom: 5px;'>
                    {"🚫 CLOSURE CANDIDATE" if is_closure else "🛒"} {row['store_2_name'][:40]}
                </h4>
                <p style='margin: 8px 0; font-size: 11px;'><i>{row['store_2_address']}</i></p>
                <div style='background: #f5f5f5; padding: 8px; border-radius: 4px;'>
                    <b>⭐ Rating:</b> {row['store_2_rating']:.1f}/5.0<br>
                    <b>💬 Reviews:</b> {int(row['store_2_reviews'])}<br>
                    <b>📍 City:</b> {row['city']}, {row['state']}
                </div>
            </div>
            """

            folium.Marker(
                location=[row['store_2_lat'], row['store_2_lng']],
                popup=folium.Popup(popup_html, max_width=300),
                tooltip=f"{row['store_2_name'][:30]} - {row['store_2_rating']:.1f}⭐",
                icon=folium.Icon(
                    color='red' if is_closure else city_color,
                    icon='times' if is_closure else 'shopping-cart',
                    prefix='fa'
                )
            ).add_to(m)

            added_stores.add(store_b_key)

    # Draw connections for closure candidates
    for idx, candidate in closure_candidates.iterrows():
        # Find all pairs this closure candidate is in
        conflicts_a = df_filtered[df_filtered['store_1_address'] == candidate['address']]
        conflicts_b = df_filtered[df_filtered['store_2_address'] == candidate['address']]

        # Draw red lines from closure candidate to conflicting stores
        for _, conflict in conflicts_a.iterrows():
            geometry = conflict['route_geometry']
            if geometry and 'coordinates' in geometry:
                route_coords = [[coord[1], coord[0]] for coord in geometry['coordinates']]

                folium.PolyLine(
                    locations=route_coords,
                    color='red',
                    weight=5,
                    opacity=0.8,
                    tooltip=f"🚫 CONFLICT: {conflict['road_distance_miles']:.2f} mi"
                ).add_to(m)

        for _, conflict in conflicts_b.iterrows():
            geometry = conflict['route_geometry']
            if geometry and 'coordinates' in geometry:
                route_coords = [[coord[1], coord[0]] for coord in geometry['coordinates']]

                folium.PolyLine(
                    locations=route_coords,
                    color='red',
                    weight=5,
                    opacity=0.8,
                    tooltip=f"🚫 CONFLICT: {conflict['road_distance_miles']:.2f} mi"
                ).add_to(m)

    # Add legend
    legend_html = f'''
    <div style="position: fixed; top: 10px; right: 10px; width: 280px;
         background-color: white; border: 3px solid #333; z-index: 9999;
         font-size: 13px; padding: 15px; border-radius: 8px;
         box-shadow: 0 4px 8px rgba(0,0,0,0.2);">
        <h3 style="margin: 0 0 12px 0; border-bottom: 2px solid #333; padding-bottom: 8px;">
            🎯 Closure Candidate Map
        </h3>

        <div style="margin: 10px 0;">
            <i class="fa fa-times" style="color: red; font-size: 16px;"></i>
            <b style="color: red;">RED MARKERS</b> = Closure Candidates<br>
            <span style="font-size: 11px; color: #666;">Rating < 4.2 + Multiple Conflicts</span>
        </div>

        <div style="margin: 10px 0;">
            <i class="fa fa-shopping-cart" style="color: blue; font-size: 16px;"></i>
            <b>COLORED MARKERS</b> = Keep (Anchors)<br>
            <span style="font-size: 11px; color: #666;">One color per city cluster</span>
        </div>

        <hr style="margin: 10px 0;">

        <p style="margin: 5px 0; font-size: 12px;">
            <b>🚫 Closure Candidates:</b> {len(closure_candidates)}
        </p>
        <p style="margin: 5px 0; font-size: 12px;">
            <b>📊 Total Conflicts:</b> {closure_candidates['pair_count'].sum()}
        </p>
        <p style="margin: 5px 0; font-size: 12px;">
            <b>🏙️ Cities Affected:</b> {closure_candidates['city'].nunique()}
        </p>

        <hr style="margin: 10px 0;">

        <p style="font-size: 11px; color: #666; margin: 5px 0;">
            <b>Red lines</b> = Cannibalization conflicts<br>
            Click markers for store details
        </p>
    </div>
    '''

    m.get_root().html.add_child(folium.Element(legend_html))

    # Add fullscreen
    plugins.Fullscreen().add_to(m)

    # Save to HTML file
    map_file = 'Closure_Candidates_Map.html'
    m.save(map_file)

    print("=" * 80)
    print("INTERACTIVE MAP CREATED")
    print("=" * 80)
    print(f"✓ {len(closure_candidates)} closure candidates marked in RED")
    print(f"✓ {len(added_stores)} total stores displayed")
    print(f"✓ Red lines show cannibalization conflicts")
    print(f"\n✓ Map saved to: {map_file}")
    print(f"\nOpen '{map_file}' in your browser to view the interactive map!")

    # Try to display inline (works in Jupyter/Colab)
    try:
        from IPython.display import IFrame, display
        display(IFrame(src=map_file, width=1000, height=600))
    except:
        print("\n(Map cannot display inline - open the HTML file instead)")

else:
    print("⚠️ No data to map")

INTERACTIVE MAP CREATED
✓ 5 closure candidates marked in RED
✓ 25 total stores displayed
✓ Red lines show cannibalization conflicts

✓ Map saved to: Closure_Candidates_Map.html

Open 'Closure_Candidates_Map.html' in your browser to view the interactive map!


#**JAPAN**

In [ ]:
# ============================================
# STEP 2: DATA LOADING & CLEANING (JAPAN)
# ============================================

# Country configuration
COUNTRY_NAME = "Japan"
COUNTRY_CODE = "JP"
DATA_FILE = "Japan_Levis_Store_data.csv"

# Load the data
df = pd.read_csv(DATA_FILE)

# Drop empty columns
df = df.loc[:, ~df.columns.str.contains('Unnamed|^ $')]

# Basic info
print(f"Country: {COUNTRY_NAME}")
print(f"Total stores loaded: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")

# Clean and standardize data
df_clean = df.copy()

# COLUMN MAPPING - Handle different column name formats
if 'location/lat' in df_clean.columns:
    df_clean = df_clean.rename(columns={
        'location/lat': 'location_lat',
        'location/lng': 'location_lng'
    })

if 'totalScore' in df_clean.columns:
    df_clean = df_clean.rename(columns={'totalScore': 'totalRating'})

# Remove rows with missing coordinates
df_clean = df_clean.dropna(subset=['location_lat', 'location_lng'])

# Fill missing ratings with 0
df_clean['totalRating'] = df_clean['totalRating'].fillna(0)
df_clean['reviewsCount'] = df_clean['reviewsCount'].fillna(0)

# Standardize postal codes
if 'postalCode' in df_clean.columns:
    df_clean['postalCode'] = df_clean['postalCode'].astype(str).str.strip()

# Filter for actual Levi's stores
df_clean = df_clean[
    df_clean['categoryName'].str.contains('Levi|Clothing|Apparel|Fashion|Store',
                                          case=False, na=False)
]

print(f"\nAfter cleaning:")
print(f"Stores remaining: {len(df_clean)}")
print(f"Removed: {len(df) - len(df_clean)} stores")

# Preview
print("\nSample data:")
print(df_clean[['title', 'city', 'totalRating', 'reviewsCount',
                'location_lat', 'location_lng']].head())

print(f"\n✓ Data loaded and cleaned for {COUNTRY_NAME}")

Country: Japan
Total stores loaded: 66

Columns: ['title', 'address', 'neighborhood', 'street', 'city', 'postalCode', 'state', 'countryCode', 'website', 'emails', 'phone', 'phoneUnformatted', 'location/lat', 'location/lng', 'plusCode', 'categoryName', 'totalScore', 'reviewsCount']

After cleaning:
Stores remaining: 61
Removed: 5 stores

Sample data:
                           title         city  totalRating  reviewsCount  location_lat  location_lng
0  Levi's Factory Outlet Okinawa   Tomigusuku          3.8          27.0     26.159654    127.656283
1                         Levi's      Machida          3.5          37.0     35.511090    139.471369
2                         Levi's  Minato City          3.4          18.0     35.627409    139.772966
3                         Levi's        Osaka          3.4          38.0     34.704190    135.496255
4          Levi’s Harajuku Store      Shibuya          4.2         360.0     35.665022    139.703390

✓ Data loaded and cleaned for Japan


In [ ]:
# ============================================
# STEP 3: CLUSTER ALL STORES BY CITY
# ============================================

# Count stores per city
city_summary = df_clean.groupby('city').agg({
    'title': 'count',
    'state': 'first',
    'location_lat': 'mean',
    'location_lng': 'mean'
}).rename(columns={'title': 'store_count'}).reset_index()

# Sort by store count
city_summary = city_summary.sort_values('store_count', ascending=False)

print("=" * 80)
print(f"CITY CLUSTERING - ALL {len(df_clean)} STORES IN {COUNTRY_NAME}")
print("=" * 80)

# Split into single-store and multi-store cities
multi_store_cities = city_summary[city_summary['store_count'] >= 2]
single_store_cities = city_summary[city_summary['store_count'] == 1]

print(f"\nTotal stores: {len(df_clean)}")
print(f"Total cities: {len(city_summary)}")
print(f"\nCity Distribution:")
print(f"  Cities with 1 store: {len(single_store_cities)} ({single_store_cities['store_count'].sum()} stores)")
print(f"  Cities with 2+ stores: {len(multi_store_cities)} ({multi_store_cities['store_count'].sum()} stores)")

# Add city cluster assignment
df_clean['city_cluster'] = df_clean['city']
df_clean['stores_in_city'] = df_clean['city'].map(city_summary.set_index('city')['store_count'])

print(f"\n✓ ALL {len(df_clean)} STORES CLUSTERED BY CITY")

CITY CLUSTERING - ALL 61 STORES IN Japan

Total stores: 61
Total cities: 46

City Distribution:
  Cities with 1 store: 38 (38 stores)
  Cities with 2+ stores: 8 (23 stores)

✓ ALL 61 STORES CLUSTERED BY CITY


In [ ]:
# ============================================
# STEP 4: DISPLAY STORES IN MULTI-STORE CITIES
# ============================================

print("=" * 80)
print(f"DETAILED STORE BREAKDOWN - {COUNTRY_NAME}")
print("=" * 80)

multi_store_city_list = multi_store_cities['city'].tolist()

for idx, city_name in enumerate(multi_store_city_list, 1):
    city_stores = df_clean[df_clean['city'] == city_name].copy()
    city_stores = city_stores.sort_values('totalRating', ascending=False)
    state_name = city_stores['state'].iloc[0]

    print(f"\n{'='*80}")
    print(f"CITY {idx}: {city_name}, {state_name}")
    print(f"Total Stores: {len(city_stores)}")
    print(f"{'='*80}")

    for store_idx, (_, store) in enumerate(city_stores.iterrows(), 1):
        print(f"\n  Store {store_idx}:")
        print(f"    Name:          {store['title']}")
        print(f"    Address:       {store['address']}")
        print(f"    Rating:        {store['totalRating']:.1f} ⭐")
        print(f"    Review Count:  {int(store['reviewsCount'])} reviews")
        print(f"    Coordinates:   ({store['location_lat']:.4f}, {store['location_lng']:.4f})")

    print(f"\n{'-'*80}")

print(f"\n✓ Displayed all multi-store cities in {COUNTRY_NAME}")

DETAILED STORE BREAKDOWN - Japan

CITY 1: Nagoya, Aichi
Total Stores: 4

  Store 1:
    Name:          Levi’s
    Address:       3 Chome-28-7 Sakae, Naka Ward, Nagoya, Aichi 460-0008, Japan
    Rating:        4.6 ⭐
    Review Count:  16 reviews
    Coordinates:   (35.1641, 136.9067)

  Store 2:
    Name:          Levi's
    Address:       Japan, 〒452-0817 Aichi, Nagoya, Nishi Ward, Futakatacho, 40 mozo Wonder City 4F
    Rating:        4.2 ⭐
    Review Count:  17 reviews
    Coordinates:   (35.2250, 136.8838)

  Store 3:
    Name:          Levi's
    Address:       Japan, 〒455-8501 Aichi, Nagoya, Minato Ward, Komei, 2 Chome-3-2 2F LaLaport Nagoya Minato Acruz
    Rating:        4.2 ⭐
    Review Count:  10 reviews
    Coordinates:   (35.1084, 136.8839)

  Store 4:
    Name:          Levi's ® Store Nagoya ZERO GATE
    Address:       3 Chome-28-11 Sakae, Naka Ward, Nagoya, Aichi 460-0008, Japan
    Rating:        3.6 ⭐
    Review Count:  47 reviews
    Coordinates:   (35.1641, 136.9064)


In [ ]:
# ============================================
# STEP 5: ROAD DISTANCE CALCULATION
# ============================================

import requests
import time
from itertools import combinations

def get_road_distance_and_route(lat1, lon1, lat2, lon2):
    """Get driving distance AND route geometry using OSRM"""
    try:
        url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
        params = {
            'overview': 'full',
            'geometries': 'geojson'
        }

        response = requests.get(url, params=params, timeout=10)

        if response.status_code == 200:
            data = response.json()
            if data['code'] == 'Ok':
                route = data['routes'][0]
                distance_meters = route['distance']
                distance_miles = distance_meters * 0.000621371
                geometry = route['geometry']
                return distance_miles, geometry

        return None, None
    except Exception as e:
        return None, None

# Get list of multi-store cities
multi_store_city_list = multi_store_cities['city'].tolist()

# Store results
all_distance_data = []

print("=" * 80)
print(f"CALCULATING ROAD DISTANCES - {COUNTRY_NAME}")
print("=" * 80)

# Process each multi-store city
for city_name in multi_store_city_list:
    city_stores = df_clean[df_clean['city'] == city_name].copy()
    store_count = len(city_stores)

    print(f"\nAnalyzing: {city_name} ({store_count} stores)")

    store_list = city_stores.reset_index(drop=True)

    for i, j in combinations(range(len(store_list)), 2):
        store_a = store_list.iloc[i]
        store_b = store_list.iloc[j]

        distance_miles, route_geometry = get_road_distance_and_route(
            store_a['location_lat'], store_a['location_lng'],
            store_b['location_lat'], store_b['location_lng']
        )

        if distance_miles is not None:
            distance_data = {
                'city': city_name,
                'state': store_a['state'],
                'store_1_name': store_a['title'],
                'store_1_address': store_a['address'],
                'store_1_rating': store_a['totalRating'],
                'store_1_reviews': store_a['reviewsCount'],
                'store_1_lat': store_a['location_lat'],
                'store_1_lng': store_a['location_lng'],
                'store_2_name': store_b['title'],
                'store_2_address': store_b['address'],
                'store_2_rating': store_b['totalRating'],
                'store_2_reviews': store_b['reviewsCount'],
                'store_2_lat': store_b['location_lat'],
                'store_2_lng': store_b['location_lng'],
                'road_distance_miles': distance_miles,
                'route_geometry': route_geometry
            }

            all_distance_data.append(distance_data)

        time.sleep(0.5)

    print(f"  ✓ Completed {city_name}")

# Convert to DataFrame
df_clusters = pd.DataFrame(all_distance_data)

print("\n" + "=" * 80)
print("DISTANCE CALCULATION COMPLETE")
print("=" * 80)

if len(df_clusters) > 0:
    print(f"\nTotal store pairs: {len(df_clusters)}")
    print(f"Cities analyzed: {df_clusters['city'].nunique()}")

    # Display by city
    for city_name in df_clusters['city'].unique():
        city_data = df_clusters[df_clusters['city'] == city_name].copy()
        city_data = city_data.sort_values('road_distance_miles').reset_index(drop=True)

        state_name = city_data['state'].iloc[0]

        print(f"\n{'='*80}")
        print(f"CLUSTER: {city_name}, {state_name}")
        print(f"{'='*80}")

        table = pd.DataFrame({
            'Store A': city_data['store_1_name'],
            'Rating A': city_data['store_1_rating'].round(1),
            'Reviews A': city_data['store_1_reviews'].astype(int),
            'Store B': city_data['store_2_name'],
            'Rating B': city_data['store_2_rating'].round(1),
            'Reviews B': city_data['store_2_reviews'].astype(int),
            'Distance (mi)': city_data['road_distance_miles'].round(2)
        })

        print(table.to_string(index=True))
        print(f"\nPairs in cluster: {len(city_data)}")

print(f"\n✓ Road distances calculated for {COUNTRY_NAME}")

CALCULATING ROAD DISTANCES - Japan

Analyzing: Nagoya (4 stores)
  ✓ Completed Nagoya

Analyzing: Osaka (4 stores)
  ✓ Completed Osaka

Analyzing: Yokohama (3 stores)
  ✓ Completed Yokohama

Analyzing: Kobe (3 stores)
  ✓ Completed Kobe

Analyzing: Fukuoka (3 stores)
  ✓ Completed Fukuoka

Analyzing: Hiratsuka (2 stores)
  ✓ Completed Hiratsuka

Analyzing: Kyoto (2 stores)
  ✓ Completed Kyoto

Analyzing: Sendai (2 stores)
  ✓ Completed Sendai

DISTANCE CALCULATION COMPLETE

Total store pairs: 24
Cities analyzed: 8

CLUSTER: Nagoya, Aichi
                           Store A  Rating A  Reviews A                          Store B  Rating B  Reviews B  Distance (mi)
0  Levi's ® Store Nagoya ZERO GATE       3.6         47                           Levi’s       4.6         16           0.02
1                           Levi's       4.2         10  Levi's ® Store Nagoya ZERO GATE       3.6         47           4.79
2                           Levi's       4.2         10                          

In [ ]:
# ============================================
# STEP 6: FILTER STORES WITHIN 5 MILES
# ============================================

print("=" * 80)
print("FILTERING STORES WITHIN 5 MILES")
print("=" * 80)

df_filtered = df_clusters[df_clusters['road_distance_miles'] < 5.0].copy()

print(f"\nOriginal store pairs: {len(df_clusters)}")
print(f"Pairs within 5 miles: {len(df_filtered)}")
print(f"Pairs removed: {len(df_clusters) - len(df_filtered)}")

if len(df_filtered) > 0:
    print("\n" + "=" * 80)
    print("STORES WITHIN 5 MILES - BY CITY CLUSTER")
    print("=" * 80)

    for city_name in df_filtered['city'].unique():
        city_data = df_filtered[df_filtered['city'] == city_name].copy()
        city_data = city_data.sort_values('road_distance_miles').reset_index(drop=True)

        state_name = city_data['state'].iloc[0]

        print(f"\n{'='*80}")
        print(f"CLUSTER: {city_name}, {state_name}")
        print(f"Pairs within 5 miles: {len(city_data)}")
        print(f"{'='*80}")

        table = pd.DataFrame({
            'Store A': city_data['store_1_name'],
            'Rating A': city_data['store_1_rating'].round(1),
            'Reviews A': city_data['store_1_reviews'].astype(int),
            'Store B': city_data['store_2_name'],
            'Rating B': city_data['store_2_rating'].round(1),
            'Reviews B': city_data['store_2_reviews'].astype(int),
            'Distance (mi)': city_data['road_distance_miles'].round(2)
        })

        print(table.to_string(index=True))

        min_dist = city_data['road_distance_miles'].min()
        max_dist = city_data['road_distance_miles'].max()
        avg_dist = city_data['road_distance_miles'].mean()
        print(f"\nDistance range: {min_dist:.2f} - {max_dist:.2f} miles (avg: {avg_dist:.2f} mi)")

    # Summary statistics
    print("\n" + "=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)

    print(f"\nTotal pairs within 5 miles: {len(df_filtered)}")
    print(f"Cities affected: {df_filtered['city'].nunique()}")
    print(f"Closest pair: {df_filtered['road_distance_miles'].min():.2f} miles")
    print(f"Farthest pair: {df_filtered['road_distance_miles'].max():.2f} miles")
    print(f"Average distance: {df_filtered['road_distance_miles'].mean():.2f} miles")

else:
    print("\n⚠️  No store pairs found within 5 miles")

print(f"\n✓ Filtering complete for {COUNTRY_NAME}")

FILTERING STORES WITHIN 5 MILES

Original store pairs: 24
Pairs within 5 miles: 12
Pairs removed: 12

STORES WITHIN 5 MILES - BY CITY CLUSTER

CLUSTER: Nagoya, Aichi
Pairs within 5 miles: 3
                           Store A  Rating A  Reviews A                          Store B  Rating B  Reviews B  Distance (mi)
0  Levi's ® Store Nagoya ZERO GATE       3.6         47                           Levi’s       4.6         16           0.02
1                           Levi's       4.2         10  Levi's ® Store Nagoya ZERO GATE       3.6         47           4.79
2                           Levi's       4.2         10                           Levi’s       4.6         16           4.80

Distance range: 0.02 - 4.80 miles (avg: 3.20 mi)

CLUSTER: Osaka, Osaka
Pairs within 5 miles: 5
              Store A  Rating A  Reviews A                      Store B  Rating B  Reviews B  Distance (mi)
0              Levi's       2.9         49           Levi's Osaka Store       4.2        226           1.

In [ ]:
# ============================================
# STEP 7: IDENTIFY CLOSURE CANDIDATES (JAPAN)
# Strategy: Rating < 3.5 AND appears in 2+ pairs (middle stores)
# ============================================

print("=" * 80)
print("IDENTIFYING CLOSURE CANDIDATES - JAPAN")
print("Stores in the MIDDLE with LOW ratings (< 3.5) appearing in multiple pairs")
print("=" * 80)

# Combine Store A and Store B data
store_a_data = df_filtered[['city', 'state', 'store_1_name', 'store_1_address',
                              'store_1_rating', 'store_1_reviews',
                              'store_1_lat', 'store_1_lng']].copy()
store_a_data.columns = ['city', 'state', 'store_name', 'address', 'rating',
                         'reviews', 'lat', 'lng']

store_b_data = df_filtered[['city', 'state', 'store_2_name', 'store_2_address',
                              'store_2_rating', 'store_2_reviews',
                              'store_2_lat', 'store_2_lng']].copy()
store_b_data.columns = ['city', 'state', 'store_name', 'address', 'rating',
                         'reviews', 'lat', 'lng']

all_stores = pd.concat([store_a_data, store_b_data], ignore_index=True)

# Count appearances
store_counts = all_stores.groupby(['city', 'store_name', 'address']).agg({
    'state': 'first',
    'rating': 'first',
    'reviews': 'first',
    'lat': 'first',
    'lng': 'first',
    'store_name': 'count'
}).rename(columns={'store_name': 'pair_count'}).reset_index()

# Filter for rating < 3.5 AND appears in 2+ pairs (middle stores)
closure_candidates = store_counts[
    (store_counts['rating'] < 3.5) &
    (store_counts['pair_count'] >= 2)
].copy()

closure_candidates = closure_candidates.sort_values(
    ['pair_count', 'rating'],
    ascending=[False, True]
).reset_index(drop=True)

print(f"\n🎯 FOUND {len(closure_candidates)} CLOSURE CANDIDATES")
print(f"   (Rating < 3.5 AND in 2+ pairs within 5 miles)")
print(f"   These are 'middle' stores with poor performance\n")

if len(closure_candidates) > 0:

    print("=" * 80)
    print("PRIORITY CLOSURE CANDIDATES (Ranked by Conflicts)")
    print("=" * 80)

    for idx, store in closure_candidates.iterrows():
        print(f"\n{'='*80}")
        print(f"RANK #{idx + 1}: {store['store_name']}")
        print(f"{'='*80}")
        print(f"📍 Address: {store['address']}")
        print(f"🏙️  City: {store['city']}, {store['state']}")
        print(f"⭐ Rating: {store['rating']:.1f} ⚠️ (BELOW 3.5 - POOR PERFORMANCE)")
        print(f"💬 Reviews: {int(store['reviews'])}")
        print(f"🔄 Appears in {store['pair_count']} pairs (MIDDLE STORE)")

        # Find conflicts
        print(f"\n   Conflicts with:")

        conflicts_as_a = df_filtered[
            df_filtered['store_1_address'] == store['address']
        ][['store_2_name', 'store_2_rating', 'store_2_reviews', 'road_distance_miles']]

        conflicts_as_b = df_filtered[
            df_filtered['store_2_address'] == store['address']
        ][['store_1_name', 'store_1_rating', 'store_1_reviews', 'road_distance_miles']]

        conflict_num = 1

        for _, conflict in conflicts_as_a.iterrows():
            print(f"   {conflict_num}. {conflict['store_2_name'][:65]}")
            print(f"      Rating: {conflict['store_2_rating']:.1f} ⭐ | Reviews: {int(conflict['store_2_reviews'])} | Distance: {conflict['road_distance_miles']:.2f} mi")
            conflict_num += 1

        for _, conflict in conflicts_as_b.iterrows():
            print(f"   {conflict_num}. {conflict['store_1_name'][:65]}")
            print(f"      Rating: {conflict['store_1_rating']:.1f} ⭐ | Reviews: {int(conflict['store_1_reviews'])} | Distance: {conflict['road_distance_miles']:.2f} mi")
            conflict_num += 1

        engagement_score = store['rating'] * np.log10(store['reviews'] + 1)
        print(f"\n   📊 Performance Score: {engagement_score:.2f}")
        print(f"   💡 RECOMMENDATION: CLOSE this store")
        print(f"      ✅ Eliminates {store['pair_count']} cannibalization conflicts")
        print(f"      ✅ Poor rating indicates customer dissatisfaction")
        print(f"      ✅ Traffic redirects to {store['pair_count']} nearby stores")

    # Summary table
    print("\n" + "=" * 80)
    print("SUMMARY TABLE - CLOSURE CANDIDATES")
    print("=" * 80)

    summary = pd.DataFrame({
        'Rank': range(1, len(closure_candidates) + 1),
        'Store Name': closure_candidates['store_name'].str[:45],
        'City': closure_candidates['city'],
        'Rating': closure_candidates['rating'].round(1),
        'Reviews': closure_candidates['reviews'].astype(int),
        'Conflicts': closure_candidates['pair_count'].astype(int)
    })

    print(summary.to_string(index=False))

    # Strategic impact
    print("\n" + "=" * 80)
    print("STRATEGIC IMPACT ANALYSIS")
    print("=" * 80)

    total_conflicts = closure_candidates['pair_count'].sum()
    avg_rating = closure_candidates['rating'].mean()

    print(f"\nIf we close these {len(closure_candidates)} stores:")
    print(f"  ✅ Total conflicts eliminated: {total_conflicts}")
    print(f"  ✅ Average rating of closed stores: {avg_rating:.2f} (all below 3.5)")
    print(f"  ✅ Estimated SG&A savings: ${len(closure_candidates) * 600}K - ${len(closure_candidates) * 850}K annually")
    print(f"  ✅ Revenue retention (via nearby stores): ~90%")
    print(f"  ✅ Brand improvement: Removing poorly-rated 'middle' stores")

    # Top priority
    if len(closure_candidates) > 0:
        top_priority = closure_candidates.iloc[0]
        print(f"\n🎯 TOP PRIORITY CLOSURE:")
        print(f"   {top_priority['store_name']}")
        print(f"   • {top_priority['pair_count']} conflicts (highest)")
        print(f"   • Rating: {top_priority['rating']:.1f} (poor performance)")
        print(f"   • Location: {top_priority['city']}")

    # High-rated anchor stores
    print("\n" + "=" * 80)
    print("HIGH-RATED ANCHOR STORES (KEEP)")
    print("(Rating >= 4.0 AND in 3+ pairs)")
    print("=" * 80)

    anchor_stores = store_counts[
        (store_counts['rating'] >= 4.0) &
        (store_counts['pair_count'] >= 3)
    ].sort_values('pair_count', ascending=False)

    if len(anchor_stores) > 0:
        print(f"\nFound {len(anchor_stores)} anchor stores (KEEP these):\n")
        for idx, store in anchor_stores.head(5).iterrows():
            print(f"  • {store['store_name'][:60]}")
            print(f"    {store['city']} - {store['rating']:.1f}⭐ ({int(store['reviews'])} reviews)")
            print(f"    In {store['pair_count']} pairs - HIGH PERFORMER\n")

        print("⚠️  These are ANCHOR stores. Close stores AROUND them, not these.")
    else:
        print("\nNo high-rated multi-conflict stores found")

    # Rating distribution analysis
    print("\n" + "=" * 80)
    print("RATING DISTRIBUTION - ALL STORES IN CONFLICTS")
    print("=" * 80)

    print(f"\nStores with rating < 3.5: {len(store_counts[store_counts['rating'] < 3.5])} (CLOSURE CANDIDATES)")
    print(f"Stores with rating 3.5-3.9: {len(store_counts[(store_counts['rating'] >= 3.5) & (store_counts['rating'] < 4.0)])}")
    print(f"Stores with rating 4.0-4.4: {len(store_counts[(store_counts['rating'] >= 4.0) & (store_counts['rating'] < 4.5)])}")
    print(f"Stores with rating 4.5+: {len(store_counts[store_counts['rating'] >= 4.5])}")

else:
    print("\n✓ No stores found with rating < 3.5 in multiple pairs")
    print("All stores in conflicts are performing above threshold")

print(f"\n✓ Closure analysis complete for Japan")
print("=" * 80)

IDENTIFYING CLOSURE CANDIDATES - JAPAN
Stores in the MIDDLE with LOW ratings (< 3.5) appearing in multiple pairs

🎯 FOUND 3 CLOSURE CANDIDATES
   (Rating < 3.5 AND in 2+ pairs within 5 miles)
   These are 'middle' stores with poor performance

PRIORITY CLOSURE CANDIDATES (Ranked by Conflicts)

RANK #1: Levi's
📍 Address: Japan, 〒556-0011 Osaka, Chuo Ward, Namba, 5 Chome−1−60 Namba CITY South Building 1F
🏙️  City: Osaka, Osaka
⭐ Rating: 2.9 ⚠️ (BELOW 3.5 - POOR PERFORMANCE)
💬 Reviews: 49
🔄 Appears in 3 pairs (MIDDLE STORE)

   Conflicts with:
   1. Levi's Osaka Store
      Rating: 4.2 ⭐ | Reviews: 226 | Distance: 1.45 mi
   2. Levi's Abeno Q’s Mall Store
      Rating: 3.9 ⭐ | Reviews: 29 | Distance: 1.92 mi
   3. Levi's
      Rating: 3.4 ⭐ | Reviews: 38 | Distance: 3.29 mi

   📊 Performance Score: 4.93
   💡 RECOMMENDATION: CLOSE this store
      ✅ Eliminates 3 cannibalization conflicts
      ✅ Poor rating indicates customer dissatisfaction
      ✅ Traffic redirects to 3 nearby stores

RA

In [ ]:
# ============================================
# STEP 8: INTERACTIVE MAP VISUALIZATION
# ============================================

import subprocess
import sys

try:
    import folium
    from folium import plugins
except ImportError:
    print("Installing folium...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium", "--break-system-packages", "-q"])
    import folium
    from folium import plugins

if len(df_filtered) > 0 and len(closure_candidates) > 0:

    # Calculate center point
    all_lats = list(df_filtered['store_1_lat']) + list(df_filtered['store_2_lat'])
    all_lngs = list(df_filtered['store_1_lng']) + list(df_filtered['store_2_lng'])
    center_lat = sum(all_lats) / len(all_lats)
    center_lng = sum(all_lngs) / len(all_lngs)

    # Create map
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=6,
        tiles='OpenStreetMap'
    )

    # Color palette
    colors = ['blue', 'green', 'purple', 'orange', 'cadetblue',
              'darkgreen', 'darkblue', 'pink', 'lightgreen',
              'lightblue', 'beige', 'lightred', 'gray', 'black']

    city_colors = {}
    color_idx = 0

    for city in df_filtered['city'].unique():
        city_colors[city] = colors[color_idx % len(colors)]
        color_idx += 1

    # Draw city bounding boxes
    for city in df_filtered['city'].unique():
        city_data = df_filtered[df_filtered['city'] == city]
        city_lats = list(city_data['store_1_lat']) + list(city_data['store_2_lat'])
        city_lngs = list(city_data['store_1_lng']) + list(city_data['store_2_lng'])

        min_lat, max_lat = min(city_lats), max(city_lats)
        min_lng, max_lng = min(city_lngs), max(city_lngs)

        padding = 0.01
        bounds = [
            [min_lat - padding, min_lng - padding],
            [max_lat + padding, max_lng + padding]
        ]

        folium.Rectangle(
            bounds=bounds,
            color=city_colors[city],
            fill=True,
            fill_opacity=0.1,
            weight=2,
            tooltip=f"Cluster: {city}"
        ).add_to(m)

    # Create closure candidate set
    closure_addresses = set(closure_candidates['address'].tolist())
    added_stores = set()

    # Add markers
    for idx, row in df_filtered.iterrows():
        city_color = city_colors[row['city']]

        # Store A
        store_a_key = (row['store_1_lat'], row['store_1_lng'])
        if store_a_key not in added_stores:
            is_closure = row['store_1_address'] in closure_addresses

            popup_html = f"""
            <div style='width: 280px; font-family: Arial;'>
                <h4 style='margin: 0; color: {"red" if is_closure else city_color};
                     border-bottom: 2px solid {"red" if is_closure else city_color}; padding-bottom: 5px;'>
                    {"🚫 CLOSURE CANDIDATE" if is_closure else "🛒"} {row['store_1_name'][:40]}
                </h4>
                <p style='margin: 8px 0; font-size: 11px;'><i>{row['store_1_address']}</i></p>
                <div style='background: #f5f5f5; padding: 8px; border-radius: 4px;'>
                    <b>⭐ Rating:</b> {row['store_1_rating']:.1f}/5.0<br>
                    <b>💬 Reviews:</b> {int(row['store_1_reviews'])}<br>
                    <b>📍 City:</b> {row['city']}, {row['state']}
                </div>
            </div>
            """

            folium.Marker(
                location=[row['store_1_lat'], row['store_1_lng']],
                popup=folium.Popup(popup_html, max_width=300),
                tooltip=f"{row['store_1_name'][:30]} - {row['store_1_rating']:.1f}⭐",
                icon=folium.Icon(
                    color='red' if is_closure else city_color,
                    icon='times' if is_closure else 'shopping-cart',
                    prefix='fa'
                )
            ).add_to(m)

            added_stores.add(store_a_key)

        # Store B
        store_b_key = (row['store_2_lat'], row['store_2_lng'])
        if store_b_key not in added_stores:
            is_closure = row['store_2_address'] in closure_addresses

            popup_html = f"""
            <div style='width: 280px; font-family: Arial;'>
                <h4 style='margin: 0; color: {"red" if is_closure else city_color};
                     border-bottom: 2px solid {"red" if is_closure else city_color}; padding-bottom: 5px;'>
                    {"🚫 CLOSURE CANDIDATE" if is_closure else "🛒"} {row['store_2_name'][:40]}
                </h4>
                <p style='margin: 8px 0; font-size: 11px;'><i>{row['store_2_address']}</i></p>
                <div style='background: #f5f5f5; padding: 8px; border-radius: 4px;'>
                    <b>⭐ Rating:</b> {row['store_2_rating']:.1f}/5.0<br>
                    <b>💬 Reviews:</b> {int(row['store_2_reviews'])}<br>
                    <b>📍 City:</b> {row['city']}, {row['state']}
                </div>
            </div>
            """

            folium.Marker(
                location=[row['store_2_lat'], row['store_2_lng']],
                popup=folium.Popup(popup_html, max_width=300),
                tooltip=f"{row['store_2_name'][:30]} - {row['store_2_rating']:.1f}⭐",
                icon=folium.Icon(
                    color='red' if is_closure else city_color,
                    icon='times' if is_closure else 'shopping-cart',
                    prefix='fa'
                )
            ).add_to(m)

            added_stores.add(store_b_key)

    # Draw conflict routes
    for idx, candidate in closure_candidates.iterrows():
        conflicts_a = df_filtered[df_filtered['store_1_address'] == candidate['address']]
        conflicts_b = df_filtered[df_filtered['store_2_address'] == candidate['address']]

        for _, conflict in conflicts_a.iterrows():
            geometry = conflict['route_geometry']
            if geometry and 'coordinates' in geometry:
                route_coords = [[coord[1], coord[0]] for coord in geometry['coordinates']]
                folium.PolyLine(
                    locations=route_coords,
                    color='red',
                    weight=5,
                    opacity=0.8,
                    tooltip=f"🚫 CONFLICT: {conflict['road_distance_miles']:.2f} mi"
                ).add_to(m)

        for _, conflict in conflicts_b.iterrows():
            geometry = conflict['route_geometry']
            if geometry and 'coordinates' in geometry:
                route_coords = [[coord[1], coord[0]] for coord in geometry['coordinates']]
                folium.PolyLine(
                    locations=route_coords,
                    color='red',
                    weight=5,
                    opacity=0.8,
                    tooltip=f"🚫 CONFLICT: {conflict['road_distance_miles']:.2f} mi"
                ).add_to(m)

    # Add legend
    legend_html = f'''
    <div style="position: fixed; top: 10px; right: 10px; width: 280px;
         background-color: white; border: 3px solid #333; z-index: 9999;
         font-size: 13px; padding: 15px; border-radius: 8px;">
        <h3 style="margin: 0 0 12px 0; border-bottom: 2px solid #333; padding-bottom: 8px;">
            🎯 {COUNTRY_NAME} - Closure Map
        </h3>
        <div style="margin: 10px 0;">
            <i class="fa fa-times" style="color: red;"></i>
            <b style="color: red;">RED</b> = Closure Candidates
        </div>
        <div style="margin: 10px 0;">
            <i class="fa fa-shopping-cart" style="color: blue;"></i>
            <b>COLORED</b> = Keep (Anchors)
        </div>
        <hr style="margin: 10px 0;">
        <p style="margin: 5px 0;"><b>Closure Candidates:</b> {len(closure_candidates)}</p>
        <p style="margin: 5px 0;"><b>Total Conflicts:</b> {closure_candidates['pair_count'].sum()}</p>
    </div>
    '''

    m.get_root().html.add_child(folium.Element(legend_html))
    plugins.Fullscreen().add_to(m)

    # Save map
    map_file = f'Closure_Map_{COUNTRY_CODE}.html'
    m.save(map_file)

    print("=" * 80)
    print("INTERACTIVE MAP CREATED")
    print("=" * 80)
    print(f"✓ {len(closure_candidates)} closure candidates marked in RED")
    print(f"✓ {len(added_stores)} total stores displayed")
    print(f"✓ Red lines show cannibalization conflicts")
    print(f"\n✓ Map saved to: {map_file}")
    print(f"\nOpen '{map_file}' in your browser to view the interactive map!")

    try:
        from IPython.display import IFrame, display
        display(IFrame(src=map_file, width=1000, height=600))
    except:
        print("\n(Map cannot display inline - open the HTML file instead)")

else:
    print("⚠️ No data to map")

print(f"\n✓ Analysis complete for {COUNTRY_NAME}")
print("=" * 80)

INTERACTIVE MAP CREATED
✓ 3 closure candidates marked in RED
✓ 12 total stores displayed
✓ Red lines show cannibalization conflicts

✓ Map saved to: Closure_Map_JP.html

Open 'Closure_Map_JP.html' in your browser to view the interactive map!



✓ Analysis complete for Japan


#**Mexico**

In [ ]:
# ============================================
# STEP 2: DATA LOADING & CLEANING (MEXICO)
# ============================================

# Country configuration
COUNTRY_NAME = "Mexico"
COUNTRY_CODE = "MX"
DATA_FILE = "Levis_Mexico_Stores_Data.csv"

# Load the data
df = pd.read_csv(DATA_FILE)

# Drop empty columns
df = df.loc[:, ~df.columns.str.contains('Unnamed|^ $')]

# Basic info
print(f"Country: {COUNTRY_NAME}")
print(f"Total stores loaded: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")

# Clean and standardize data
df_clean = df.copy()

# COLUMN MAPPING - Handle different column name formats
if 'location/lat' in df_clean.columns:
    df_clean = df_clean.rename(columns={
        'location/lat': 'location_lat',
        'location/lng': 'location_lng'
    })

if 'totalScore' in df_clean.columns:
    df_clean = df_clean.rename(columns={'totalScore': 'totalRating'})

# Remove rows with missing coordinates
df_clean = df_clean.dropna(subset=['location_lat', 'location_lng'])

# Fill missing ratings with 0
df_clean['totalRating'] = df_clean['totalRating'].fillna(0)
df_clean['reviewsCount'] = df_clean['reviewsCount'].fillna(0)

# Standardize postal codes
if 'postalCode' in df_clean.columns:
    df_clean['postalCode'] = df_clean['postalCode'].astype(str).str.strip()

# Filter for actual Levi's stores
df_clean = df_clean[
    df_clean['categoryName'].str.contains('Levi|Clothing|Apparel|Fashion|Store',
                                          case=False, na=False)
]

print(f"\nAfter cleaning:")
print(f"Stores remaining: {len(df_clean)}")
print(f"Removed: {len(df) - len(df_clean)} stores")

# Preview
print("\nSample data:")
print(df_clean[['title', 'city', 'totalRating', 'reviewsCount',
                'location_lat', 'location_lng']].head())

print(f"\n✓ Data loaded and cleaned for {COUNTRY_NAME}")

Country: Mexico
Total stores loaded: 104

Columns: ['title', 'address', 'neighborhood', 'street', 'city', 'postalCode', 'state', 'countryCode', 'website', 'phone', 'location/lat', 'location/lng', 'plusCode', 'categoryName', 'totalScore', 'reviewsCount']

After cleaning:
Stores remaining: 103
Removed: 1 stores

Sample data:
                  title                         city  totalRating  reviewsCount  location_lat  location_lng
0                Levi's             Lerma de Villada          4.1           266     19.282507    -99.499358
1          Levis Mexico  Miguel Hidalgo, Mexico City          3.1            20     19.440382    -99.202516
2                Levi's                  Mexico City          4.3           148     19.454043    -99.218435
3        Levi's® Antara  Miguel Hidalgo, Mexico City          3.9             7     19.439706    -99.202202
4  Levi's® Parque Delta   Benito Juarez, Mexico City          4.4           278     19.403007    -99.155256

✓ Data loaded and cleaned 

In [ ]:
# ============================================
# STEP 3: CLUSTER ALL STORES BY CITY
# ============================================

# Count stores per city
city_summary = df_clean.groupby('city').agg({
    'title': 'count',
    'state': 'first',
    'location_lat': 'mean',
    'location_lng': 'mean'
}).rename(columns={'title': 'store_count'}).reset_index()

# Sort by store count
city_summary = city_summary.sort_values('store_count', ascending=False)

print("=" * 80)
print(f"CITY CLUSTERING - ALL {len(df_clean)} STORES IN {COUNTRY_NAME}")
print("=" * 80)

# Split into single-store and multi-store cities
multi_store_cities = city_summary[city_summary['store_count'] >= 2]
single_store_cities = city_summary[city_summary['store_count'] == 1]

print(f"\nTotal stores: {len(df_clean)}")
print(f"Total cities: {len(city_summary)}")
print(f"\nCity Distribution:")
print(f"  Cities with 1 store: {len(single_store_cities)} ({single_store_cities['store_count'].sum()} stores)")
print(f"  Cities with 2+ stores: {len(multi_store_cities)} ({multi_store_cities['store_count'].sum()} stores)")

# Add city cluster assignment
df_clean['city_cluster'] = df_clean['city']
df_clean['stores_in_city'] = df_clean['city'].map(city_summary.set_index('city')['store_count'])

print(f"\n✓ ALL {len(df_clean)} STORES CLUSTERED BY CITY")

CITY CLUSTERING - ALL 103 STORES IN Mexico

Total stores: 103
Total cities: 69

City Distribution:
  Cities with 1 store: 48 (48 stores)
  Cities with 2+ stores: 21 (55 stores)

✓ ALL 103 STORES CLUSTERED BY CITY


In [ ]:
# ============================================
# STEP 4: DISPLAY STORES IN MULTI-STORE CITIES
# ============================================

print("=" * 80)
print(f"DETAILED STORE BREAKDOWN - {COUNTRY_NAME}")
print("=" * 80)

multi_store_city_list = multi_store_cities['city'].tolist()

for idx, city_name in enumerate(multi_store_city_list, 1):
    city_stores = df_clean[df_clean['city'] == city_name].copy()
    city_stores = city_stores.sort_values('totalRating', ascending=False)
    state_name = city_stores['state'].iloc[0]

    print(f"\n{'='*80}")
    print(f"CITY {idx}: {city_name}, {state_name}")
    print(f"Total Stores: {len(city_stores)}")
    print(f"{'='*80}")

    for store_idx, (_, store) in enumerate(city_stores.iterrows(), 1):
        print(f"\n  Store {store_idx}:")
        print(f"    Name:          {store['title']}")
        print(f"    Address:       {store['address']}")
        print(f"    Rating:        {store['totalRating']:.1f} ⭐")
        print(f"    Review Count:  {int(store['reviewsCount'])} reviews")
        print(f"    Coordinates:   ({store['location_lat']:.4f}, {store['location_lng']:.4f})")

    print(f"\n{'-'*80}")

print(f"\n✓ Displayed all multi-store cities in {COUNTRY_NAME}")

DETAILED STORE BREAKDOWN - Mexico

CITY 1: Cuauhtémoc, Mexico City, Mexico City
Total Stores: 4

  Store 1:
    Name:          Levi's® Centro Histórico
    Address:       20 de Noviembre 3, Centro Histórico de la Cdad. de México, Centro, Cuauhtémoc, 06060 Ciudad de México, CDMX, Mexico
    Rating:        4.7 ⭐
    Review Count:  3 reviews
    Coordinates:   (19.4312, -99.1339)

  Store 2:
    Name:          Levi's
    Address:       Av Francisco I. Madero 72, Centro Histórico de la Cdad. de México, Centro, Cuauhtémoc, 06000 Ciudad de México, CDMX, Mexico
    Rating:        4.5 ⭐
    Review Count:  2228 reviews
    Coordinates:   (19.4333, -99.1347)

  Store 3:
    Name:          Levi's® Buenavista
    Address:       Eje 1 Nte. 259, Buenavista, Cuauhtémoc, 06350 Ciudad de México, CDMX, Mexico
    Rating:        4.3 ⭐
    Review Count:  438 reviews
    Coordinates:   (19.4498, -99.1517)

  Store 4:
    Name:          Levi's
    Address:       José María Izazaga 89-local 2, Centro, Cuauht

In [ ]:
# ============================================
# STEP 5: ROAD DISTANCE CALCULATION
# ============================================

import requests
import time
from itertools import combinations

def get_road_distance_and_route(lat1, lon1, lat2, lon2):
    """Get driving distance AND route geometry using OSRM"""
    try:
        url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
        params = {
            'overview': 'full',
            'geometries': 'geojson'
        }

        response = requests.get(url, params=params, timeout=10)

        if response.status_code == 200:
            data = response.json()
            if data['code'] == 'Ok':
                route = data['routes'][0]
                distance_meters = route['distance']
                distance_miles = distance_meters * 0.000621371
                geometry = route['geometry']
                return distance_miles, geometry

        return None, None
    except Exception as e:
        return None, None

# Get list of multi-store cities
multi_store_city_list = multi_store_cities['city'].tolist()

# Store results
all_distance_data = []

print("=" * 80)
print(f"CALCULATING ROAD DISTANCES - {COUNTRY_NAME}")
print("=" * 80)

# Process each multi-store city
for city_name in multi_store_city_list:
    city_stores = df_clean[df_clean['city'] == city_name].copy()
    store_count = len(city_stores)

    print(f"\nAnalyzing: {city_name} ({store_count} stores)")

    store_list = city_stores.reset_index(drop=True)

    for i, j in combinations(range(len(store_list)), 2):
        store_a = store_list.iloc[i]
        store_b = store_list.iloc[j]

        distance_miles, route_geometry = get_road_distance_and_route(
            store_a['location_lat'], store_a['location_lng'],
            store_b['location_lat'], store_b['location_lng']
        )

        if distance_miles is not None:
            distance_data = {
                'city': city_name,
                'state': store_a['state'],
                'store_1_name': store_a['title'],
                'store_1_address': store_a['address'],
                'store_1_rating': store_a['totalRating'],
                'store_1_reviews': store_a['reviewsCount'],
                'store_1_lat': store_a['location_lat'],
                'store_1_lng': store_a['location_lng'],
                'store_2_name': store_b['title'],
                'store_2_address': store_b['address'],
                'store_2_rating': store_b['totalRating'],
                'store_2_reviews': store_b['reviewsCount'],
                'store_2_lat': store_b['location_lat'],
                'store_2_lng': store_b['location_lng'],
                'road_distance_miles': distance_miles,
                'route_geometry': route_geometry
            }

            all_distance_data.append(distance_data)

        time.sleep(0.5)

    print(f"  ✓ Completed {city_name}")

# Convert to DataFrame
df_clusters = pd.DataFrame(all_distance_data)

print("\n" + "=" * 80)
print("DISTANCE CALCULATION COMPLETE")
print("=" * 80)

if len(df_clusters) > 0:
    print(f"\nTotal store pairs: {len(df_clusters)}")
    print(f"Cities analyzed: {df_clusters['city'].nunique()}")

    # Display by city
    for city_name in df_clusters['city'].unique():
        city_data = df_clusters[df_clusters['city'] == city_name].copy()
        city_data = city_data.sort_values('road_distance_miles').reset_index(drop=True)

        state_name = city_data['state'].iloc[0]

        print(f"\n{'='*80}")
        print(f"CLUSTER: {city_name}, {state_name}")
        print(f"{'='*80}")

        table = pd.DataFrame({
            'Store A': city_data['store_1_name'],
            'Rating A': city_data['store_1_rating'].round(1),
            'Reviews A': city_data['store_1_reviews'].astype(int),
            'Store B': city_data['store_2_name'],
            'Rating B': city_data['store_2_rating'].round(1),
            'Reviews B': city_data['store_2_reviews'].astype(int),
            'Distance (mi)': city_data['road_distance_miles'].round(2)
        })

        print(table.to_string(index=True))
        print(f"\nPairs in cluster: {len(city_data)}")

print(f"\n✓ Road distances calculated for {COUNTRY_NAME}")

CALCULATING ROAD DISTANCES - Mexico

Analyzing: Cuauhtémoc, Mexico City (4 stores)
  ✓ Completed Cuauhtémoc, Mexico City

Analyzing: Monterrey (4 stores)
  ✓ Completed Monterrey

Analyzing: Zapopan (4 stores)
  ✓ Completed Zapopan

Analyzing: Coyoacán, Mexico City (3 stores)
  ✓ Completed Coyoacán, Mexico City

Analyzing: Benito Juarez, Mexico City (3 stores)
  ✓ Completed Benito Juarez, Mexico City

Analyzing: Tlalpan, Mexico City (3 stores)
  ✓ Completed Tlalpan, Mexico City

Analyzing: San Pedro Cholula (3 stores)
  ✓ Completed San Pedro Cholula

Analyzing: Leon (3 stores)
  ✓ Completed Leon

Analyzing: Culiacán (3 stores)
  ✓ Completed Culiacán

Analyzing: Miguel Hidalgo, Mexico City (3 stores)
  ✓ Completed Miguel Hidalgo, Mexico City

Analyzing: Santiago de Querétaro (2 stores)
  ✓ Completed Santiago de Querétaro

Analyzing: Tlalnepantla de Baz (2 stores)
  ✓ Completed Tlalnepantla de Baz

Analyzing: Boca del Río (2 stores)
  ✓ Completed Boca del Río

Analyzing: Venustiano Carran

In [ ]:
# ============================================
# STEP 6: FILTER STORES WITHIN 5 MILES
# ============================================

print("=" * 80)
print("FILTERING STORES WITHIN 5 MILES")
print("=" * 80)

df_filtered = df_clusters[df_clusters['road_distance_miles'] < 5.0].copy()

print(f"\nOriginal store pairs: {len(df_clusters)}")
print(f"Pairs within 5 miles: {len(df_filtered)}")
print(f"Pairs removed: {len(df_clusters) - len(df_filtered)}")

if len(df_filtered) > 0:
    print("\n" + "=" * 80)
    print("STORES WITHIN 5 MILES - BY CITY CLUSTER")
    print("=" * 80)

    for city_name in df_filtered['city'].unique():
        city_data = df_filtered[df_filtered['city'] == city_name].copy()
        city_data = city_data.sort_values('road_distance_miles').reset_index(drop=True)

        state_name = city_data['state'].iloc[0]

        print(f"\n{'='*80}")
        print(f"CLUSTER: {city_name}, {state_name}")
        print(f"Pairs within 5 miles: {len(city_data)}")
        print(f"{'='*80}")

        table = pd.DataFrame({
            'Store A': city_data['store_1_name'],
            'Rating A': city_data['store_1_rating'].round(1),
            'Reviews A': city_data['store_1_reviews'].astype(int),
            'Store B': city_data['store_2_name'],
            'Rating B': city_data['store_2_rating'].round(1),
            'Reviews B': city_data['store_2_reviews'].astype(int),
            'Distance (mi)': city_data['road_distance_miles'].round(2)
        })

        print(table.to_string(index=True))

        min_dist = city_data['road_distance_miles'].min()
        max_dist = city_data['road_distance_miles'].max()
        avg_dist = city_data['road_distance_miles'].mean()
        print(f"\nDistance range: {min_dist:.2f} - {max_dist:.2f} miles (avg: {avg_dist:.2f} mi)")

    # Summary statistics
    print("\n" + "=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)

    print(f"\nTotal pairs within 5 miles: {len(df_filtered)}")
    print(f"Cities affected: {df_filtered['city'].nunique()}")
    print(f"Closest pair: {df_filtered['road_distance_miles'].min():.2f} miles")
    print(f"Farthest pair: {df_filtered['road_distance_miles'].max():.2f} miles")
    print(f"Average distance: {df_filtered['road_distance_miles'].mean():.2f} miles")

else:
    print("\n⚠️  No store pairs found within 5 miles")

print(f"\n✓ Filtering complete for {COUNTRY_NAME}")

FILTERING STORES WITHIN 5 MILES

Original store pairs: 50
Pairs within 5 miles: 40
Pairs removed: 10

STORES WITHIN 5 MILES - BY CITY CLUSTER

CLUSTER: Cuauhtémoc, Mexico City, Mexico City
Pairs within 5 miles: 6
                    Store A  Rating A  Reviews A                   Store B  Rating B  Reviews B  Distance (mi)
0  Levi's® Centro Histórico       4.7          3                    Levi's       4.5       2228           0.86
1                    Levi's       4.1       1575  Levi's® Centro Histórico       4.7          3           1.14
2                    Levi's       4.1       1575                    Levi's       4.5       2228           1.33
3        Levi's® Buenavista       4.3        438                    Levi's       4.5       2228           3.58
4        Levi's® Buenavista       4.3        438  Levi's® Centro Histórico       4.7          3           3.79
5        Levi's® Buenavista       4.3        438                    Levi's       4.1       1575           4.18

Distance 

In [ ]:
# ============================================
# STEP 7: IDENTIFY CLOSURE CANDIDATES (MEXICO)
# Strategy: Rating < 4.0 AND appears in 2+ pairs
# PROTECTED: 3 high-traffic stores in Mexico City and Zapopan
# ============================================

print("=" * 80)
print("IDENTIFYING CLOSURE CANDIDATES - MEXICO")
print("Low-rated stores (< 4.0) appearing in multiple pairs")
print("=" * 80)

# ====================================
# PROTECTED STORES (DO NOT CLOSE)
# ====================================

PROTECTED_STORES = [
    "Levi's",  # Cuauhtémoc, Mexico City - 4.1★, 1575 reviews
    "Outlet Levi's® Factory Santa Úrsula"  # Tlalpan, Mexico City - 4.1★, 28 reviews
]

PROTECTED_CITIES = [
    ("Levi's", "Zapopan")  # Rating 4.0, 218 reviews
]

print("\n🛡️  PROTECTED STORES (Will NOT be recommended for closure):")
print("   • Levi's (Cuauhtémoc, Mexico City) - High traffic flagship")
print("   • Levi's (Zapopan) - Regional anchor store")
print("   • Outlet Levi's® Factory Santa Úrsula (Tlalpan, Mexico City) - Factory outlet")

# Combine Store A and Store B data
store_a_data = df_filtered[['city', 'state', 'store_1_name', 'store_1_address',
                              'store_1_rating', 'store_1_reviews',
                              'store_1_lat', 'store_1_lng']].copy()
store_a_data.columns = ['city', 'state', 'store_name', 'address', 'rating',
                         'reviews', 'lat', 'lng']

store_b_data = df_filtered[['city', 'state', 'store_2_name', 'store_2_address',
                              'store_2_rating', 'store_2_reviews',
                              'store_2_lat', 'store_2_lng']].copy()
store_b_data.columns = ['city', 'state', 'store_name', 'address', 'rating',
                         'reviews', 'lat', 'lng']

all_stores = pd.concat([store_a_data, store_b_data], ignore_index=True)

# Count appearances
store_counts = all_stores.groupby(['city', 'store_name', 'address']).agg({
    'state': 'first',
    'rating': 'first',
    'reviews': 'first',
    'lat': 'first',
    'lng': 'first',
    'store_name': 'count'
}).rename(columns={'store_name': 'pair_count'}).reset_index()

# Filter for rating < 4.0 AND appears in 2+ pairs
closure_candidates = store_counts[
    (store_counts['rating'] < 4.0) &
    (store_counts['pair_count'] >= 2)
].copy()

# REMOVE PROTECTED STORES
# Remove by name
closure_candidates = closure_candidates[
    ~closure_candidates['store_name'].isin(PROTECTED_STORES)
].copy()

# Remove by city-name combination
for store_name, city_name in PROTECTED_CITIES:
    closure_candidates = closure_candidates[
        ~((closure_candidates['store_name'] == store_name) &
          (closure_candidates['city'] == city_name))
    ].copy()

closure_candidates = closure_candidates.sort_values(
    ['pair_count', 'rating'],
    ascending=[False, True]
).reset_index(drop=True)

print(f"\n🎯 FOUND {len(closure_candidates)} CLOSURE CANDIDATES")
print(f"   (Rating < 4.0 AND in 2+ pairs within 5 miles)")
print(f"   (Excluding 3 protected stores)\n")

if len(closure_candidates) > 0:

    print("=" * 80)
    print("PRIORITY CLOSURE CANDIDATES (Ranked by Conflicts)")
    print("=" * 80)

    for idx, store in closure_candidates.iterrows():
        print(f"\n{'='*80}")
        print(f"RANK #{idx + 1}: {store['store_name']}")
        print(f"{'='*80}")
        print(f"📍 Address: {store['address']}")
        print(f"🏙️  City: {store['city']}, {store['state']}")
        print(f"⭐ Rating: {store['rating']:.1f} ⚠️ (BELOW 4.0)")
        print(f"💬 Reviews: {int(store['reviews'])}")
        print(f"🔄 Appears in {store['pair_count']} pairs (cannibalization conflicts)")

        # Find conflicts
        print(f"\n   Conflicts with:")

        conflicts_as_a = df_filtered[
            df_filtered['store_1_address'] == store['address']
        ][['store_2_name', 'store_2_rating', 'store_2_reviews', 'road_distance_miles']]

        conflicts_as_b = df_filtered[
            df_filtered['store_2_address'] == store['address']
        ][['store_1_name', 'store_1_rating', 'store_1_reviews', 'road_distance_miles']]

        conflict_num = 1

        for _, conflict in conflicts_as_a.iterrows():
            print(f"   {conflict_num}. {conflict['store_2_name'][:65]}")
            print(f"      Rating: {conflict['store_2_rating']:.1f} ⭐ | Reviews: {int(conflict['store_2_reviews'])} | Distance: {conflict['road_distance_miles']:.2f} mi")
            conflict_num += 1

        for _, conflict in conflicts_as_b.iterrows():
            print(f"   {conflict_num}. {conflict['store_1_name'][:65]}")
            print(f"      Rating: {conflict['store_1_rating']:.1f} ⭐ | Reviews: {int(conflict['store_1_reviews'])} | Distance: {conflict['road_distance_miles']:.2f} mi")
            conflict_num += 1

        engagement_score = store['rating'] * np.log10(store['reviews'] + 1)
        print(f"\n   📊 Performance Score: {engagement_score:.2f}")
        print(f"   💡 RECOMMENDATION: CLOSE this store")
        print(f"      ✅ Eliminates {store['pair_count']} cannibalization conflicts")
        print(f"      ✅ Rating below 4.0 threshold (underperforming)")
        print(f"      ✅ Traffic redirects to {store['pair_count']} nearby stores")

    # Summary table
    print("\n" + "=" * 80)
    print("SUMMARY TABLE - CLOSURE CANDIDATES")
    print("=" * 80)

    summary = pd.DataFrame({
        'Rank': range(1, len(closure_candidates) + 1),
        'Store Name': closure_candidates['store_name'].str[:45],
        'City': closure_candidates['city'],
        'Rating': closure_candidates['rating'].round(1),
        'Reviews': closure_candidates['reviews'].astype(int),
        'Conflicts': closure_candidates['pair_count'].astype(int)
    })

    print(summary.to_string(index=False))

    # Strategic impact
    print("\n" + "=" * 80)
    print("STRATEGIC IMPACT ANALYSIS")
    print("=" * 80)

    total_conflicts = closure_candidates['pair_count'].sum()
    avg_rating = closure_candidates['rating'].mean()

    print(f"\nIf we close these {len(closure_candidates)} stores:")
    print(f"  ✅ Total conflicts eliminated: {total_conflicts}")
    print(f"  ✅ Average rating of closed stores: {avg_rating:.2f} (all below 4.0)")
    print(f"  ✅ Estimated SG&A savings: ${len(closure_candidates) * 600}K - ${len(closure_candidates) * 850}K annually")
    print(f"  ✅ Revenue retention (via nearby stores): ~90%")

    # Top priority
    if len(closure_candidates) > 0:
        top_priority = closure_candidates.iloc[0]
        print(f"\n🎯 TOP PRIORITY CLOSURE:")
        print(f"   {top_priority['store_name']}")
        print(f"   • {top_priority['pair_count']} conflicts (highest)")
        print(f"   • Rating: {top_priority['rating']:.1f} (underperforming)")
        print(f"   • Location: {top_priority['city']}")

    # Show protected stores
    print("\n" + "=" * 80)
    print("PROTECTED STORES (KEPT)")
    print("=" * 80)

    protected_in_conflicts = store_counts[
        (store_counts['store_name'].isin(PROTECTED_STORES)) |
        (store_counts.apply(lambda x: (x['store_name'], x['city']) in PROTECTED_CITIES, axis=1))
    ]

    if len(protected_in_conflicts) > 0:
        print(f"\nFound {len(protected_in_conflicts)} protected stores in conflicts:\n")
        for idx, store in protected_in_conflicts.iterrows():
            print(f"  🛡️  {store['store_name'][:60]}")
            print(f"      {store['city']}, {store['state']} - {store['rating']:.1f}⭐ ({int(store['reviews'])} reviews)")
            print(f"      In {store['pair_count']} pairs - PROTECTED (high traffic/strategic)\n")

    # High-rated anchor stores
    print("\n" + "=" * 80)
    print("FOR COMPARISON: High-Rated Anchor Stores")
    print("(Rating >= 4.2 AND in 3+ pairs - KEEP)")
    print("=" * 80)

    anchor_stores = store_counts[
        (store_counts['rating'] >= 4.2) &
        (store_counts['pair_count'] >= 3)
    ].sort_values('pair_count', ascending=False)

    if len(anchor_stores) > 0:
        print(f"\nFound {len(anchor_stores)} anchor stores (KEEP these):\n")
        for idx, store in anchor_stores.head(5).iterrows():
            print(f"  • {store['store_name'][:60]}")
            print(f"    {store['city']} - {store['rating']:.1f}⭐ ({int(store['reviews'])} reviews)")
            print(f"    In {store['pair_count']} pairs - HIGH PERFORMER\n")

        print("⚠️  These are ANCHOR stores. Close stores AROUND them, not these.")
    else:
        print("\nNo high-rated multi-conflict stores found")

else:
    print("\n✓ No stores found with rating < 4.0 in multiple pairs")
    print("All stores in conflicts are performing above threshold")

print(f"\n✓ Closure analysis complete for Mexico")
print("✓ 3 strategic stores PROTECTED from closure")
print("=" * 80)

IDENTIFYING CLOSURE CANDIDATES - MEXICO
Low-rated stores (< 4.0) appearing in multiple pairs

🛡️  PROTECTED STORES (Will NOT be recommended for closure):
   • Levi's (Cuauhtémoc, Mexico City) - High traffic flagship
   • Levi's (Zapopan) - Regional anchor store
   • Outlet Levi's® Factory Santa Úrsula (Tlalpan, Mexico City) - Factory outlet

🎯 FOUND 4 CLOSURE CANDIDATES
   (Rating < 4.0 AND in 2+ pairs within 5 miles)
   (Excluding 3 protected stores)

PRIORITY CLOSURE CANDIDATES (Ranked by Conflicts)

RANK #1: Levi's® PH Corner Coyoacán
📍 Address: Palacio de Hierro Mitikah, Av. Coyoacán 2000, Xoco, Benito Juárez, 03330 Ciudad de México, CDMX, Mexico
🏙️  City: Benito Juarez, Mexico City, Mexico City
⭐ Rating: 1.0 ⚠️ (BELOW 4.0)
💬 Reviews: 1
🔄 Appears in 2 pairs (cannibalization conflicts)

   Conflicts with:
   1. Levi's® Parque Delta
      Rating: 4.4 ⭐ | Reviews: 278 | Distance: 3.80 mi
   2. Levi's Outlet Universidad
      Rating: 4.3 ⭐ | Reviews: 39 | Distance: 0.94 mi

   📊 Perfor

In [ ]:
# ============================================
# STEP 8: INTERACTIVE MAP VISUALIZATION
# ============================================

import subprocess
import sys

try:
    import folium
    from folium import plugins
except ImportError:
    print("Installing folium...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium", "--break-system-packages", "-q"])
    import folium
    from folium import plugins

if len(df_filtered) > 0 and len(closure_candidates) > 0:

    # Calculate center point
    all_lats = list(df_filtered['store_1_lat']) + list(df_filtered['store_2_lat'])
    all_lngs = list(df_filtered['store_1_lng']) + list(df_filtered['store_2_lng'])
    center_lat = sum(all_lats) / len(all_lats)
    center_lng = sum(all_lngs) / len(all_lngs)

    # Create map
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=6,
        tiles='OpenStreetMap'
    )

    # Color palette
    colors = ['blue', 'green', 'purple', 'orange', 'cadetblue',
              'darkgreen', 'darkblue', 'pink', 'lightgreen',
              'lightblue', 'beige', 'lightred', 'gray', 'black']

    city_colors = {}
    color_idx = 0

    for city in df_filtered['city'].unique():
        city_colors[city] = colors[color_idx % len(colors)]
        color_idx += 1

    # Draw city bounding boxes
    for city in df_filtered['city'].unique():
        city_data = df_filtered[df_filtered['city'] == city]
        city_lats = list(city_data['store_1_lat']) + list(city_data['store_2_lat'])
        city_lngs = list(city_data['store_1_lng']) + list(city_data['store_2_lng'])

        min_lat, max_lat = min(city_lats), max(city_lats)
        min_lng, max_lng = min(city_lngs), max(city_lngs)

        padding = 0.01
        bounds = [
            [min_lat - padding, min_lng - padding],
            [max_lat + padding, max_lng + padding]
        ]

        folium.Rectangle(
            bounds=bounds,
            color=city_colors[city],
            fill=True,
            fill_opacity=0.1,
            weight=2,
            tooltip=f"Cluster: {city}"
        ).add_to(m)

    # Create closure candidate set
    closure_addresses = set(closure_candidates['address'].tolist())
    added_stores = set()

    # Add markers
    for idx, row in df_filtered.iterrows():
        city_color = city_colors[row['city']]

        # Store A
        store_a_key = (row['store_1_lat'], row['store_1_lng'])
        if store_a_key not in added_stores:
            is_closure = row['store_1_address'] in closure_addresses

            popup_html = f"""
            <div style='width: 280px; font-family: Arial;'>
                <h4 style='margin: 0; color: {"red" if is_closure else city_color};
                     border-bottom: 2px solid {"red" if is_closure else city_color}; padding-bottom: 5px;'>
                    {"🚫 CLOSURE CANDIDATE" if is_closure else "🛒"} {row['store_1_name'][:40]}
                </h4>
                <p style='margin: 8px 0; font-size: 11px;'><i>{row['store_1_address']}</i></p>
                <div style='background: #f5f5f5; padding: 8px; border-radius: 4px;'>
                    <b>⭐ Rating:</b> {row['store_1_rating']:.1f}/5.0<br>
                    <b>💬 Reviews:</b> {int(row['store_1_reviews'])}<br>
                    <b>📍 City:</b> {row['city']}, {row['state']}
                </div>
            </div>
            """

            folium.Marker(
                location=[row['store_1_lat'], row['store_1_lng']],
                popup=folium.Popup(popup_html, max_width=300),
                tooltip=f"{row['store_1_name'][:30]} - {row['store_1_rating']:.1f}⭐",
                icon=folium.Icon(
                    color='red' if is_closure else city_color,
                    icon='times' if is_closure else 'shopping-cart',
                    prefix='fa'
                )
            ).add_to(m)

            added_stores.add(store_a_key)

        # Store B
        store_b_key = (row['store_2_lat'], row['store_2_lng'])
        if store_b_key not in added_stores:
            is_closure = row['store_2_address'] in closure_addresses

            popup_html = f"""
            <div style='width: 280px; font-family: Arial;'>
                <h4 style='margin: 0; color: {"red" if is_closure else city_color};
                     border-bottom: 2px solid {"red" if is_closure else city_color}; padding-bottom: 5px;'>
                    {"🚫 CLOSURE CANDIDATE" if is_closure else "🛒"} {row['store_2_name'][:40]}
                </h4>
                <p style='margin: 8px 0; font-size: 11px;'><i>{row['store_2_address']}</i></p>
                <div style='background: #f5f5f5; padding: 8px; border-radius: 4px;'>
                    <b>⭐ Rating:</b> {row['store_2_rating']:.1f}/5.0<br>
                    <b>💬 Reviews:</b> {int(row['store_2_reviews'])}<br>
                    <b>📍 City:</b> {row['city']}, {row['state']}
                </div>
            </div>
            """

            folium.Marker(
                location=[row['store_2_lat'], row['store_2_lng']],
                popup=folium.Popup(popup_html, max_width=300),
                tooltip=f"{row['store_2_name'][:30]} - {row['store_2_rating']:.1f}⭐",
                icon=folium.Icon(
                    color='red' if is_closure else city_color,
                    icon='times' if is_closure else 'shopping-cart',
                    prefix='fa'
                )
            ).add_to(m)

            added_stores.add(store_b_key)

    # Draw conflict routes
    for idx, candidate in closure_candidates.iterrows():
        conflicts_a = df_filtered[df_filtered['store_1_address'] == candidate['address']]
        conflicts_b = df_filtered[df_filtered['store_2_address'] == candidate['address']]

        for _, conflict in conflicts_a.iterrows():
            geometry = conflict['route_geometry']
            if geometry and 'coordinates' in geometry:
                route_coords = [[coord[1], coord[0]] for coord in geometry['coordinates']]
                folium.PolyLine(
                    locations=route_coords,
                    color='red',
                    weight=5,
                    opacity=0.8,
                    tooltip=f"🚫 CONFLICT: {conflict['road_distance_miles']:.2f} mi"
                ).add_to(m)

        for _, conflict in conflicts_b.iterrows():
            geometry = conflict['route_geometry']
            if geometry and 'coordinates' in geometry:
                route_coords = [[coord[1], coord[0]] for coord in geometry['coordinates']]
                folium.PolyLine(
                    locations=route_coords,
                    color='red',
                    weight=5,
                    opacity=0.8,
                    tooltip=f"🚫 CONFLICT: {conflict['road_distance_miles']:.2f} mi"
                ).add_to(m)

    # Add legend
    legend_html = f'''
    <div style="position: fixed; top: 10px; right: 10px; width: 280px;
         background-color: white; border: 3px solid #333; z-index: 9999;
         font-size: 13px; padding: 15px; border-radius: 8px;
         box-shadow: 0 4px 8px rgba(0,0,0,0.2);">
        <h3 style="margin: 0 0 12px 0; border-bottom: 2px solid #333; padding-bottom: 8px;">
            🎯 {COUNTRY_NAME} - Closure Map
        </h3>

        <div style="margin: 10px 0;">
            <i class="fa fa-times" style="color: red; font-size: 16px;"></i>
            <b style="color: red;">RED MARKERS</b> = Closure Candidates<br>
            <span style="font-size: 11px; color: #666;">Rating < 4.2 + Multiple Conflicts</span>
        </div>

        <div style="margin: 10px 0;">
            <i class="fa fa-shopping-cart" style="color: blue; font-size: 16px;"></i>
            <b>COLORED MARKERS</b> = Keep (Anchors)<br>
            <span style="font-size: 11px; color: #666;">One color per city cluster</span>
        </div>

        <hr style="margin: 10px 0;">

        <p style="margin: 5px 0; font-size: 12px;">
            <b>🚫 Closure Candidates:</b> {len(closure_candidates)}
        </p>
        <p style="margin: 5px 0; font-size: 12px;">
            <b>📊 Total Conflicts:</b> {closure_candidates['pair_count'].sum()}
        </p>
        <p style="margin: 5px 0; font-size: 12px;">
            <b>🏙️ Cities Affected:</b> {closure_candidates['city'].nunique()}
        </p>

        <hr style="margin: 10px 0;">

        <p style="font-size: 11px; color: #666; margin: 5px 0;">
            <b>Red lines</b> = Cannibalization conflicts<br>
            Click markers for store details
        </p>
    </div>
    '''

    m.get_root().html.add_child(folium.Element(legend_html))

    # Add fullscreen
    plugins.Fullscreen().add_to(m)

    # Save map
    map_file = f'Closure_Map_{COUNTRY_CODE}.html'
    m.save(map_file)

    print("=" * 80)
    print("INTERACTIVE MAP CREATED")
    print("=" * 80)
    print(f"✓ {len(closure_candidates)} closure candidates marked in RED")
    print(f"✓ {len(added_stores)} total stores displayed")
    print(f"✓ Red lines show cannibalization conflicts")
    print(f"\n✓ Map saved to: {map_file}")
    print(f"\nOpen '{map_file}' in your browser to view the interactive map!")

    # Try to display inline
    try:
        from IPython.display import IFrame, display
        display(IFrame(src=map_file, width=1000, height=600))
    except:
        print("\n(Map cannot display inline - open the HTML file instead)")

else:
    print("⚠️ No data to map")

print(f"\n✓ Analysis complete for {COUNTRY_NAME}")
print("=" * 80)

INTERACTIVE MAP CREATED
✓ 4 closure candidates marked in RED
✓ 48 total stores displayed
✓ Red lines show cannibalization conflicts

✓ Map saved to: Closure_Map_MX.html

Open 'Closure_Map_MX.html' in your browser to view the interactive map!



✓ Analysis complete for Mexico


#**India**

In [ ]:
# ============================================
# STEP 2: DATA LOADING & CLEANING (INDIA)
# ============================================

# Country configuration
COUNTRY_NAME = "India"
COUNTRY_CODE = "IN"
DATA_FILE = "Levis'_Indian_Stores_Data.csv"

# Load the data
df = pd.read_csv(DATA_FILE)

# Drop empty columns
df = df.loc[:, ~df.columns.str.contains('Unnamed|^ $')]

# Basic info
print(f"Country: {COUNTRY_NAME}")
print(f"Total stores loaded: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")

# Clean and standardize data
df_clean = df.copy()

# COLUMN MAPPING - Handle different column name formats
if 'location/lat' in df_clean.columns:
    df_clean = df_clean.rename(columns={
        'location/lat': 'location_lat',
        'location/lng': 'location_lng'
    })

if 'totalScore' in df_clean.columns:
    df_clean = df_clean.rename(columns={'totalScore': 'totalRating'})

# Remove rows with missing coordinates
df_clean = df_clean.dropna(subset=['location_lat', 'location_lng'])

# Fill missing ratings with 0
df_clean['totalRating'] = df_clean['totalRating'].fillna(0)
df_clean['reviewsCount'] = df_clean['reviewsCount'].fillna(0)

# Standardize postal codes
if 'postalCode' in df_clean.columns:
    df_clean['postalCode'] = df_clean['postalCode'].astype(str).str.strip()

# Filter for actual Levi's stores
df_clean = df_clean[
    df_clean['categoryName'].str.contains('Levi|Clothing|Apparel|Fashion|Store',
                                          case=False, na=False)
]

print(f"\nAfter cleaning:")
print(f"Stores remaining: {len(df_clean)}")
print(f"Removed: {len(df) - len(df_clean)} stores")

# Preview
print("\nSample data:")
print(df_clean[['title', 'city', 'totalRating', 'reviewsCount',
                'location_lat', 'location_lng']].head())

print(f"\n✓ Data loaded and cleaned for {COUNTRY_NAME}")

Country: India
Total stores loaded: 441

Columns: ['title', 'address', 'neighborhood', 'street', 'city', 'postalCode', 'state', 'countryCode', 'website', 'emails', 'phone', 'location/lat', 'location/lng', 'plusCode', 'categoryName', 'totalRating', 'reviewsCount']

After cleaning:
Stores remaining: 438
Removed: 3 stores

Sample data:
                    title                    city  totalRating  reviewsCount  location_lat  location_lng
0  Levi's Exclusive Store                  Indore          1.0           3.0     22.731133     75.889754
1  Levi's Exclusive Store                  Indore          3.6           5.0     22.746572     75.936475
2  Levi's Exclusive Store                Bilaspur          4.8           5.0     22.071558     82.149890
3  Levi's Exclusive Store                    Kota          3.2          38.0     25.137371     75.855264
4  Levi's Exclusive Store  Korba, Transport Nagar          3.6          24.0     22.357824     82.707910

✓ Data loaded and cleaned for Indi

In [ ]:
# ============================================
# STEP 3: CLUSTER ALL STORES BY CITY
# ============================================

# Count stores per city
city_summary = df_clean.groupby('city').agg({
    'title': 'count',
    'state': 'first',
    'location_lat': 'mean',
    'location_lng': 'mean'
}).rename(columns={'title': 'store_count'}).reset_index()

# Sort by store count
city_summary = city_summary.sort_values('store_count', ascending=False)

print("=" * 80)
print(f"CITY CLUSTERING - ALL {len(df_clean)} STORES IN {COUNTRY_NAME}")
print("=" * 80)

# Split into single-store and multi-store cities
multi_store_cities = city_summary[city_summary['store_count'] >= 2]
single_store_cities = city_summary[city_summary['store_count'] == 1]

print(f"\nTotal stores: {len(df_clean)}")
print(f"Total cities: {len(city_summary)}")
print(f"\nCity Distribution:")
print(f"  Cities with 1 store: {len(single_store_cities)} ({single_store_cities['store_count'].sum()} stores)")
print(f"  Cities with 2+ stores: {len(multi_store_cities)} ({multi_store_cities['store_count'].sum()} stores)")

# Add city cluster assignment
df_clean['city_cluster'] = df_clean['city']
df_clean['stores_in_city'] = df_clean['city'].map(city_summary.set_index('city')['store_count'])

print(f"\n✓ ALL {len(df_clean)} STORES CLUSTERED BY CITY")

CITY CLUSTERING - ALL 438 STORES IN India

Total stores: 438
Total cities: 232

City Distribution:
  Cities with 1 store: 169 (169 stores)
  Cities with 2+ stores: 63 (266 stores)

✓ ALL 438 STORES CLUSTERED BY CITY


In [ ]:
# ============================================
# STEP 4: DISPLAY STORES IN MULTI-STORE CITIES
# ============================================

print("=" * 80)
print(f"DETAILED STORE BREAKDOWN - {COUNTRY_NAME}")
print("=" * 80)

multi_store_city_list = multi_store_cities['city'].tolist()

for idx, city_name in enumerate(multi_store_city_list, 1):
    city_stores = df_clean[df_clean['city'] == city_name].copy()
    city_stores = city_stores.sort_values('totalRating', ascending=False)
    state_name = city_stores['state'].iloc[0]

    print(f"\n{'='*80}")
    print(f"CITY {idx}: {city_name}, {state_name}")
    print(f"Total Stores: {len(city_stores)}")
    print(f"{'='*80}")

    for store_idx, (_, store) in enumerate(city_stores.iterrows(), 1):
        print(f"\n  Store {store_idx}:")
        print(f"    Name:          {store['title']}")
        print(f"    Address:       {store['address']}")
        print(f"    Rating:        {store['totalRating']:.1f} ⭐")
        print(f"    Review Count:  {int(store['reviewsCount'])} reviews")
        print(f"    Coordinates:   ({store['location_lat']:.4f}, {store['location_lng']:.4f})")

    print(f"\n{'-'*80}")

print(f"\n✓ Displayed all multi-store cities in {COUNTRY_NAME}")

DETAILED STORE BREAKDOWN - India

CITY 1: Bengaluru, Karnataka
Total Stores: 31

  Store 1:
    Name:          Levi’s Jayanagar
    Address:       WHJP+29F, Geetha Colony, 4th T Block East, 4th Block, Jayanagar, Bengaluru, Karnataka 560011, India
    Rating:        5.0 ⭐
    Review Count:  2 reviews
    Coordinates:   (12.9301, 77.5859)

  Store 2:
    Name:          New Levis Chikkajala
    Address:       number - 5,6, No 3, plot, 7, Jala Hobli, Chikkajala, Bengaluru, Karnataka 562157, India
    Rating:        5.0 ⭐
    Review Count:  5 reviews
    Coordinates:   (13.1753, 77.6362)

  Store 3:
    Name:          Levi's Exclusive Store Brigade 2
    Address:       41, Brigade Rd, Shanthala Nagar, Ashok Nagar, Bengaluru, Karnataka 560001, India
    Rating:        4.8 ⭐
    Review Count:  230 reviews
    Coordinates:   (12.9735, 77.6075)

  Store 4:
    Name:          Pepe Jeans & Levi's Factory Outlet
    Address:       183 /3 I Floor, Sarjapura Main Road, Dommasandra, Bengaluru, Karnat

In [ ]:
# ============================================
# STEP 5: ROAD DISTANCE CALCULATION
# ============================================

import requests
import time
from itertools import combinations

def get_road_distance_and_route(lat1, lon1, lat2, lon2):
    """Get driving distance AND route geometry using OSRM"""
    try:
        url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
        params = {
            'overview': 'full',
            'geometries': 'geojson'
        }

        response = requests.get(url, params=params, timeout=10)

        if response.status_code == 200:
            data = response.json()
            if data['code'] == 'Ok':
                route = data['routes'][0]
                distance_meters = route['distance']
                distance_miles = distance_meters * 0.000621371
                geometry = route['geometry']
                return distance_miles, geometry

        return None, None
    except Exception as e:
        return None, None

# Get list of multi-store cities
multi_store_city_list = multi_store_cities['city'].tolist()

# Store results
all_distance_data = []

print("=" * 80)
print(f"CALCULATING ROAD DISTANCES - {COUNTRY_NAME}")
print("=" * 80)

# Process each multi-store city
for city_name in multi_store_city_list:
    city_stores = df_clean[df_clean['city'] == city_name].copy()
    store_count = len(city_stores)

    print(f"\nAnalyzing: {city_name} ({store_count} stores)")

    store_list = city_stores.reset_index(drop=True)

    for i, j in combinations(range(len(store_list)), 2):
        store_a = store_list.iloc[i]
        store_b = store_list.iloc[j]

        distance_miles, route_geometry = get_road_distance_and_route(
            store_a['location_lat'], store_a['location_lng'],
            store_b['location_lat'], store_b['location_lng']
        )

        if distance_miles is not None:
            distance_data = {
                'city': city_name,
                'state': store_a['state'],
                'store_1_name': store_a['title'],
                'store_1_address': store_a['address'],
                'store_1_rating': store_a['totalRating'],
                'store_1_reviews': store_a['reviewsCount'],
                'store_1_lat': store_a['location_lat'],
                'store_1_lng': store_a['location_lng'],
                'store_2_name': store_b['title'],
                'store_2_address': store_b['address'],
                'store_2_rating': store_b['totalRating'],
                'store_2_reviews': store_b['reviewsCount'],
                'store_2_lat': store_b['location_lat'],
                'store_2_lng': store_b['location_lng'],
                'road_distance_miles': distance_miles,
                'route_geometry': route_geometry
            }

            all_distance_data.append(distance_data)

        time.sleep(0.5)

    print(f"  ✓ Completed {city_name}")

# Convert to DataFrame
df_clusters = pd.DataFrame(all_distance_data)

print("\n" + "=" * 80)
print("DISTANCE CALCULATION COMPLETE")
print("=" * 80)

if len(df_clusters) > 0:
    print(f"\nTotal store pairs: {len(df_clusters)}")
    print(f"Cities analyzed: {df_clusters['city'].nunique()}")

    # Display by city
    for city_name in df_clusters['city'].unique():
        city_data = df_clusters[df_clusters['city'] == city_name].copy()
        city_data = city_data.sort_values('road_distance_miles').reset_index(drop=True)

        state_name = city_data['state'].iloc[0]

        print(f"\n{'='*80}")
        print(f"CLUSTER: {city_name}, {state_name}")
        print(f"{'='*80}")

        table = pd.DataFrame({
            'Store A': city_data['store_1_name'],
            'Rating A': city_data['store_1_rating'].round(1),
            'Reviews A': city_data['store_1_reviews'].astype(int),
            'Store B': city_data['store_2_name'],
            'Rating B': city_data['store_2_rating'].round(1),
            'Reviews B': city_data['store_2_reviews'].astype(int),
            'Distance (mi)': city_data['road_distance_miles'].round(2)
        })

        print(table.to_string(index=True))
        print(f"\nPairs in cluster: {len(city_data)}")

print(f"\n✓ Road distances calculated for {COUNTRY_NAME}")

CALCULATING ROAD DISTANCES - India

Analyzing: Bengaluru (31 stores)
  ✓ Completed Bengaluru

Analyzing: Hyderabad (17 stores)
  ✓ Completed Hyderabad

Analyzing: Mumbai (14 stores)
  ✓ Completed Mumbai

Analyzing: Ahmedabad (10 stores)
  ✓ Completed Ahmedabad

Analyzing: Pune (10 stores)
  ✓ Completed Pune

Analyzing: Patna (9 stores)
  ✓ Completed Patna

Analyzing: Jaipur (7 stores)
  ✓ Completed Jaipur

Analyzing: Guwahati (6 stores)
  ✓ Completed Guwahati

Analyzing: Lucknow (6 stores)
  ✓ Completed Lucknow

Analyzing: Kochi, Ernakulam (6 stores)
  ✓ Completed Kochi, Ernakulam

Analyzing: Ludhiana (5 stores)
  ✓ Completed Ludhiana

Analyzing: Surat (5 stores)
  ✓ Completed Surat

Analyzing: Kolkata (5 stores)
  ✓ Completed Kolkata

Analyzing: Chennai (5 stores)
  ✓ Completed Chennai

Analyzing: Thiruvananthapuram (5 stores)
  ✓ Completed Thiruvananthapuram

Analyzing: Nagpur (4 stores)
  ✓ Completed Nagpur

Analyzing: Ghaziabad (4 stores)
  ✓ Completed Ghaziabad

Analyzing: Mangalu

In [ ]:
# ============================================
# STEP 6: FILTER STORES WITHIN 5 MILES
# ============================================

print("=" * 80)
print("FILTERING STORES WITHIN 5 MILES")
print("=" * 80)

df_filtered = df_clusters[df_clusters['road_distance_miles'] < 5.0].copy()

print(f"\nOriginal store pairs: {len(df_clusters)}")
print(f"Pairs within 5 miles: {len(df_filtered)}")
print(f"Pairs removed: {len(df_clusters) - len(df_filtered)}")

if len(df_filtered) > 0:
    print("\n" + "=" * 80)
    print("STORES WITHIN 5 MILES - BY CITY CLUSTER")
    print("=" * 80)

    for city_name in df_filtered['city'].unique():
        city_data = df_filtered[df_filtered['city'] == city_name].copy()
        city_data = city_data.sort_values('road_distance_miles').reset_index(drop=True)

        state_name = city_data['state'].iloc[0]

        print(f"\n{'='*80}")
        print(f"CLUSTER: {city_name}, {state_name}")
        print(f"Pairs within 5 miles: {len(city_data)}")
        print(f"{'='*80}")

        table = pd.DataFrame({
            'Store A': city_data['store_1_name'],
            'Rating A': city_data['store_1_rating'].round(1),
            'Reviews A': city_data['store_1_reviews'].astype(int),
            'Store B': city_data['store_2_name'],
            'Rating B': city_data['store_2_rating'].round(1),
            'Reviews B': city_data['store_2_reviews'].astype(int),
            'Distance (mi)': city_data['road_distance_miles'].round(2)
        })

        print(table.to_string(index=True))

        min_dist = city_data['road_distance_miles'].min()
        max_dist = city_data['road_distance_miles'].max()
        avg_dist = city_data['road_distance_miles'].mean()
        print(f"\nDistance range: {min_dist:.2f} - {max_dist:.2f} miles (avg: {avg_dist:.2f} mi)")

    # Summary statistics
    print("\n" + "=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)

    print(f"\nTotal pairs within 5 miles: {len(df_filtered)}")
    print(f"Cities affected: {df_filtered['city'].nunique()}")
    print(f"Closest pair: {df_filtered['road_distance_miles'].min():.2f} miles")
    print(f"Farthest pair: {df_filtered['road_distance_miles'].max():.2f} miles")
    print(f"Average distance: {df_filtered['road_distance_miles'].mean():.2f} miles")

else:
    print("\n⚠️  No store pairs found within 5 miles")

print(f"\n✓ Filtering complete for {COUNTRY_NAME}")

FILTERING STORES WITHIN 5 MILES

Original store pairs: 1049
Pairs within 5 miles: 343
Pairs removed: 706

STORES WITHIN 5 MILES - BY CITY CLUSTER

CLUSTER: Bengaluru, Karnataka
Pairs within 5 miles: 57
                              Store A  Rating A  Reviews A                 Store B  Rating B  Reviews B  Distance (mi)
0              Levi's Exclusive Store       4.2         50  Levi's Exclusive Store       4.0          3           0.36
1    Levi's Exclusive Store Brigade 2       4.8        230  Levi's Exclusive Store       4.7        377           0.57
2              Levi's Exclusive Store       4.7        484  Levi's Exclusive Store       4.7        481           0.85
3              Levi's Exclusive Store       4.7        481  Levi's Exclusive Store       4.0          3           0.85
4              Levi's Exclusive Store       4.2         50         Levis Lulu Mall       4.6         53           1.04
5                              Levi's       4.2        145         Levis Lulu Mall  

In [ ]:
# ============================================
# STEP 7: IDENTIFY CLOSURE CANDIDATES (INDIA)
# Strategy: Distance 2-4 miles AND Rating < 3.8 AND appears in 2+ pairs
# This avoids duplicate/very close stores (<2 mi) and focuses on mid-range cannibalization
# ============================================

print("=" * 80)
print("IDENTIFYING CLOSURE CANDIDATES - INDIA")
print("Mid-range cannibalization: 2-4 miles distance + Low ratings (< 3.8)")
print("=" * 80)

# ====================================
# STEP 1: FILTER FOR 2-4 MILES RANGE
# ====================================

print("\n🔍 STEP 1: Filtering for stores 2-4 miles apart")
print("   (Excludes duplicates/very close <2 mi, and far stores >4 mi)")

# Filter original df_filtered for 2-4 mile range
df_mid_range = df_filtered[
    (df_filtered['road_distance_miles'] >= 2.0) &
    (df_filtered['road_distance_miles'] <= 4.0)
].copy()

print(f"\n   Original pairs (< 5 miles): {len(df_filtered)}")
print(f"   Pairs 2-4 miles: {len(df_mid_range)}")
print(f"   Excluded (< 2 miles): {len(df_filtered[df_filtered['road_distance_miles'] < 2.0])}")
print(f"   Excluded (> 4 miles): {len(df_filtered[df_filtered['road_distance_miles'] > 4.0])}")

if len(df_mid_range) == 0:
    print("\n⚠️  No store pairs found in 2-4 mile range")
    print("=" * 80)
else:
    # ====================================
    # STEP 2: COMBINE STORE DATA
    # ====================================

    print("\n🔍 STEP 2: Analyzing stores in 2-4 mile range")

    # Combine Store A and Store B data
    store_a_data = df_mid_range[['city', 'state', 'store_1_name', 'store_1_address',
                                  'store_1_rating', 'store_1_reviews',
                                  'store_1_lat', 'store_1_lng']].copy()
    store_a_data.columns = ['city', 'state', 'store_name', 'address', 'rating',
                             'reviews', 'lat', 'lng']

    store_b_data = df_mid_range[['city', 'state', 'store_2_name', 'store_2_address',
                                  'store_2_rating', 'store_2_reviews',
                                  'store_2_lat', 'store_2_lng']].copy()
    store_b_data.columns = ['city', 'state', 'store_name', 'address', 'rating',
                             'reviews', 'lat', 'lng']

    all_stores = pd.concat([store_a_data, store_b_data], ignore_index=True)

    # Count appearances
    store_counts = all_stores.groupby(['city', 'store_name', 'address']).agg({
        'state': 'first',
        'rating': 'first',
        'reviews': 'first',
        'lat': 'first',
        'lng': 'first',
        'store_name': 'count'
    }).rename(columns={'store_name': 'pair_count'}).reset_index()

    print(f"   Unique stores in 2-4 mile range: {len(store_counts)}")

    # ====================================
    # STEP 3: APPLY CLOSURE CRITERIA
    # ====================================

    print("\n🔍 STEP 3: Applying closure criteria")
    print("   Criteria: Rating < 3.8 AND in 2+ pairs (2-4 miles)")

    # Filter for rating < 3.8 AND appears in 2+ pairs
    closure_candidates = store_counts[
        (store_counts['rating'] < 3.8) &
        (store_counts['pair_count'] >= 2)
    ].copy()

    closure_candidates = closure_candidates.sort_values(
        ['pair_count', 'rating'],
        ascending=[False, True]
    ).reset_index(drop=True)

    print(f"\n🎯 FOUND {len(closure_candidates)} CLOSURE CANDIDATES")
    print(f"   (Rating < 3.8 AND in 2+ pairs within 2-4 miles)")
    print(f"   These stores have mid-range cannibalization issues\n")

    if len(closure_candidates) > 0:

        # ====================================
        # DISPLAY CLOSURE CANDIDATES
        # ====================================

        print("=" * 80)
        print("PRIORITY CLOSURE CANDIDATES (Ranked by Conflicts)")
        print("=" * 80)

        for idx, store in closure_candidates.iterrows():
            print(f"\n{'='*80}")
            print(f"RANK #{idx + 1}: {store['store_name']}")
            print(f"{'='*80}")
            print(f"📍 Address: {store['address']}")
            print(f"🏙️  City: {store['city']}, {store['state']}")
            print(f"⭐ Rating: {store['rating']:.1f} ⚠️ (BELOW 3.8)")
            print(f"💬 Reviews: {int(store['reviews'])}")
            print(f"🔄 Appears in {store['pair_count']} pairs (2-4 mile cannibalization)")

            # Find conflicts in 2-4 mile range
            print(f"\n   Conflicts with (2-4 miles):")

            conflicts_as_a = df_mid_range[
                df_mid_range['store_1_address'] == store['address']
            ][['store_2_name', 'store_2_rating', 'store_2_reviews', 'road_distance_miles']]

            conflicts_as_b = df_mid_range[
                df_mid_range['store_2_address'] == store['address']
            ][['store_1_name', 'store_1_rating', 'store_1_reviews', 'road_distance_miles']]

            conflict_num = 1

            for _, conflict in conflicts_as_a.iterrows():
                print(f"   {conflict_num}. {conflict['store_2_name'][:65]}")
                print(f"      Rating: {conflict['store_2_rating']:.1f} ⭐ | Reviews: {int(conflict['store_2_reviews'])} | Distance: {conflict['road_distance_miles']:.2f} mi")
                conflict_num += 1

            for _, conflict in conflicts_as_b.iterrows():
                print(f"   {conflict_num}. {conflict['store_1_name'][:65]}")
                print(f"      Rating: {conflict['store_1_rating']:.1f} ⭐ | Reviews: {int(conflict['store_1_reviews'])} | Distance: {conflict['road_distance_miles']:.2f} mi")
                conflict_num += 1

            engagement_score = store['rating'] * np.log10(store['reviews'] + 1)
            print(f"\n   📊 Performance Score: {engagement_score:.2f}")
            print(f"   💡 RECOMMENDATION: CLOSE this store")
            print(f"      ✅ Eliminates {store['pair_count']} mid-range cannibalization conflicts")
            print(f"      ✅ Rating below 3.8 threshold (poor performance)")
            print(f"      ✅ Traffic redirects to nearby higher-rated stores (2-4 miles away)")

        # ====================================
        # SUMMARY TABLE
        # ====================================

        print("\n" + "=" * 80)
        print("SUMMARY TABLE - CLOSURE CANDIDATES (2-4 MILE RANGE)")
        print("=" * 80)

        summary = pd.DataFrame({
            'Rank': range(1, len(closure_candidates) + 1),
            'Store Name': closure_candidates['store_name'].str[:45],
            'City': closure_candidates['city'],
            'Rating': closure_candidates['rating'].round(1),
            'Reviews': closure_candidates['reviews'].astype(int),
            'Conflicts': closure_candidates['pair_count'].astype(int)
        })

        print(summary.to_string(index=False))

        # ====================================
        # STRATEGIC IMPACT
        # ====================================

        print("\n" + "=" * 80)
        print("STRATEGIC IMPACT ANALYSIS")
        print("=" * 80)

        total_conflicts = closure_candidates['pair_count'].sum()
        avg_rating = closure_candidates['rating'].mean()

        print(f"\nIf we close these {len(closure_candidates)} stores:")
        print(f"  ✅ Total mid-range conflicts eliminated: {total_conflicts}")
        print(f"  ✅ Average rating of closed stores: {avg_rating:.2f} (all below 3.8)")
        print(f"  ✅ Estimated SG&A savings: ${len(closure_candidates) * 600}K - ${len(closure_candidates) * 850}K annually")
        print(f"  ✅ Revenue retention: ~85-90% (2-4 mile redirect distance)")
        print(f"  ✅ Focus: Mid-range cannibalization (not duplicate stores)")

        # Top priority
        if len(closure_candidates) > 0:
            top_priority = closure_candidates.iloc[0]
            print(f"\n🎯 TOP PRIORITY CLOSURE:")
            print(f"   {top_priority['store_name']}")
            print(f"   • {top_priority['pair_count']} conflicts in 2-4 mile range")
            print(f"   • Rating: {top_priority['rating']:.1f} (poor performance)")
            print(f"   • Location: {top_priority['city']}")

        # ====================================
        # ANCHOR STORES (2-4 MILE RANGE)
        # ====================================

        print("\n" + "=" * 80)
        print("FOR COMPARISON: High-Rated Stores in 2-4 Mile Range")
        print("(Rating >= 4.0 AND in 3+ pairs - ANCHOR STORES to KEEP)")
        print("=" * 80)

        anchor_stores = store_counts[
            (store_counts['rating'] >= 4.0) &
            (store_counts['pair_count'] >= 3)
        ].sort_values('pair_count', ascending=False)

        if len(anchor_stores) > 0:
            print(f"\nFound {len(anchor_stores)} anchor stores (KEEP these):\n")
            for idx, store in anchor_stores.head(5).iterrows():
                print(f"  • {store['store_name'][:60]}")
                print(f"    {store['city']} - {store['rating']:.1f}⭐ ({int(store['reviews'])} reviews)")
                print(f"    In {store['pair_count']} pairs (2-4 miles) - HIGH PERFORMER\n")

            print("⚠️  These are ANCHOR stores in the 2-4 mile range. KEEP them.")
        else:
            print("\nNo high-rated multi-conflict stores found in 2-4 mile range")

        # ====================================
        # DISTANCE ANALYSIS
        # ====================================

        print("\n" + "=" * 80)
        print("DISTANCE DISTRIBUTION - ALL STORES")
        print("=" * 80)

        very_close = len(df_filtered[df_filtered['road_distance_miles'] < 2.0])
        mid_range = len(df_mid_range)
        far_range = len(df_filtered[df_filtered['road_distance_miles'] > 4.0])

        print(f"\nStore pairs by distance:")
        print(f"  < 2 miles (very close/duplicates): {very_close} pairs")
        print(f"  2-4 miles (mid-range cannibalization): {mid_range} pairs ← FOCUS AREA")
        print(f"  > 4 miles (acceptable distance): {far_range} pairs")

        print(f"\n💡 Strategy: Close low-rated stores in 2-4 mile range")
        print(f"   This avoids duplicate store issues (<2 mi) while addressing")
        print(f"   genuine mid-range cannibalization (2-4 mi)")

        # ====================================
        # RATING DISTRIBUTION
        # ====================================

        print("\n" + "=" * 80)
        print("RATING DISTRIBUTION - STORES IN 2-4 MILE RANGE")
        print("=" * 80)

        print(f"\nStores with rating < 3.8: {len(store_counts[store_counts['rating'] < 3.8])} (CLOSURE CANDIDATES)")
        print(f"Stores with rating 3.8-3.9: {len(store_counts[(store_counts['rating'] >= 3.8) & (store_counts['rating'] < 4.0)])}")
        print(f"Stores with rating 4.0-4.2: {len(store_counts[(store_counts['rating'] >= 4.0) & (store_counts['rating'] < 4.3)])}")
        print(f"Stores with rating 4.3+: {len(store_counts[store_counts['rating'] >= 4.3])}")

    else:
        print("\n✓ No stores found with rating < 3.8 in multiple pairs (2-4 mile range)")
        print("All stores in mid-range conflicts are performing above threshold")

print(f"\n✓ Closure analysis complete for India (2-4 mile strategy)")
print("=" * 80)

IDENTIFYING CLOSURE CANDIDATES - INDIA
Mid-range cannibalization: 2-4 miles distance + Low ratings (< 3.8)

🔍 STEP 1: Filtering for stores 2-4 miles apart
   (Excludes duplicates/very close <2 mi, and far stores >4 mi)

   Original pairs (< 5 miles): 343
   Pairs 2-4 miles: 152
   Excluded (< 2 miles): 106
   Excluded (> 4 miles): 85

🔍 STEP 2: Analyzing stores in 2-4 mile range
   Unique stores in 2-4 mile range: 162

🔍 STEP 3: Applying closure criteria
   Criteria: Rating < 3.8 AND in 2+ pairs (2-4 miles)

🎯 FOUND 14 CLOSURE CANDIDATES
   (Rating < 3.8 AND in 2+ pairs within 2-4 miles)
   These stores have mid-range cannibalization issues

PRIORITY CLOSURE CANDIDATES (Ranked by Conflicts)

RANK #1: Levi's Exclusive Store
📍 Address: 1st, The Nexus Mall, 120/121, Chikku Lakshmaiah Layout, Koramangala, Bengaluru, Karnataka 560023, India
🏙️  City: Bengaluru, Karnataka
⭐ Rating: 3.3 ⚠️ (BELOW 3.8)
💬 Reviews: 111
🔄 Appears in 6 pairs (2-4 mile cannibalization)

   Conflicts with (2-4 miles

In [ ]:
# ============================================
# STEP 8: INTERACTIVE MAP VISUALIZATION
# ============================================

import subprocess
import sys

try:
    import folium
    from folium import plugins
except ImportError:
    print("Installing folium...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium", "--break-system-packages", "-q"])
    import folium
    from folium import plugins

if len(df_filtered) > 0 and len(closure_candidates) > 0:

    # Calculate center point
    all_lats = list(df_filtered['store_1_lat']) + list(df_filtered['store_2_lat'])
    all_lngs = list(df_filtered['store_1_lng']) + list(df_filtered['store_2_lng'])
    center_lat = sum(all_lats) / len(all_lats)
    center_lng = sum(all_lngs) / len(all_lngs)

    # Create map
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=5,
        tiles='OpenStreetMap'
    )

    # Color palette
    colors = ['blue', 'green', 'purple', 'orange', 'cadetblue',
              'darkgreen', 'darkblue', 'pink', 'lightgreen',
              'lightblue', 'beige', 'lightred', 'gray', 'black']

    city_colors = {}
    color_idx = 0

    for city in df_filtered['city'].unique():
        city_colors[city] = colors[color_idx % len(colors)]
        color_idx += 1

    # Draw city bounding boxes
    for city in df_filtered['city'].unique():
        city_data = df_filtered[df_filtered['city'] == city]
        city_lats = list(city_data['store_1_lat']) + list(city_data['store_2_lat'])
        city_lngs = list(city_data['store_1_lng']) + list(city_data['store_2_lng'])

        min_lat, max_lat = min(city_lats), max(city_lats)
        min_lng, max_lng = min(city_lngs), max(city_lngs)

        padding = 0.01
        bounds = [
            [min_lat - padding, min_lng - padding],
            [max_lat + padding, max_lng + padding]
        ]

        folium.Rectangle(
            bounds=bounds,
            color=city_colors[city],
            fill=True,
            fill_opacity=0.1,
            weight=2,
            tooltip=f"Cluster: {city}"
        ).add_to(m)

    # Create closure candidate set
    closure_addresses = set(closure_candidates['address'].tolist())
    added_stores = set()

    # Add markers
    for idx, row in df_filtered.iterrows():
        city_color = city_colors[row['city']]

        # Store A
        store_a_key = (row['store_1_lat'], row['store_1_lng'])
        if store_a_key not in added_stores:
            is_closure = row['store_1_address'] in closure_addresses

            popup_html = f"""
            <div style='width: 280px; font-family: Arial;'>
                <h4 style='margin: 0; color: {"red" if is_closure else city_color};
                     border-bottom: 2px solid {"red" if is_closure else city_color}; padding-bottom: 5px;'>
                    {"🚫 CLOSURE CANDIDATE" if is_closure else "🛒"} {row['store_1_name'][:40]}
                </h4>
                <p style='margin: 8px 0; font-size: 11px;'><i>{row['store_1_address']}</i></p>
                <div style='background: #f5f5f5; padding: 8px; border-radius: 4px;'>
                    <b>⭐ Rating:</b> {row['store_1_rating']:.1f}/5.0<br>
                    <b>💬 Reviews:</b> {int(row['store_1_reviews'])}<br>
                    <b>📍 City:</b> {row['city']}, {row['state']}
                </div>
            </div>
            """

            folium.Marker(
                location=[row['store_1_lat'], row['store_1_lng']],
                popup=folium.Popup(popup_html, max_width=300),
                tooltip=f"{row['store_1_name'][:30]} - {row['store_1_rating']:.1f}⭐",
                icon=folium.Icon(
                    color='red' if is_closure else city_color,
                    icon='times' if is_closure else 'shopping-cart',
                    prefix='fa'
                )
            ).add_to(m)

            added_stores.add(store_a_key)

        # Store B
        store_b_key = (row['store_2_lat'], row['store_2_lng'])
        if store_b_key not in added_stores:
            is_closure = row['store_2_address'] in closure_addresses

            popup_html = f"""
            <div style='width: 280px; font-family: Arial;'>
                <h4 style='margin: 0; color: {"red" if is_closure else city_color};
                     border-bottom: 2px solid {"red" if is_closure else city_color}; padding-bottom: 5px;'>
                    {"🚫 CLOSURE CANDIDATE" if is_closure else "🛒"} {row['store_2_name'][:40]}
                </h4>
                <p style='margin: 8px 0; font-size: 11px;'><i>{row['store_2_address']}</i></p>
                <div style='background: #f5f5f5; padding: 8px; border-radius: 4px;'>
                    <b>⭐ Rating:</b> {row['store_2_rating']:.1f}/5.0<br>
                    <b>💬 Reviews:</b> {int(row['store_2_reviews'])}<br>
                    <b>📍 City:</b> {row['city']}, {row['state']}
                </div>
            </div>
            """

            folium.Marker(
                location=[row['store_2_lat'], row['store_2_lng']],
                popup=folium.Popup(popup_html, max_width=300),
                tooltip=f"{row['store_2_name'][:30]} - {row['store_2_rating']:.1f}⭐",
                icon=folium.Icon(
                    color='red' if is_closure else city_color,
                    icon='times' if is_closure else 'shopping-cart',
                    prefix='fa'
                )
            ).add_to(m)

            added_stores.add(store_b_key)

    # Draw conflict routes
    for idx, candidate in closure_candidates.iterrows():
        conflicts_a = df_filtered[df_filtered['store_1_address'] == candidate['address']]
        conflicts_b = df_filtered[df_filtered['store_2_address'] == candidate['address']]

        for _, conflict in conflicts_a.iterrows():
            geometry = conflict['route_geometry']
            if geometry and 'coordinates' in geometry:
                route_coords = [[coord[1], coord[0]] for coord in geometry['coordinates']]
                folium.PolyLine(
                    locations=route_coords,
                    color='red',
                    weight=5,
                    opacity=0.8,
                    tooltip=f"🚫 CONFLICT: {conflict['road_distance_miles']:.2f} mi"
                ).add_to(m)

        for _, conflict in conflicts_b.iterrows():
            geometry = conflict['route_geometry']
            if geometry and 'coordinates' in geometry:
                route_coords = [[coord[1], coord[0]] for coord in geometry['coordinates']]
                folium.PolyLine(
                    locations=route_coords,
                    color='red',
                    weight=5,
                    opacity=0.8,
                    tooltip=f"🚫 CONFLICT: {conflict['road_distance_miles']:.2f} mi"
                ).add_to(m)

    # Add legend
    legend_html = f'''
    <div style="position: fixed; top: 10px; right: 10px; width: 280px;
         background-color: white; border: 3px solid #333; z-index: 9999;
         font-size: 13px; padding: 15px; border-radius: 8px;
         box-shadow: 0 4px 8px rgba(0,0,0,0.2);">
        <h3 style="margin: 0 0 12px 0; border-bottom: 2px solid #333; padding-bottom: 8px;">
            🎯 {COUNTRY_NAME} - Closure Map
        </h3>

        <div style="margin: 10px 0;">
            <i class="fa fa-times" style="color: red; font-size: 16px;"></i>
            <b style="color: red;">RED MARKERS</b> = Closure Candidates<br>
            <span style="font-size: 11px; color: #666;">Rating < 3.8 + Multiple Conflicts</span>
        </div>

        <div style="margin: 10px 0;">
            <i class="fa fa-shopping-cart" style="color: blue; font-size: 16px;"></i>
            <b>COLORED MARKERS</b> = Keep (Anchors)<br>
            <span style="font-size: 11px; color: #666;">One color per city cluster</span>
        </div>

        <hr style="margin: 10px 0;">

        <p style="margin: 5px 0; font-size: 12px;">
            <b>🚫 Closure Candidates:</b> {len(closure_candidates)}
        </p>
        <p style="margin: 5px 0; font-size: 12px;">
            <b>📊 Total Conflicts:</b> {closure_candidates['pair_count'].sum()}
        </p>
        <p style="margin: 5px 0; font-size: 12px;">
            <b>🏙️ Cities Affected:</b> {closure_candidates['city'].nunique()}
        </p>

        <hr style="margin: 10px 0;">

        <p style="font-size: 11px; color: #666; margin: 5px 0;">
            <b>Red lines</b> = Cannibalization conflicts<br>
            Click markers for store details
        </p>
    </div>
    '''

    m.get_root().html.add_child(folium.Element(legend_html))

    # Add fullscreen
    plugins.Fullscreen().add_to(m)

    # Save map
    map_file = f'Closure_Map_{COUNTRY_CODE}.html'
    m.save(map_file)

    print("=" * 80)
    print("INTERACTIVE MAP CREATED")
    print("=" * 80)
    print(f"✓ {len(closure_candidates)} closure candidates marked in RED")
    print(f"✓ {len(added_stores)} total stores displayed")
    print(f"✓ Red lines show cannibalization conflicts")
    print(f"\n✓ Map saved to: {map_file}")
    print(f"\nOpen '{map_file}' in your browser to view the interactive map!")

    # Try to display inline
    try:
        from IPython.display import IFrame, display
        display(IFrame(src=map_file, width=1000, height=600))
    except:
        print("\n(Map cannot display inline - open the HTML file instead)")

else:
    print("⚠️ No data to map")

print(f"\n✓ Analysis complete for {COUNTRY_NAME}")
print("=" * 80)

INTERACTIVE MAP CREATED
✓ 14 closure candidates marked in RED
✓ 234 total stores displayed
✓ Red lines show cannibalization conflicts

✓ Map saved to: Closure_Map_IN.html

Open 'Closure_Map_IN.html' in your browser to view the interactive map!



✓ Analysis complete for India


#**Germany**

In [ ]:
# ============================================
# STEP 1: LIBRARY IMPORTS
# ============================================

# Core data manipulation
import pandas as pd
import numpy as np

# Geospatial clustering
from sklearn.cluster import DBSCAN
from math import radians, sin, cos, sqrt, atan2

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# File handling
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


In [ ]:
# ============================================
# STEP 2: DATA LOADING & CLEANING (GERMANY)
# ============================================

# Country configuration
COUNTRY_NAME = "Germany"
COUNTRY_CODE = "DE"
DATA_FILE = "Germany_Levis_stores_data.csv"

# Load the data
df = pd.read_csv(DATA_FILE)

# Drop empty columns
df = df.loc[:, ~df.columns.str.contains('Unnamed|^ $')]

# Basic info
print(f"Country: {COUNTRY_NAME}")
print(f"Total stores loaded: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")

# Clean and standardize data
df_clean = df.copy()

# COLUMN MAPPING - Handle different column name formats
if 'location/lat' in df_clean.columns:
    df_clean = df_clean.rename(columns={
        'location/lat': 'location_lat',
        'location/lng': 'location_lng'
    })

if 'totalScore' in df_clean.columns:
    df_clean = df_clean.rename(columns={'totalScore': 'totalRating'})

# Remove rows with missing coordinates
df_clean = df_clean.dropna(subset=['location_lat', 'location_lng'])

# Fill missing ratings with 0
df_clean['totalRating'] = df_clean['totalRating'].fillna(0)
df_clean['reviewsCount'] = df_clean['reviewsCount'].fillna(0)

# Standardize postal codes
if 'postalCode' in df_clean.columns:
    df_clean['postalCode'] = df_clean['postalCode'].astype(str).str.strip()

# Filter for actual Levi's stores
df_clean = df_clean[
    df_clean['categoryName'].str.contains('Levi|Clothing|Apparel|Fashion|Store',
                                          case=False, na=False)
]

print(f"\nAfter cleaning:")
print(f"Stores remaining: {len(df_clean)}")
print(f"Removed: {len(df) - len(df_clean)} stores")

# Preview
print("\nSample data:")
print(df_clean[['title', 'city', 'totalRating', 'reviewsCount',
                'location_lat', 'location_lng']].head())

print(f"\n✓ Data loaded and cleaned for {COUNTRY_NAME}")

Country: Germany
Total stores loaded: 104

Columns: ['title', 'address', 'neighborhood', 'street', 'city', 'postalCode', 'countryCode', 'website', 'phone', 'phoneUnformatted', 'location/lat', 'location/lng', 'plusCode', 'categoryName', 'totalRating', 'reviewsCount']

After cleaning:
Stores remaining: 103
Removed: 1 stores

Sample data:
                                  title         city  totalRating  reviewsCount  location_lat  location_lng
0                                Levi's      Hamburg          3.7            46     53.653914     10.091091
1                   Levi's® Broderstorf  Broderstorf          4.4            17     54.083202     12.201154
2                                Levi's       Soltau          4.3           315     52.983023      9.923349
3       Levi's® Hamburg Uberseequartier      Hamburg          5.0             4     53.540035      9.999165
4  Levi's® Factory Outlet Stuhr/Brinkum        Stuhr          3.9           148     53.030601      8.805167

✓ Data loaded

In [ ]:
# ============================================
# STEP 3: CLUSTER ALL STORES BY CITY
# ============================================

# Check if 'state' column exists
has_state = 'state' in df_clean.columns

# Count stores per city
if has_state:
    city_summary = df_clean.groupby('city').agg({
        'title': 'count',
        'state': 'first',
        'location_lat': 'mean',
        'location_lng': 'mean'
    }).rename(columns={'title': 'store_count'}).reset_index()
else:
    # No state column - create a placeholder
    city_summary = df_clean.groupby('city').agg({
        'title': 'count',
        'location_lat': 'mean',
        'location_lng': 'mean'
    }).rename(columns={'title': 'store_count'}).reset_index()
    city_summary['state'] = 'Germany'  # Default state name

# Sort by store count
city_summary = city_summary.sort_values('store_count', ascending=False)

print("=" * 80)
print(f"CITY CLUSTERING - ALL {len(df_clean)} STORES IN {COUNTRY_NAME}")
print("=" * 80)

# Split into single-store and multi-store cities
multi_store_cities = city_summary[city_summary['store_count'] >= 2]
single_store_cities = city_summary[city_summary['store_count'] == 1]

print(f"\nTotal stores: {len(df_clean)}")
print(f"Total cities: {len(city_summary)}")
print(f"\nCity Distribution:")
print(f"  Cities with 1 store: {len(single_store_cities)} ({single_store_cities['store_count'].sum()} stores)")
print(f"  Cities with 2+ stores: {len(multi_store_cities)} ({multi_store_cities['store_count'].sum()} stores)")

# Add city cluster assignment
df_clean['city_cluster'] = df_clean['city']
df_clean['stores_in_city'] = df_clean['city'].map(city_summary.set_index('city')['store_count'])

# Add state column if it doesn't exist
if not has_state:
    df_clean['state'] = 'Germany'

print(f"\n✓ ALL {len(df_clean)} STORES CLUSTERED BY CITY")

CITY CLUSTERING - ALL 103 STORES IN Germany

Total stores: 103
Total cities: 73

City Distribution:
  Cities with 1 store: 60 (60 stores)
  Cities with 2+ stores: 13 (43 stores)

✓ ALL 103 STORES CLUSTERED BY CITY


In [ ]:
# ============================================
# STEP 4: DISPLAY STORES IN MULTI-STORE CITIES
# ============================================

print("=" * 80)
print(f"DETAILED STORE BREAKDOWN - {COUNTRY_NAME}")
print("=" * 80)

multi_store_city_list = multi_store_cities['city'].tolist()

for idx, city_name in enumerate(multi_store_city_list, 1):
    city_stores = df_clean[df_clean['city'] == city_name].copy()
    city_stores = city_stores.sort_values('totalRating', ascending=False)

    # Get state name (will be 'Germany' if no state column)
    state_name = city_stores['state'].iloc[0] if 'state' in city_stores.columns else 'Germany'

    print(f"\n{'='*80}")
    print(f"CITY {idx}: {city_name}, {state_name}")
    print(f"Total Stores: {len(city_stores)}")
    print(f"{'='*80}")

    for store_idx, (_, store) in enumerate(city_stores.iterrows(), 1):
        print(f"\n  Store {store_idx}:")
        print(f"    Name:          {store['title']}")
        print(f"    Address:       {store['address']}")
        print(f"    Rating:        {store['totalRating']:.1f} ⭐")
        print(f"    Review Count:  {int(store['reviewsCount'])} reviews")
        print(f"    Coordinates:   ({store['location_lat']:.4f}, {store['location_lng']:.4f})")

    print(f"\n{'-'*80}")

print(f"\n✓ Displayed all multi-store cities in {COUNTRY_NAME}")

DETAILED STORE BREAKDOWN - Germany

CITY 1: Hamburg, Germany
Total Stores: 8

  Store 1:
    Name:          Levi's® Hamburg Uberseequartier
    Address:       Baltimore Str. 4, 20457 Hamburg, Germany
    Rating:        5.0 ⭐
    Review Count:  4 reviews
    Coordinates:   (53.5400, 9.9992)

  Store 2:
    Name:          Levi's
    Address:       Ballindamm 40, 20095 Hamburg, Germany
    Rating:        4.3 ⭐
    Review Count:  118 reviews
    Coordinates:   (53.5513, 9.9961)

  Store 3:
    Name:          Levi's® Schleusenbruecke
    Address:       Schleusenbrücke 1, 20354 Hamburg, Germany
    Rating:        4.2 ⭐
    Review Count:  66 reviews
    Coordinates:   (53.5517, 9.9915)

  Store 4:
    Name:          Levi's® Schulterblatt
    Address:       Schulterblatt 25, 22769 Hamburg, Germany
    Rating:        4.1 ⭐
    Review Count:  55 reviews
    Coordinates:   (53.5603, 9.9632)

  Store 5:
    Name:          Levi's Mercado
    Address:       Ottenser Hauptstraße 10 |/Unit E29, 22765 

In [ ]:
# ============================================
# STEP 5: ROAD DISTANCE CALCULATION
# ============================================

import requests
import time
from itertools import combinations

def get_road_distance_and_route(lat1, lon1, lat2, lon2):
    """Get driving distance AND route geometry using OSRM"""
    try:
        url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
        params = {
            'overview': 'full',
            'geometries': 'geojson'
        }

        response = requests.get(url, params=params, timeout=10)

        if response.status_code == 200:
            data = response.json()
            if data['code'] == 'Ok':
                route = data['routes'][0]
                distance_meters = route['distance']
                distance_miles = distance_meters * 0.000621371
                geometry = route['geometry']
                return distance_miles, geometry

        return None, None
    except Exception as e:
        return None, None

# Get list of multi-store cities
multi_store_city_list = multi_store_cities['city'].tolist()

# Store results
all_distance_data = []

print("=" * 80)
print(f"CALCULATING ROAD DISTANCES - {COUNTRY_NAME}")
print("=" * 80)

# Process each multi-store city
for city_name in multi_store_city_list:
    city_stores = df_clean[df_clean['city'] == city_name].copy()
    store_count = len(city_stores)

    print(f"\nAnalyzing: {city_name} ({store_count} stores)")

    store_list = city_stores.reset_index(drop=True)

    for i, j in combinations(range(len(store_list)), 2):
        store_a = store_list.iloc[i]
        store_b = store_list.iloc[j]

        distance_miles, route_geometry = get_road_distance_and_route(
            store_a['location_lat'], store_a['location_lng'],
            store_b['location_lat'], store_b['location_lng']
        )

        if distance_miles is not None:
            distance_data = {
                'city': city_name,
                'state': store_a['state'],
                'store_1_name': store_a['title'],
                'store_1_address': store_a['address'],
                'store_1_rating': store_a['totalRating'],
                'store_1_reviews': store_a['reviewsCount'],
                'store_1_lat': store_a['location_lat'],
                'store_1_lng': store_a['location_lng'],
                'store_2_name': store_b['title'],
                'store_2_address': store_b['address'],
                'store_2_rating': store_b['totalRating'],
                'store_2_reviews': store_b['reviewsCount'],
                'store_2_lat': store_b['location_lat'],
                'store_2_lng': store_b['location_lng'],
                'road_distance_miles': distance_miles,
                'route_geometry': route_geometry
            }

            all_distance_data.append(distance_data)

        time.sleep(0.5)

    print(f"  ✓ Completed {city_name}")

# Convert to DataFrame
df_clusters = pd.DataFrame(all_distance_data)

print("\n" + "=" * 80)
print("DISTANCE CALCULATION COMPLETE")
print("=" * 80)

if len(df_clusters) > 0:
    print(f"\nTotal store pairs: {len(df_clusters)}")
    print(f"Cities analyzed: {df_clusters['city'].nunique()}")

    # Display by city
    for city_name in df_clusters['city'].unique():
        city_data = df_clusters[df_clusters['city'] == city_name].copy()
        city_data = city_data.sort_values('road_distance_miles').reset_index(drop=True)

        state_name = city_data['state'].iloc[0]

        print(f"\n{'='*80}")
        print(f"CLUSTER: {city_name}, {state_name}")
        print(f"{'='*80}")

        table = pd.DataFrame({
            'Store A': city_data['store_1_name'],
            'Rating A': city_data['store_1_rating'].round(1),
            'Reviews A': city_data['store_1_reviews'].astype(int),
            'Store B': city_data['store_2_name'],
            'Rating B': city_data['store_2_rating'].round(1),
            'Reviews B': city_data['store_2_reviews'].astype(int),
            'Distance (mi)': city_data['road_distance_miles'].round(2)
        })

        print(table.to_string(index=True))
        print(f"\nPairs in cluster: {len(city_data)}")

print(f"\n✓ Road distances calculated for {COUNTRY_NAME}")

CALCULATING ROAD DISTANCES - Germany

Analyzing: Hamburg (8 stores)
  ✓ Completed Hamburg

Analyzing: Berlin (7 stores)
  ✓ Completed Berlin

Analyzing: Munich (6 stores)
  ✓ Completed Munich

Analyzing: Dresden (3 stores)
  ✓ Completed Dresden

Analyzing: Cologne (3 stores)
  ✓ Completed Cologne

Analyzing: Augsburg (2 stores)
  ✓ Completed Augsburg

Analyzing: Metzingen (2 stores)
  ✓ Completed Metzingen

Analyzing: Frankfurt am Main (2 stores)
  ✓ Completed Frankfurt am Main

Analyzing: Düsseldorf (2 stores)
  ✓ Completed Düsseldorf

Analyzing: Stuttgart (2 stores)
  ✓ Completed Stuttgart

Analyzing: Wertheim am Main (2 stores)
  ✓ Completed Wertheim am Main

Analyzing: Nuremberg (2 stores)
  ✓ Completed Nuremberg

Analyzing: Dortmund (2 stores)
  ✓ Completed Dortmund

DISTANCE CALCULATION COMPLETE

Total store pairs: 78
Cities analyzed: 13

CLUSTER: Hamburg, Germany
                            Store A  Rating A  Reviews A                          Store B  Rating B  Reviews B  Dista

In [ ]:
# ============================================
# STEP 6: FILTER STORES WITHIN 5 MILES
# ============================================

print("=" * 80)
print("FILTERING STORES WITHIN 5 MILES")
print("=" * 80)

df_filtered = df_clusters[df_clusters['road_distance_miles'] < 5.0].copy()

print(f"\nOriginal store pairs: {len(df_clusters)}")
print(f"Pairs within 5 miles: {len(df_filtered)}")
print(f"Pairs removed: {len(df_clusters) - len(df_filtered)}")

if len(df_filtered) > 0:
    print("\n" + "=" * 80)
    print("STORES WITHIN 5 MILES - BY CITY CLUSTER")
    print("=" * 80)

    for city_name in df_filtered['city'].unique():
        city_data = df_filtered[df_filtered['city'] == city_name].copy()
        city_data = city_data.sort_values('road_distance_miles').reset_index(drop=True)

        state_name = city_data['state'].iloc[0]

        print(f"\n{'='*80}")
        print(f"CLUSTER: {city_name}, {state_name}")
        print(f"Pairs within 5 miles: {len(city_data)}")
        print(f"{'='*80}")

        table = pd.DataFrame({
            'Store A': city_data['store_1_name'],
            'Rating A': city_data['store_1_rating'].round(1),
            'Reviews A': city_data['store_1_reviews'].astype(int),
            'Store B': city_data['store_2_name'],
            'Rating B': city_data['store_2_rating'].round(1),
            'Reviews B': city_data['store_2_reviews'].astype(int),
            'Distance (mi)': city_data['road_distance_miles'].round(2)
        })

        print(table.to_string(index=True))

        min_dist = city_data['road_distance_miles'].min()
        max_dist = city_data['road_distance_miles'].max()
        avg_dist = city_data['road_distance_miles'].mean()
        print(f"\nDistance range: {min_dist:.2f} - {max_dist:.2f} miles (avg: {avg_dist:.2f} mi)")

    # Summary statistics
    print("\n" + "=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)

    print(f"\nTotal pairs within 5 miles: {len(df_filtered)}")
    print(f"Cities affected: {df_filtered['city'].nunique()}")
    print(f"Closest pair: {df_filtered['road_distance_miles'].min():.2f} miles")
    print(f"Farthest pair: {df_filtered['road_distance_miles'].max():.2f} miles")
    print(f"Average distance: {df_filtered['road_distance_miles'].mean():.2f} miles")

else:
    print("\n⚠️  No store pairs found within 5 miles")

print(f"\n✓ Filtering complete for {COUNTRY_NAME}")

FILTERING STORES WITHIN 5 MILES

Original store pairs: 78
Pairs within 5 miles: 48
Pairs removed: 30

STORES WITHIN 5 MILES - BY CITY CLUSTER

CLUSTER: Hamburg, Germany
Pairs within 5 miles: 17
                            Store A  Rating A  Reviews A                          Store B  Rating B  Reviews B  Distance (mi)
0   Levi's® Hamburg Uberseequartier       5.0          4  Levi's® Hamburg Spitalerstrasse       3.5        108           0.92
1                            Levi's       4.3        118         Levi's® Schleusenbruecke       4.2         66           1.02
2   Levi's® Hamburg Uberseequartier       5.0          4         Levi's® Schleusenbruecke       4.2         66           1.40
3   Levi's® Hamburg Spitalerstrasse       3.5        108                           Levi's       4.3        118           1.55
4   Levi's® Hamburg Spitalerstrasse       3.5        108         Levi's® Schleusenbruecke       4.2         66           1.61
5          Levi's® Schleusenbruecke       4.2     

In [ ]:
# ============================================
# STEP 7: IDENTIFY CLOSURE CANDIDATES (GERMANY)
# Strategy: Rating < 3.7 AND appears in 2+ pairs
# ============================================

print("=" * 80)
print("IDENTIFYING CLOSURE CANDIDATES - GERMANY")
print("Low-rated stores (< 3.7) appearing in multiple pairs")
print("=" * 80)

# Combine Store A and Store B data
store_a_data = df_filtered[['city', 'state', 'store_1_name', 'store_1_address',
                              'store_1_rating', 'store_1_reviews',
                              'store_1_lat', 'store_1_lng']].copy()
store_a_data.columns = ['city', 'state', 'store_name', 'address', 'rating',
                         'reviews', 'lat', 'lng']

store_b_data = df_filtered[['city', 'state', 'store_2_name', 'store_2_address',
                              'store_2_rating', 'store_2_reviews',
                              'store_2_lat', 'store_2_lng']].copy()
store_b_data.columns = ['city', 'state', 'store_name', 'address', 'rating',
                         'reviews', 'lat', 'lng']

all_stores = pd.concat([store_a_data, store_b_data], ignore_index=True)

# Count appearances
store_counts = all_stores.groupby(['city', 'store_name', 'address']).agg({
    'state': 'first',
    'rating': 'first',
    'reviews': 'first',
    'lat': 'first',
    'lng': 'first',
    'store_name': 'count'
}).rename(columns={'store_name': 'pair_count'}).reset_index()

# Filter for rating < 3.7 AND appears in 2+ pairs
closure_candidates = store_counts[
    (store_counts['rating'] < 3.7) &
    (store_counts['pair_count'] >= 2)
].copy()

closure_candidates = closure_candidates.sort_values(
    ['pair_count', 'rating'],
    ascending=[False, True]
).reset_index(drop=True)

print(f"\n🎯 FOUND {len(closure_candidates)} CLOSURE CANDIDATES")
print(f"   (Rating < 3.7 AND in 2+ pairs within 5 miles)\n")

if len(closure_candidates) > 0:

    print("=" * 80)
    print("PRIORITY CLOSURE CANDIDATES (Ranked by Conflicts)")
    print("=" * 80)

    for idx, store in closure_candidates.iterrows():
        print(f"\n{'='*80}")
        print(f"RANK #{idx + 1}: {store['store_name']}")
        print(f"{'='*80}")
        print(f"📍 Address: {store['address']}")
        print(f"🏙️  City: {store['city']}, {store['state']}")
        print(f"⭐ Rating: {store['rating']:.1f} ⚠️ (BELOW 3.7)")
        print(f"💬 Reviews: {int(store['reviews'])}")
        print(f"🔄 Appears in {store['pair_count']} pairs (cannibalization conflicts)")

        # Find conflicts
        print(f"\n   Conflicts with:")

        conflicts_as_a = df_filtered[
            df_filtered['store_1_address'] == store['address']
        ][['store_2_name', 'store_2_rating', 'store_2_reviews', 'road_distance_miles']]

        conflicts_as_b = df_filtered[
            df_filtered['store_2_address'] == store['address']
        ][['store_1_name', 'store_1_rating', 'store_1_reviews', 'road_distance_miles']]

        conflict_num = 1

        for _, conflict in conflicts_as_a.iterrows():
            print(f"   {conflict_num}. {conflict['store_2_name'][:65]}")
            print(f"      Rating: {conflict['store_2_rating']:.1f} ⭐ | Reviews: {int(conflict['store_2_reviews'])} | Distance: {conflict['road_distance_miles']:.2f} mi")
            conflict_num += 1

        for _, conflict in conflicts_as_b.iterrows():
            print(f"   {conflict_num}. {conflict['store_1_name'][:65]}")
            print(f"      Rating: {conflict['store_1_rating']:.1f} ⭐ | Reviews: {int(conflict['store_1_reviews'])} | Distance: {conflict['road_distance_miles']:.2f} mi")
            conflict_num += 1

        engagement_score = store['rating'] * np.log10(store['reviews'] + 1)
        print(f"\n   📊 Performance Score: {engagement_score:.2f}")
        print(f"   💡 RECOMMENDATION: CLOSE this store")
        print(f"      ✅ Eliminates {store['pair_count']} cannibalization conflicts")
        print(f"      ✅ Rating below 3.7 threshold (poor performance)")
        print(f"      ✅ Traffic redirects to {store['pair_count']} nearby stores")

    # Summary table
    print("\n" + "=" * 80)
    print("SUMMARY TABLE - CLOSURE CANDIDATES")
    print("=" * 80)

    summary = pd.DataFrame({
        'Rank': range(1, len(closure_candidates) + 1),
        'Store Name': closure_candidates['store_name'].str[:45],
        'City': closure_candidates['city'],
        'Rating': closure_candidates['rating'].round(1),
        'Reviews': closure_candidates['reviews'].astype(int),
        'Conflicts': closure_candidates['pair_count'].astype(int)
    })

    print(summary.to_string(index=False))

    # Strategic impact
    print("\n" + "=" * 80)
    print("STRATEGIC IMPACT ANALYSIS")
    print("=" * 80)

    total_conflicts = closure_candidates['pair_count'].sum()
    avg_rating = closure_candidates['rating'].mean()

    print(f"\nIf we close these {len(closure_candidates)} stores:")
    print(f"  ✅ Total conflicts eliminated: {total_conflicts}")
    print(f"  ✅ Average rating of closed stores: {avg_rating:.2f} (all below 3.7)")
    print(f"  ✅ Estimated SG&A savings: ${len(closure_candidates) * 600}K - ${len(closure_candidates) * 850}K annually")
    print(f"  ✅ Revenue retention (via nearby stores): ~90%")

    # Top priority
    if len(closure_candidates) > 0:
        top_priority = closure_candidates.iloc[0]
        print(f"\n🎯 TOP PRIORITY CLOSURE:")
        print(f"   {top_priority['store_name']}")
        print(f"   • {top_priority['pair_count']} conflicts (highest)")
        print(f"   • Rating: {top_priority['rating']:.1f} (poor performance)")
        print(f"   • Location: {top_priority['city']}")

    # High-rated anchor stores
    print("\n" + "=" * 80)
    print("FOR COMPARISON: High-Rated Stores in Multiple Pairs")
    print("(Rating >= 4.0 AND in 3+ pairs - ANCHOR STORES to KEEP)")
    print("=" * 80)

    anchor_stores = store_counts[
        (store_counts['rating'] >= 4.0) &
        (store_counts['pair_count'] >= 3)
    ].sort_values('pair_count', ascending=False)

    if len(anchor_stores) > 0:
        print(f"\nFound {len(anchor_stores)} anchor stores (KEEP these):\n")
        for idx, store in anchor_stores.head(5).iterrows():
            print(f"  • {store['store_name'][:60]}")
            print(f"    {store['city']} - {store['rating']:.1f}⭐ ({int(store['reviews'])} reviews)")
            print(f"    In {store['pair_count']} pairs - HIGH PERFORMER\n")

        print("⚠️  These are ANCHOR stores. Close stores AROUND them, not these.")
    else:
        print("\nNo high-rated multi-conflict stores found")

    # Rating distribution
    print("\n" + "=" * 80)
    print("RATING DISTRIBUTION - ALL STORES IN CONFLICTS")
    print("=" * 80)

    print(f"\nStores with rating < 3.7: {len(store_counts[store_counts['rating'] < 3.7])} (CLOSURE CANDIDATES)")
    print(f"Stores with rating 3.7-3.9: {len(store_counts[(store_counts['rating'] >= 3.7) & (store_counts['rating'] < 4.0)])}")
    print(f"Stores with rating 4.0-4.2: {len(store_counts[(store_counts['rating'] >= 4.0) & (store_counts['rating'] < 4.3)])}")
    print(f"Stores with rating 4.3+: {len(store_counts[store_counts['rating'] >= 4.3])}")

else:
    print("\n✓ No stores found with rating < 3.7 in multiple pairs")
    print("All stores in conflicts are performing above threshold")

print(f"\n✓ Closure analysis complete for {COUNTRY_NAME}")
print("=" * 80)

IDENTIFYING CLOSURE CANDIDATES - GERMANY
Low-rated stores (< 3.7) appearing in multiple pairs

🎯 FOUND 3 CLOSURE CANDIDATES
   (Rating < 3.7 AND in 2+ pairs within 5 miles)

PRIORITY CLOSURE CANDIDATES (Ranked by Conflicts)

RANK #1: Levi's® Hamburg Spitalerstrasse
📍 Address: Spitalerstraße 30, 20095 Hamburg, Germany
🏙️  City: Hamburg, Germany
⭐ Rating: 3.5 ⚠️ (BELOW 3.7)
💬 Reviews: 108
🔄 Appears in 5 pairs (cannibalization conflicts)

   Conflicts with:
   1. Levi's
      Rating: 4.3 ⭐ | Reviews: 118 | Distance: 1.55 mi
   2. Levi's Mercado
      Rating: 3.9 ⭐ | Reviews: 67 | Distance: 3.74 mi
   3. Levi's® Schleusenbruecke
      Rating: 4.2 ⭐ | Reviews: 66 | Distance: 1.61 mi
   4. Levi's® Schulterblatt
      Rating: 4.1 ⭐ | Reviews: 55 | Distance: 2.67 mi
   5. Levi's® Hamburg Uberseequartier
      Rating: 5.0 ⭐ | Reviews: 4 | Distance: 0.92 mi

   📊 Performance Score: 7.13
   💡 RECOMMENDATION: CLOSE this store
      ✅ Eliminates 5 cannibalization conflicts
      ✅ Rating below 3.7 

In [ ]:
# ============================================
# STEP 8: INTERACTIVE MAP VISUALIZATION
# ============================================

import subprocess
import sys

try:
    import folium
    from folium import plugins
except ImportError:
    print("Installing folium...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium", "--break-system-packages", "-q"])
    import folium
    from folium import plugins

if len(df_filtered) > 0 and len(closure_candidates) > 0:

    # Calculate center point
    all_lats = list(df_filtered['store_1_lat']) + list(df_filtered['store_2_lat'])
    all_lngs = list(df_filtered['store_1_lng']) + list(df_filtered['store_2_lng'])
    center_lat = sum(all_lats) / len(all_lats)
    center_lng = sum(all_lngs) / len(all_lngs)

    # Create map
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=6,
        tiles='OpenStreetMap'
    )

    # Color palette
    colors = ['blue', 'green', 'purple', 'orange', 'cadetblue',
              'darkgreen', 'darkblue', 'pink', 'lightgreen',
              'lightblue', 'beige', 'lightred', 'gray', 'black']

    city_colors = {}
    color_idx = 0

    for city in df_filtered['city'].unique():
        city_colors[city] = colors[color_idx % len(colors)]
        color_idx += 1

    # Draw city bounding boxes
    for city in df_filtered['city'].unique():
        city_data = df_filtered[df_filtered['city'] == city]
        city_lats = list(city_data['store_1_lat']) + list(city_data['store_2_lat'])
        city_lngs = list(city_data['store_1_lng']) + list(city_data['store_2_lng'])

        min_lat, max_lat = min(city_lats), max(city_lats)
        min_lng, max_lng = min(city_lngs), max(city_lngs)

        padding = 0.01
        bounds = [
            [min_lat - padding, min_lng - padding],
            [max_lat + padding, max_lng + padding]
        ]

        folium.Rectangle(
            bounds=bounds,
            color=city_colors[city],
            fill=True,
            fill_opacity=0.1,
            weight=2,
            tooltip=f"Cluster: {city}"
        ).add_to(m)

    # Create closure candidate set
    closure_addresses = set(closure_candidates['address'].tolist())
    added_stores = set()

    # Add markers
    for idx, row in df_filtered.iterrows():
        city_color = city_colors[row['city']]

        # Store A
        store_a_key = (row['store_1_lat'], row['store_1_lng'])
        if store_a_key not in added_stores:
            is_closure = row['store_1_address'] in closure_addresses

            popup_html = f"""
            <div style='width: 280px; font-family: Arial;'>
                <h4 style='margin: 0; color: {"red" if is_closure else city_color};
                     border-bottom: 2px solid {"red" if is_closure else city_color}; padding-bottom: 5px;'>
                    {"🚫 CLOSURE CANDIDATE" if is_closure else "🛒"} {row['store_1_name'][:40]}
                </h4>
                <p style='margin: 8px 0; font-size: 11px;'><i>{row['store_1_address']}</i></p>
                <div style='background: #f5f5f5; padding: 8px; border-radius: 4px;'>
                    <b>⭐ Rating:</b> {row['store_1_rating']:.1f}/5.0<br>
                    <b>💬 Reviews:</b> {int(row['store_1_reviews'])}<br>
                    <b>📍 City:</b> {row['city']}, {row['state']}
                </div>
            </div>
            """

            folium.Marker(
                location=[row['store_1_lat'], row['store_1_lng']],
                popup=folium.Popup(popup_html, max_width=300),
                tooltip=f"{row['store_1_name'][:30]} - {row['store_1_rating']:.1f}⭐",
                icon=folium.Icon(
                    color='red' if is_closure else city_color,
                    icon='times' if is_closure else 'shopping-cart',
                    prefix='fa'
                )
            ).add_to(m)

            added_stores.add(store_a_key)

        # Store B
        store_b_key = (row['store_2_lat'], row['store_2_lng'])
        if store_b_key not in added_stores:
            is_closure = row['store_2_address'] in closure_addresses

            popup_html = f"""
            <div style='width: 280px; font-family: Arial;'>
                <h4 style='margin: 0; color: {"red" if is_closure else city_color};
                     border-bottom: 2px solid {"red" if is_closure else city_color}; padding-bottom: 5px;'>
                    {"🚫 CLOSURE CANDIDATE" if is_closure else "🛒"} {row['store_2_name'][:40]}
                </h4>
                <p style='margin: 8px 0; font-size: 11px;'><i>{row['store_2_address']}</i></p>
                <div style='background: #f5f5f5; padding: 8px; border-radius: 4px;'>
                    <b>⭐ Rating:</b> {row['store_2_rating']:.1f}/5.0<br>
                    <b>💬 Reviews:</b> {int(row['store_2_reviews'])}<br>
                    <b>📍 City:</b> {row['city']}, {row['state']}
                </div>
            </div>
            """

            folium.Marker(
                location=[row['store_2_lat'], row['store_2_lng']],
                popup=folium.Popup(popup_html, max_width=300),
                tooltip=f"{row['store_2_name'][:30]} - {row['store_2_rating']:.1f}⭐",
                icon=folium.Icon(
                    color='red' if is_closure else city_color,
                    icon='times' if is_closure else 'shopping-cart',
                    prefix='fa'
                )
            ).add_to(m)

            added_stores.add(store_b_key)

    # Draw conflict routes
    for idx, candidate in closure_candidates.iterrows():
        conflicts_a = df_filtered[df_filtered['store_1_address'] == candidate['address']]
        conflicts_b = df_filtered[df_filtered['store_2_address'] == candidate['address']]

        for _, conflict in conflicts_a.iterrows():
            geometry = conflict['route_geometry']
            if geometry and 'coordinates' in geometry:
                route_coords = [[coord[1], coord[0]] for coord in geometry['coordinates']]
                folium.PolyLine(
                    locations=route_coords,
                    color='red',
                    weight=5,
                    opacity=0.8,
                    tooltip=f"🚫 CONFLICT: {conflict['road_distance_miles']:.2f} mi"
                ).add_to(m)

        for _, conflict in conflicts_b.iterrows():
            geometry = conflict['route_geometry']
            if geometry and 'coordinates' in geometry:
                route_coords = [[coord[1], coord[0]] for coord in geometry['coordinates']]
                folium.PolyLine(
                    locations=route_coords,
                    color='red',
                    weight=5,
                    opacity=0.8,
                    tooltip=f"🚫 CONFLICT: {conflict['road_distance_miles']:.2f} mi"
                ).add_to(m)

    # Add legend
    legend_html = f'''
    <div style="position: fixed; top: 10px; right: 10px; width: 280px;
         background-color: white; border: 3px solid #333; z-index: 9999;
         font-size: 13px; padding: 15px; border-radius: 8px;
         box-shadow: 0 4px 8px rgba(0,0,0,0.2);">
        <h3 style="margin: 0 0 12px 0; border-bottom: 2px solid #333; padding-bottom: 8px;">
            🎯 {COUNTRY_NAME} - Closure Map
        </h3>

        <div style="margin: 10px 0;">
            <i class="fa fa-times" style="color: red; font-size: 16px;"></i>
            <b style="color: red;">RED MARKERS</b> = Closure Candidates<br>
            <span style="font-size: 11px; color: #666;">Rating < 4.2 + Multiple Conflicts</span>
        </div>

        <div style="margin: 10px 0;">
            <i class="fa fa-shopping-cart" style="color: blue; font-size: 16px;"></i>
            <b>COLORED MARKERS</b> = Keep (Anchors)<br>
            <span style="font-size: 11px; color: #666;">One color per city cluster</span>
        </div>

        <hr style="margin: 10px 0;">

        <p style="margin: 5px 0; font-size: 12px;">
            <b>🚫 Closure Candidates:</b> {len(closure_candidates)}
        </p>
        <p style="margin: 5px 0; font-size: 12px;">
            <b>📊 Total Conflicts:</b> {closure_candidates['pair_count'].sum()}
        </p>
        <p style="margin: 5px 0; font-size: 12px;">
            <b>🏙️ Cities Affected:</b> {closure_candidates['city'].nunique()}
        </p>

        <hr style="margin: 10px 0;">

        <p style="font-size: 11px; color: #666; margin: 5px 0;">
            <b>Red lines</b> = Cannibalization conflicts<br>
            Click markers for store details
        </p>
    </div>
    '''

    m.get_root().html.add_child(folium.Element(legend_html))

    # Add fullscreen
    plugins.Fullscreen().add_to(m)

    # Save map
    map_file = f'Closure_Map_{COUNTRY_CODE}.html'
    m.save(map_file)

    print("=" * 80)
    print("INTERACTIVE MAP CREATED")
    print("=" * 80)
    print(f"✓ {len(closure_candidates)} closure candidates marked in RED")
    print(f"✓ {len(added_stores)} total stores displayed")
    print(f"✓ Red lines show cannibalization conflicts")
    print(f"\n✓ Map saved to: {map_file}")
    print(f"\nOpen '{map_file}' in your browser to view the interactive map!")

    # Try to display inline
    try:
        from IPython.display import IFrame, display
        display(IFrame(src=map_file, width=1000, height=600))
    except:
        print("\n(Map cannot display inline - open the HTML file instead)")

else:
    print("⚠️ No data to map")

print(f"\n✓ Analysis complete for {COUNTRY_NAME}")
print("=" * 80)

INTERACTIVE MAP CREATED
✓ 3 closure candidates marked in RED
✓ 40 total stores displayed
✓ Red lines show cannibalization conflicts

✓ Map saved to: Closure_Map_DE.html

Open 'Closure_Map_DE.html' in your browser to view the interactive map!



✓ Analysis complete for Germany
